In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

for path in Path("/content/drive/MyDrive").rglob("*"):
    if path.is_dir() and "delaware" in path.name.lower():
        print(path)

In [ ]:
DELAWARE_FOLDER_ID = "1RV-tYijUIOBV4O7jb8RcvnWC3RdV8U5J"

In [ ]:
# ============================================================
# DELAWARE CINDERELLA RECALL EXTRACTION
# Uses Google Drive folder ID only — no Drive folder paths
# ============================================================

!pip -q install pydub google-api-python-client

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

from pathlib import Path
from pydub import AudioSegment

import io
import re
import json
import shutil
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DELAWARE_FOLDER_ID = "1RV-tYijUIOBV4O7jb8RcvnWC3RdV8U5J"

LOCAL_ROOT = Path("/content/delaware_recall_processing")
DOWNLOAD_ROOT = LOCAL_ROOT / "downloaded"
OUTPUT_ROOT = LOCAL_ROOT / "Delaware_Recall_Processed"

PAR_AUDIO_ROOT = OUTPUT_ROOT / "par_audio"
TRANSCRIPT_ROOT = OUTPUT_ROOT / "transcripts"
AUDIT_ROOT = OUTPUT_ROOT / "audit"

for path in [
    DOWNLOAD_ROOT,
    PAR_AUDIO_ROOT,
    TRANSCRIPT_ROOT,
    AUDIT_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

drive_service = build("drive", "v3")


# ============================================================
# GOOGLE DRIVE UTILITIES
# ============================================================

def list_folder(folder_id):
    """List every direct child of a Google Drive folder."""
    items = []
    page_token = None

    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields=(
                "nextPageToken,"
                "files(id,name,mimeType,size,parents)"
            ),
            pageToken=page_token,
            pageSize=1000,
        ).execute()

        items.extend(response.get("files", []))
        page_token = response.get("nextPageToken")

        if not page_token:
            break

    return items


def walk_drive_folder(folder_id, relative_path=Path()):
    """Recursively list files beneath a Drive folder ID."""
    records = []

    for item in list_folder(folder_id):
        current_relative_path = relative_path / item["name"]

        if item["mimeType"] == "application/vnd.google-apps.folder":
            records.extend(
                walk_drive_folder(
                    item["id"],
                    current_relative_path,
                )
            )
        else:
            records.append({
                "file_id": item["id"],
                "name": item["name"],
                "relative_path": str(current_relative_path),
                "mime_type": item["mimeType"],
                "size": int(item.get("size", 0)),
            })

    return records


def download_drive_file(file_id, destination):
    """Download one regular Drive file."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    request = drive_service.files().get_media(fileId=file_id)

    with destination.open("wb") as output_file:
        downloader = MediaIoBaseDownload(
            output_file,
            request,
            chunksize=16 * 1024 * 1024,
        )

        done = False
        while not done:
            _, done = downloader.next_chunk()

    return destination


def get_or_create_folder(parent_id, folder_name):
    """Return a child folder ID, creating it if necessary."""
    escaped_name = folder_name.replace("'", "\\'")

    response = drive_service.files().list(
        q=(
            f"'{parent_id}' in parents and "
            f"name = '{escaped_name}' and "
            "mimeType = 'application/vnd.google-apps.folder' and "
            "trashed = false"
        ),
        fields="files(id,name)",
    ).execute()

    matches = response.get("files", [])

    if matches:
        return matches[0]["id"]

    metadata = {
        "name": folder_name,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [parent_id],
    }

    created = drive_service.files().create(
        body=metadata,
        fields="id",
    ).execute()

    return created["id"]


def upload_file(local_path, parent_id, drive_name=None):
    """Upload one local file to a Drive folder."""
    local_path = Path(local_path)
    drive_name = drive_name or local_path.name

    metadata = {
        "name": drive_name,
        "parents": [parent_id],
    }

    media = MediaFileUpload(
        str(local_path),
        resumable=True,
    )

    return drive_service.files().create(
        body=metadata,
        media_body=media,
        fields="id,name",
    ).execute()


# ============================================================
# INDEX DELAWARE FILES
# ============================================================

print("=" * 100)
print("INDEXING LINKED DELAWARE FOLDER")
print("=" * 100)

drive_records = walk_drive_folder(DELAWARE_FOLDER_ID)
drive_index = pd.DataFrame(drive_records)

print("Files found:", len(drive_index))

if len(drive_index) == 0:
    raise RuntimeError(
        "No files were found. Make sure the folder is accessible "
        "to the Google account authenticated in Colab."
    )

display(
    drive_index[
        ["relative_path", "mime_type", "size"]
    ].head(30)
)


# ============================================================
# IDENTIFY AUDIO AND TRANSCRIPT FILES
# ============================================================

def normalize_stem(filename):
    return Path(filename).stem.strip().lower()


def infer_diagnosis(relative_path):
    parts = [
        part.lower()
        for part in Path(relative_path).parts
    ]

    if "control" in parts:
        return "Control"

    if "mci" in parts:
        return "MCI"

    return "Unknown"


audio_records = {}
transcript_records = {}

for record in drive_records:
    suffix = Path(record["name"]).suffix.lower()
    diagnosis = infer_diagnosis(record["relative_path"])
    stem = normalize_stem(record["name"])

    # Diagnosis is included in the key so identical IDs across
    # diagnosis folders cannot accidentally match.
    key = (diagnosis, stem)

    if suffix in {".mp3", ".wav", ".m4a", ".flac"}:
        audio_records[key] = record

    elif suffix == ".cha":
        transcript_records[key] = record


all_keys = sorted(
    set(audio_records) | set(transcript_records)
)

pair_rows = []

for diagnosis, stem in all_keys:
    pair_rows.append({
        "diagnosis": diagnosis,
        "recording_id": stem,
        "has_audio": (diagnosis, stem) in audio_records,
        "has_transcript": (diagnosis, stem) in transcript_records,
        "audio_path": (
            audio_records[(diagnosis, stem)]["relative_path"]
            if (diagnosis, stem) in audio_records
            else None
        ),
        "transcript_path": (
            transcript_records[(diagnosis, stem)]["relative_path"]
            if (diagnosis, stem) in transcript_records
            else None
        ),
    })

pair_table = pd.DataFrame(pair_rows)

print()
print("Audio files:", len(audio_records))
print("CHAT transcripts:", len(transcript_records))
print(
    "Complete pairs:",
    int(
        (
            pair_table["has_audio"]
            & pair_table["has_transcript"]
        ).sum()
    ),
)

display(
    pair_table.groupby(
        ["diagnosis", "has_audio", "has_transcript"]
    ).size().reset_index(name="count")
)

pair_table.to_csv(
    AUDIT_ROOT / "drive_file_pairing.csv",
    index=False,
)


# ============================================================
# CHAT TRANSCRIPT PARSING
# ============================================================

TIMESTAMP_PATTERN = re.compile(
    r"\x15\s*(\d+)_(\d+)\s*\x15"
)


def clean_chat_utterance(text):
    """Create a readable transcript from a CHAT speaker line."""
    text = TIMESTAMP_PATTERN.sub("", text)

    # Remove common CHAT annotation codes.
    text = re.sub(r"\[[^\]]*\]", " ", text)
    text = re.sub(r"<([^>]*)>", r"\1", text)
    text = re.sub(r"&[=+\-]?\w+", " ", text)
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"\bxxx\b|\byyy\b|\bwww\b", " ", text)

    # Remove CHAT replacement markers and extra spacing.
    text = text.replace("↫", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip(" \t\r\n")


def find_cinderella_section(lines):
    """
    Return lines after @G: Cinderella and before the next @G marker.

    Cinderella_Intro is intentionally excluded.
    """
    start_index = None
    end_index = None

    for index, line in enumerate(lines):
        if not line.startswith("@G:"):
            continue

        marker = line.split(":", 1)[1].strip().lower()

        if marker == "cinderella":
            start_index = index + 1
            continue

        if start_index is not None:
            end_index = index
            break

    if start_index is None:
        return None

    if end_index is None:
        end_index = len(lines)

    return lines[start_index:end_index]


def parse_cinderella_par_utterances(cha_text):
    """
    Extract participant utterances and timestamps from the
    Cinderella section only.
    """
    lines = cha_text.splitlines()
    section = find_cinderella_section(lines)

    if section is None:
        return {
            "status": "missing_cinderella_marker",
            "utterances": [],
        }

    utterances = []
    current_speaker_line = None

    # CHAT utterances can continue onto tab-indented lines.
    combined_lines = []

    for line in section:
        if line.startswith("*"):
            if current_speaker_line is not None:
                combined_lines.append(current_speaker_line)

            current_speaker_line = line

        elif (
            current_speaker_line is not None
            and line.startswith("\t")
        ):
            current_speaker_line += " " + line.strip()

        else:
            if current_speaker_line is not None:
                combined_lines.append(current_speaker_line)
                current_speaker_line = None

    if current_speaker_line is not None:
        combined_lines.append(current_speaker_line)

    for line in combined_lines:
        if not line.startswith("*PAR:"):
            continue

        matches = TIMESTAMP_PATTERN.findall(line)

        if not matches:
            continue

        start_ms = min(int(start) for start, _ in matches)
        end_ms = max(int(end) for _, end in matches)

        raw_text = line.split(":", 1)[1].strip()
        clean_text = clean_chat_utterance(raw_text)

        if end_ms <= start_ms:
            continue

        utterances.append({
            "start_ms": start_ms,
            "end_ms": end_ms,
            "duration_ms": end_ms - start_ms,
            "raw_text": raw_text,
            "clean_text": clean_text,
        })

    if not utterances:
        return {
            "status": "no_timestamped_par_utterances",
            "utterances": [],
        }

    return {
        "status": "ok",
        "utterances": utterances,
    }


# ============================================================
# AUDIO EXTRACTION
# ============================================================

def create_participant_only_audio(
    source_audio_path,
    utterances,
    output_path,
    inter_utterance_silence_ms=250,
    edge_padding_ms=80,
):
    """
    Concatenate only participant intervals.

    This removes interviewer speech instead of retaining the
    entire continuous Cinderella section.
    """
    source_audio = AudioSegment.from_file(source_audio_path)

    output_audio = AudioSegment.silent(duration=0)
    separator = AudioSegment.silent(
        duration=inter_utterance_silence_ms
    )

    retained_intervals = []

    for utterance in utterances:
        start_ms = max(
            0,
            utterance["start_ms"] - edge_padding_ms,
        )
        end_ms = min(
            len(source_audio),
            utterance["end_ms"] + edge_padding_ms,
        )

        if end_ms <= start_ms:
            continue

        segment = source_audio[start_ms:end_ms]

        if len(output_audio) > 0:
            output_audio += separator

        output_audio += segment

        retained_intervals.append({
            **utterance,
            "padded_start_ms": start_ms,
            "padded_end_ms": end_ms,
        })

    if not retained_intervals:
        raise ValueError("No valid participant audio intervals.")

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    output_audio = (
        output_audio
        .set_channels(1)
        .set_frame_rate(16000)
        .set_sample_width(2)
    )

    output_audio.export(
        output_path,
        format="wav",
    )

    return {
        "source_duration_ms": len(source_audio),
        "output_duration_ms": len(output_audio),
        "retained_intervals": retained_intervals,
    }


# ============================================================
# PROCESS MATCHED RECORDINGS
# ============================================================

matched_pairs = pair_table[
    pair_table["has_audio"]
    & pair_table["has_transcript"]
].copy()

manifest_rows = []
utterance_audit_rows = []

print()
print("=" * 100)
print("EXTRACTING DELAWARE CINDERELLA RECALL")
print("=" * 100)

for row_number, row in enumerate(
    matched_pairs.itertuples(index=False),
    start=1,
):
    diagnosis = row.diagnosis
    recording_id = row.recording_id
    key = (diagnosis, recording_id)

    safe_diagnosis = diagnosis.replace(" ", "_")

    audio_record = audio_records[key]
    transcript_record = transcript_records[key]

    audio_suffix = Path(audio_record["name"]).suffix.lower()

    local_audio = (
        DOWNLOAD_ROOT
        / safe_diagnosis
        / "audio"
        / f"{recording_id}{audio_suffix}"
    )

    local_cha = (
        DOWNLOAD_ROOT
        / safe_diagnosis
        / "transcripts"
        / f"{recording_id}.cha"
    )

    output_audio = (
        PAR_AUDIO_ROOT
        / safe_diagnosis
        / f"{recording_id}_Cinderella_PAR.wav"
    )

    output_transcript = (
        TRANSCRIPT_ROOT
        / safe_diagnosis
        / f"{recording_id}_Cinderella.txt"
    )

    print(
        f"[{row_number}/{len(matched_pairs)}] "
        f"{diagnosis} | {recording_id}"
    )

    status = "unknown"
    error_message = ""

    try:
        if not local_audio.exists():
            download_drive_file(
                audio_record["file_id"],
                local_audio,
            )

        if not local_cha.exists():
            download_drive_file(
                transcript_record["file_id"],
                local_cha,
            )

        cha_text = local_cha.read_text(
            encoding="utf-8",
            errors="replace",
        )

        parsed = parse_cinderella_par_utterances(
            cha_text
        )

        status = parsed["status"]
        utterances = parsed["utterances"]

        if status != "ok":
            raise ValueError(status)

        audio_result = create_participant_only_audio(
            source_audio_path=local_audio,
            utterances=utterances,
            output_path=output_audio,
        )

        clean_utterances = [
            item["clean_text"]
            for item in utterances
            if item["clean_text"]
        ]

        clean_transcript = "\n".join(clean_utterances)

        output_transcript.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        output_transcript.write_text(
            clean_transcript.strip() + "\n",
            encoding="utf-8",
        )

        subject_id = recording_id.split("-", 1)[0]

        manifest_rows.append({
            "dataset": "Delaware",
            "task": "Cinderella_recall",
            "diagnosis": diagnosis,
            "binary_label": (
                0 if diagnosis == "Control"
                else 1 if diagnosis == "MCI"
                else None
            ),
            "subject_id": subject_id,
            "recording_id": recording_id,
            "visit": (
                recording_id.split("-", 1)[1]
                if "-" in recording_id
                else None
            ),
            "audio_path": str(
                output_audio.relative_to(OUTPUT_ROOT)
            ),
            "transcript_path": str(
                output_transcript.relative_to(OUTPUT_ROOT)
            ),
            "utterance_count": len(utterances),
            "word_count": len(
                clean_transcript.split()
            ),
            "source_duration_seconds": round(
                audio_result["source_duration_ms"] / 1000,
                3,
            ),
            "par_audio_duration_seconds": round(
                audio_result["output_duration_ms"] / 1000,
                3,
            ),
            "first_par_timestamp_ms": min(
                item["start_ms"]
                for item in utterances
            ),
            "last_par_timestamp_ms": max(
                item["end_ms"]
                for item in utterances
            ),
            "source_audio_drive_path": (
                audio_record["relative_path"]
            ),
            "source_transcript_drive_path": (
                transcript_record["relative_path"]
            ),
            "status": "processed",
            "error": "",
        })

        for utterance_number, item in enumerate(
            utterances,
            start=1,
        ):
            utterance_audit_rows.append({
                "diagnosis": diagnosis,
                "subject_id": subject_id,
                "recording_id": recording_id,
                "utterance_number": utterance_number,
                "start_ms": item["start_ms"],
                "end_ms": item["end_ms"],
                "duration_ms": item["duration_ms"],
                "clean_text": item["clean_text"],
                "raw_text": item["raw_text"],
            })

        print(
            "  Processed:",
            len(utterances),
            "PAR utterances,",
            f"{audio_result['output_duration_ms'] / 1000:.1f}s"
        )

    except Exception as exception:
        error_message = str(exception)

        manifest_rows.append({
            "dataset": "Delaware",
            "task": "Cinderella_recall",
            "diagnosis": diagnosis,
            "binary_label": (
                0 if diagnosis == "Control"
                else 1 if diagnosis == "MCI"
                else None
            ),
            "subject_id": recording_id.split("-", 1)[0],
            "recording_id": recording_id,
            "visit": (
                recording_id.split("-", 1)[1]
                if "-" in recording_id
                else None
            ),
            "audio_path": None,
            "transcript_path": None,
            "utterance_count": 0,
            "word_count": 0,
            "source_duration_seconds": None,
            "par_audio_duration_seconds": None,
            "first_par_timestamp_ms": None,
            "last_par_timestamp_ms": None,
            "source_audio_drive_path": (
                audio_record["relative_path"]
            ),
            "source_transcript_drive_path": (
                transcript_record["relative_path"]
            ),
            "status": "failed",
            "error": error_message,
        })

        print("  FAILED:", error_message)


# ============================================================
# SAVE MANIFESTS AND QUALITY REPORT
# ============================================================

manifest = pd.DataFrame(manifest_rows)
utterance_audit = pd.DataFrame(utterance_audit_rows)

manifest_path = OUTPUT_ROOT / "delaware_recall_manifest.csv"
utterance_audit_path = (
    AUDIT_ROOT / "cinderella_utterance_audit.csv"
)

manifest.to_csv(
    manifest_path,
    index=False,
)

utterance_audit.to_csv(
    utterance_audit_path,
    index=False,
)

processed = manifest[
    manifest["status"] == "processed"
].copy()

summary = {
    "dataset": "Delaware",
    "task": "Cinderella recall",
    "target": "MCI versus Control",
    "audio_type": "participant-only concatenated WAV",
    "sample_rate_hz": 16000,
    "total_matched_recordings": int(len(matched_pairs)),
    "processed_recordings": int(
        (manifest["status"] == "processed").sum()
    ),
    "failed_recordings": int(
        (manifest["status"] == "failed").sum()
    ),
    "unique_subjects_processed": int(
        processed["subject_id"].nunique()
    ),
    "diagnosis_recording_counts": (
        processed["diagnosis"]
        .value_counts()
        .to_dict()
    ),
    "diagnosis_subject_counts": (
        processed.groupby("diagnosis")["subject_id"]
        .nunique()
        .to_dict()
    ),
}

summary_path = OUTPUT_ROOT / "processing_summary.json"

summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print()
print("=" * 100)
print("PROCESSING SUMMARY")
print("=" * 100)
print(json.dumps(summary, indent=2))

if len(processed):
    display(
        processed.groupby("diagnosis").agg(
            recordings=("recording_id", "count"),
            unique_subjects=("subject_id", "nunique"),
            mean_audio_seconds=(
                "par_audio_duration_seconds",
                "mean",
            ),
            mean_words=("word_count", "mean"),
        ).reset_index()
    )

if (manifest["status"] == "failed").any():
    display(
        manifest.loc[
            manifest["status"] == "failed",
            [
                "diagnosis",
                "recording_id",
                "error",
            ],
        ]
    )


# ============================================================
# ZIP PROCESSED DATASET
# ============================================================

zip_base = LOCAL_ROOT / "Delaware_Recall_Processed"

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUTPUT_ROOT.parent,
    base_dir=OUTPUT_ROOT.name,
)

print("Created ZIP:", zip_path)


# ============================================================
# UPLOAD RESULTS TO THE SAME LINKED DRIVE FOLDER
# ============================================================

processed_drive_folder_id = get_or_create_folder(
    DELAWARE_FOLDER_ID,
    "Delaware_Recall_Processed",
)

upload_results = []

for local_path in [
    manifest_path,
    summary_path,
    AUDIT_ROOT / "drive_file_pairing.csv",
    utterance_audit_path,
    Path(zip_path),
]:
    result = upload_file(
        local_path,
        processed_drive_folder_id,
    )
    upload_results.append(result)
    print("Uploaded:", result["name"])

print()
print("=" * 100)
print("DONE")
print("=" * 100)
print("Local output:", OUTPUT_ROOT)
print(
    "Drive output folder created under linked folder:",
    "Delaware_Recall_Processed"
)

In [ ]:
# ============================================================
# DELAWARE RECALL QUALITY CONTROL + FINAL MODELING DATASET
# Uses only the Google Drive folder ID
# ============================================================

!pip -q install pydub google-api-python-client

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

from pathlib import Path
from pydub import AudioSegment
from pydub.silence import detect_nonsilent

import hashlib
import json
import math
import re
import shutil
import zipfile
import pandas as pd
import numpy as np


# ============================================================
# CONFIGURATION
# ============================================================

DELAWARE_FOLDER_ID = "1RV-tYijUIOBV4O7jb8RcvnWC3RdV8U5J"

LOCAL_ROOT = Path("/content/delaware_recall_qc")
INPUT_ROOT = LOCAL_ROOT / "input"
EXTRACT_ROOT = INPUT_ROOT / "extracted"
FINAL_ROOT = LOCAL_ROOT / "Delaware_Recall_Final"

FINAL_AUDIO_ROOT = FINAL_ROOT / "audio"
FINAL_TRANSCRIPT_ROOT = FINAL_ROOT / "transcripts"
AUDIT_ROOT = FINAL_ROOT / "audit"

for path in [
    INPUT_ROOT,
    EXTRACT_ROOT,
    FINAL_ROOT,
    FINAL_AUDIO_ROOT,
    FINAL_TRANSCRIPT_ROOT,
    AUDIT_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)


# These thresholds create review flags.
# They do not automatically prove a recording is unusable.
REVIEW_MIN_DURATION_SECONDS = 30
REVIEW_MAX_DURATION_SECONDS = 480
REVIEW_MIN_WORDS = 20
REVIEW_MAX_WORDS = 1500

# These are hard integrity requirements.
HARD_MIN_DURATION_SECONDS = 10
HARD_MIN_WORDS = 3
MAX_SILENCE_PROPORTION = 0.98

# Select only one visit per subject for the first model.
ONE_VISIT_PER_SUBJECT = True

drive_service = build("drive", "v3")


# ============================================================
# DRIVE UTILITIES
# ============================================================

def list_folder(folder_id):
    items = []
    page_token = None

    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields=(
                "nextPageToken,"
                "files(id,name,mimeType,size,modifiedTime,parents)"
            ),
            pageToken=page_token,
            pageSize=1000,
        ).execute()

        items.extend(response.get("files", []))
        page_token = response.get("nextPageToken")

        if not page_token:
            break

    return items


def walk_drive_folder(folder_id, relative_path=Path()):
    records = []

    for item in list_folder(folder_id):
        current_relative_path = relative_path / item["name"]

        if item["mimeType"] == "application/vnd.google-apps.folder":
            records.extend(
                walk_drive_folder(
                    item["id"],
                    current_relative_path,
                )
            )
        else:
            records.append({
                "file_id": item["id"],
                "name": item["name"],
                "relative_path": str(current_relative_path),
                "mime_type": item["mimeType"],
                "size": int(item.get("size", 0)),
                "modified_time": item.get("modifiedTime"),
            })

    return records


def download_drive_file(file_id, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    request = drive_service.files().get_media(fileId=file_id)

    with destination.open("wb") as output_file:
        downloader = MediaIoBaseDownload(
            output_file,
            request,
            chunksize=16 * 1024 * 1024,
        )

        done = False

        while not done:
            _, done = downloader.next_chunk()

    return destination


def get_or_create_folder(parent_id, folder_name):
    safe_name = folder_name.replace("'", "\\'")

    response = drive_service.files().list(
        q=(
            f"'{parent_id}' in parents and "
            f"name = '{safe_name}' and "
            "mimeType = 'application/vnd.google-apps.folder' and "
            "trashed = false"
        ),
        fields="files(id,name)",
    ).execute()

    matches = response.get("files", [])

    if matches:
        return matches[0]["id"]

    metadata = {
        "name": folder_name,
        "mimeType": "application/vnd.google-apps.folder",
        "parents": [parent_id],
    }

    created = drive_service.files().create(
        body=metadata,
        fields="id",
    ).execute()

    return created["id"]


def upload_file(local_path, parent_id, drive_name=None):
    local_path = Path(local_path)
    drive_name = drive_name or local_path.name

    metadata = {
        "name": drive_name,
        "parents": [parent_id],
    }

    media = MediaFileUpload(
        str(local_path),
        resumable=True,
    )

    return drive_service.files().create(
        body=metadata,
        media_body=media,
        fields="id,name",
    ).execute()


# ============================================================
# FIND THE PROCESSED ZIP
# ============================================================

print("=" * 100)
print("SEARCHING LINKED DRIVE FOLDER")
print("=" * 100)

drive_records = walk_drive_folder(DELAWARE_FOLDER_ID)

zip_candidates = [
    record
    for record in drive_records
    if record["name"].lower() == "delaware_recall_processed.zip"
]

if not zip_candidates:
    raise FileNotFoundError(
        "Could not find Delaware_Recall_Processed.zip "
        "inside the linked Google Drive folder."
    )

# Prefer the most recently modified copy if duplicates exist.
zip_candidates = sorted(
    zip_candidates,
    key=lambda item: item.get("modified_time") or "",
    reverse=True,
)

selected_zip_record = zip_candidates[0]

print("Using ZIP:")
print(selected_zip_record["relative_path"])

local_zip_path = INPUT_ROOT / "Delaware_Recall_Processed.zip"

download_drive_file(
    selected_zip_record["file_id"],
    local_zip_path,
)

print("Downloaded:", local_zip_path)


# ============================================================
# EXTRACT ZIP
# ============================================================

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(local_zip_path, "r") as zip_file:
    zip_file.extractall(EXTRACT_ROOT)

possible_project_roots = [
    path.parent
    for path in EXTRACT_ROOT.rglob("delaware_recall_manifest.csv")
]

if not possible_project_roots:
    raise FileNotFoundError(
        "The ZIP does not contain delaware_recall_manifest.csv."
    )

PROCESSED_ROOT = possible_project_roots[0]

manifest_path = PROCESSED_ROOT / "delaware_recall_manifest.csv"

print("Processed project root:")
print(PROCESSED_ROOT)


# ============================================================
# LOAD MANIFEST
# ============================================================

manifest = pd.read_csv(manifest_path)

required_columns = {
    "diagnosis",
    "subject_id",
    "recording_id",
    "audio_path",
    "transcript_path",
    "status",
}

missing_columns = required_columns - set(manifest.columns)

if missing_columns:
    raise ValueError(
        f"Manifest is missing columns: {sorted(missing_columns)}"
    )

manifest["subject_id"] = manifest["subject_id"].astype(str)
manifest["recording_id"] = manifest["recording_id"].astype(str)

print()
print("Manifest rows:", len(manifest))
print(
    "Originally processed:",
    int((manifest["status"] == "processed").sum()),
)
print(
    "Originally failed:",
    int((manifest["status"] != "processed").sum()),
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def safe_float(value):
    try:
        value = float(value)

        if math.isnan(value):
            return None

        return value

    except Exception:
        return None


def parse_visit_number(recording_id):
    """
    Examples:
    01-1 -> 1
    01-2 -> 2
    385-1 -> 1
    """
    match = re.search(r"-(\d+)$", str(recording_id))

    if not match:
        return 999999

    return int(match.group(1))


def compute_audio_metrics(audio_path):
    audio = AudioSegment.from_file(audio_path)

    duration_seconds = len(audio) / 1000
    sample_rate = audio.frame_rate
    channels = audio.channels
    sample_width_bytes = audio.sample_width
    dbfs = audio.dBFS

    # Detect sections that are not silent.
    nonsilent_ranges = detect_nonsilent(
        audio,
        min_silence_len=500,
        silence_thresh=-45,
        seek_step=20,
    )

    nonsilent_ms = sum(
        end_ms - start_ms
        for start_ms, end_ms in nonsilent_ranges
    )

    silence_proportion = (
        1 - nonsilent_ms / len(audio)
        if len(audio) > 0
        else 1
    )

    return {
        "actual_duration_seconds": duration_seconds,
        "sample_rate_hz": sample_rate,
        "channels": channels,
        "sample_width_bytes": sample_width_bytes,
        "dbfs": dbfs,
        "silence_proportion": silence_proportion,
    }


def count_transcript_words(text):
    words = re.findall(
        r"\b[\w'-]+\b",
        text,
        flags=re.UNICODE,
    )

    return len(words)


def join_flags(flags):
    return ";".join(flags)


# ============================================================
# QUALITY-CHECK EVERY MANIFEST ROW
# ============================================================

print()
print("=" * 100)
print("RUNNING QUALITY CHECKS")
print("=" * 100)

quality_rows = []

for row_number, row in enumerate(
    manifest.itertuples(index=False),
    start=1,
):
    diagnosis = str(row.diagnosis)
    subject_id = str(row.subject_id)
    recording_id = str(row.recording_id)
    original_status = str(row.status)

    review_flags = []
    hard_failure_flags = []

    audio_file = None
    transcript_file = None
    transcript_text = ""
    actual_word_count = None

    audio_metrics = {
        "actual_duration_seconds": None,
        "sample_rate_hz": None,
        "channels": None,
        "sample_width_bytes": None,
        "dbfs": None,
        "silence_proportion": None,
    }

    if original_status != "processed":
        hard_failure_flags.append(
            f"original_extraction_failed:{getattr(row, 'error', '')}"
        )

    if original_status == "processed":
        if pd.isna(row.audio_path):
            hard_failure_flags.append("missing_audio_path")
        else:
            audio_file = PROCESSED_ROOT / str(row.audio_path)

            if not audio_file.exists():
                hard_failure_flags.append("audio_file_missing")
            else:
                try:
                    audio_metrics = compute_audio_metrics(audio_file)
                except Exception as exception:
                    hard_failure_flags.append(
                        f"audio_read_error:{type(exception).__name__}"
                    )

        if pd.isna(row.transcript_path):
            hard_failure_flags.append("missing_transcript_path")
        else:
            transcript_file = PROCESSED_ROOT / str(row.transcript_path)

            if not transcript_file.exists():
                hard_failure_flags.append("transcript_file_missing")
            else:
                try:
                    transcript_text = transcript_file.read_text(
                        encoding="utf-8",
                        errors="replace",
                    ).strip()

                    actual_word_count = count_transcript_words(
                        transcript_text
                    )

                except Exception as exception:
                    hard_failure_flags.append(
                        f"transcript_read_error:{type(exception).__name__}"
                    )

    duration_seconds = audio_metrics["actual_duration_seconds"]
    silence_proportion = audio_metrics["silence_proportion"]

    # Hard quality failures.
    if duration_seconds is not None:
        if duration_seconds < HARD_MIN_DURATION_SECONDS:
            hard_failure_flags.append("audio_under_10_seconds")

    if actual_word_count is not None:
        if actual_word_count < HARD_MIN_WORDS:
            hard_failure_flags.append("transcript_under_3_words")

    if silence_proportion is not None:
        if silence_proportion > MAX_SILENCE_PROPORTION:
            hard_failure_flags.append("audio_almost_entirely_silent")

    # Manual-review flags.
    if duration_seconds is not None:
        if duration_seconds < REVIEW_MIN_DURATION_SECONDS:
            review_flags.append("shorter_than_30_seconds")

        if duration_seconds > REVIEW_MAX_DURATION_SECONDS:
            review_flags.append("longer_than_8_minutes")

    if actual_word_count is not None:
        if actual_word_count < REVIEW_MIN_WORDS:
            review_flags.append("fewer_than_20_words")

        if actual_word_count > REVIEW_MAX_WORDS:
            review_flags.append("more_than_1500_words")

    if audio_metrics["sample_rate_hz"] not in {16000, None}:
        review_flags.append("sample_rate_not_16000_hz")

    if audio_metrics["channels"] not in {1, None}:
        review_flags.append("audio_not_mono")

    if silence_proportion is not None:
        if silence_proportion > 0.80:
            review_flags.append("over_80_percent_silence")

    if (
        duration_seconds is not None
        and actual_word_count is not None
        and duration_seconds > 0
    ):
        words_per_minute = (
            actual_word_count / duration_seconds * 60
        )

        if words_per_minute < 20:
            review_flags.append("speech_rate_under_20_wpm")

        if words_per_minute > 250:
            review_flags.append("speech_rate_over_250_wpm")
    else:
        words_per_minute = None

    hard_pass = len(hard_failure_flags) == 0
    manual_review = len(review_flags) > 0

    quality_rows.append({
        "dataset": "Delaware",
        "task": "Cinderella_recall",
        "diagnosis": diagnosis,
        "binary_label": (
            0 if diagnosis == "Control"
            else 1 if diagnosis == "MCI"
            else None
        ),
        "subject_id": subject_id,
        "recording_id": recording_id,
        "visit_number": parse_visit_number(recording_id),
        "original_status": original_status,
        "hard_pass": hard_pass,
        "manual_review": manual_review,
        "hard_failure_flags": join_flags(hard_failure_flags),
        "review_flags": join_flags(review_flags),
        "actual_duration_seconds": duration_seconds,
        "actual_word_count": actual_word_count,
        "words_per_minute": words_per_minute,
        "sample_rate_hz": audio_metrics["sample_rate_hz"],
        "channels": audio_metrics["channels"],
        "sample_width_bytes": audio_metrics["sample_width_bytes"],
        "dbfs": audio_metrics["dbfs"],
        "silence_proportion": silence_proportion,
        "source_audio_path": (
            str(audio_file)
            if audio_file is not None
            else None
        ),
        "source_transcript_path": (
            str(transcript_file)
            if transcript_file is not None
            else None
        ),
    })

    if row_number % 50 == 0 or row_number == len(manifest):
        print(f"Checked {row_number}/{len(manifest)}")


quality_report = pd.DataFrame(quality_rows)

quality_report_path = AUDIT_ROOT / "quality_report_all_recordings.csv"

quality_report.to_csv(
    quality_report_path,
    index=False,
)


# ============================================================
# SELECT ONE VISIT PER SUBJECT
# ============================================================

eligible = quality_report[
    quality_report["hard_pass"]
].copy()

# Prefer recordings that do not need manual review.
# Within that group, choose the earliest visit.
eligible["review_priority"] = (
    eligible["manual_review"]
    .astype(int)
)

eligible = eligible.sort_values(
    [
        "diagnosis",
        "subject_id",
        "review_priority",
        "visit_number",
        "recording_id",
    ]
)

if ONE_VISIT_PER_SUBJECT:
    selected = (
        eligible
        .groupby(
            ["diagnosis", "subject_id"],
            as_index=False,
            sort=False,
        )
        .first()
    )
else:
    selected = eligible.copy()

selected_recording_ids = set(
    zip(
        selected["diagnosis"],
        selected["recording_id"],
    )
)

quality_report["selected_for_model"] = quality_report.apply(
    lambda row: (
        row["diagnosis"],
        row["recording_id"],
    ) in selected_recording_ids,
    axis=1,
)

quality_report.to_csv(
    quality_report_path,
    index=False,
)


# ============================================================
# COPY SELECTED DATA
# ============================================================

print()
print("=" * 100)
print("BUILDING FINAL MODELING DATASET")
print("=" * 100)

final_manifest_rows = []

for row_number, row in enumerate(
    selected.itertuples(index=False),
    start=1,
):
    diagnosis = str(row.diagnosis)
    subject_id = str(row.subject_id)
    recording_id = str(row.recording_id)

    source_audio = Path(row.source_audio_path)
    source_transcript = Path(row.source_transcript_path)

    destination_audio = (
        FINAL_AUDIO_ROOT
        / diagnosis
        / f"{recording_id}_Cinderella_PAR.wav"
    )

    destination_transcript = (
        FINAL_TRANSCRIPT_ROOT
        / diagnosis
        / f"{recording_id}_Cinderella.txt"
    )

    destination_audio.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    destination_transcript.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        source_audio,
        destination_audio,
    )

    shutil.copy2(
        source_transcript,
        destination_transcript,
    )

    final_manifest_rows.append({
        "dataset": "Delaware",
        "task": "Cinderella_recall",
        "diagnosis": diagnosis,
        "binary_label": (
            0 if diagnosis == "Control"
            else 1 if diagnosis == "MCI"
            else None
        ),
        "subject_id": subject_id,
        "recording_id": recording_id,
        "visit_number": row.visit_number,
        "audio_path": str(
            destination_audio.relative_to(FINAL_ROOT)
        ),
        "transcript_path": str(
            destination_transcript.relative_to(FINAL_ROOT)
        ),
        "duration_seconds": row.actual_duration_seconds,
        "word_count": row.actual_word_count,
        "words_per_minute": row.words_per_minute,
        "dbfs": row.dbfs,
        "silence_proportion": row.silence_proportion,
        "manual_review": row.manual_review,
        "review_flags": row.review_flags,
        "audio_sha256": sha256_file(destination_audio),
        "transcript_sha256": sha256_file(
            destination_transcript
        ),
    })

    if row_number % 50 == 0 or row_number == len(selected):
        print(f"Copied {row_number}/{len(selected)}")


final_manifest = pd.DataFrame(final_manifest_rows)

final_manifest_path = FINAL_ROOT / "recall_final_manifest.csv"

final_manifest.to_csv(
    final_manifest_path,
    index=False,
)


# ============================================================
# SAVE EXCLUSION AND REVIEW REPORTS
# ============================================================

excluded_recordings = quality_report[
    ~quality_report["selected_for_model"]
].copy()

excluded_recordings["exclusion_reason"] = np.where(
    ~excluded_recordings["hard_pass"],
    excluded_recordings["hard_failure_flags"],
    "additional_visit_not_selected",
)

excluded_path = AUDIT_ROOT / "excluded_recordings.csv"

excluded_recordings.to_csv(
    excluded_path,
    index=False,
)

manual_review_table = final_manifest[
    final_manifest["manual_review"]
].copy()

manual_review_path = AUDIT_ROOT / "selected_manual_review.csv"

manual_review_table.to_csv(
    manual_review_path,
    index=False,
)

failed_extraction_table = quality_report[
    quality_report["original_status"] != "processed"
].copy()

failed_extraction_path = (
    AUDIT_ROOT / "failed_cinderella_extractions.csv"
)

failed_extraction_table.to_csv(
    failed_extraction_path,
    index=False,
)


# ============================================================
# DATASET SUMMARY
# ============================================================

summary = {
    "dataset": "Delaware",
    "task": "Cinderella recall",
    "classification_target": "MCI versus Control",
    "selection_rule": (
        "One recording per subject. Prefer no-review recordings, "
        "then choose earliest visit."
    ),
    "original_manifest_rows": int(len(manifest)),
    "hard_quality_pass_recordings": int(
        quality_report["hard_pass"].sum()
    ),
    "hard_quality_failed_recordings": int(
        (~quality_report["hard_pass"]).sum()
    ),
    "selected_recordings": int(len(final_manifest)),
    "selected_unique_subjects": int(
        final_manifest["subject_id"].nunique()
    ),
    "selected_diagnosis_recordings": {
        str(key): int(value)
        for key, value in (
            final_manifest["diagnosis"]
            .value_counts()
            .to_dict()
            .items()
        )
    },
    "selected_diagnosis_subjects": {
        str(key): int(value)
        for key, value in (
            final_manifest.groupby("diagnosis")["subject_id"]
            .nunique()
            .to_dict()
            .items()
        )
    },
    "selected_manual_review_count": int(
        final_manifest["manual_review"].sum()
    ),
    "failed_original_extraction_count": int(
        (
            quality_report["original_status"]
            != "processed"
        ).sum()
    ),
    "quality_thresholds": {
        "hard_minimum_duration_seconds": (
            HARD_MIN_DURATION_SECONDS
        ),
        "hard_minimum_words": HARD_MIN_WORDS,
        "hard_maximum_silence_proportion": (
            MAX_SILENCE_PROPORTION
        ),
        "review_minimum_duration_seconds": (
            REVIEW_MIN_DURATION_SECONDS
        ),
        "review_maximum_duration_seconds": (
            REVIEW_MAX_DURATION_SECONDS
        ),
        "review_minimum_words": REVIEW_MIN_WORDS,
        "review_maximum_words": REVIEW_MAX_WORDS,
    },
}

summary_path = FINAL_ROOT / "quality_summary.json"

summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print()
print("=" * 100)
print("FINAL SUMMARY")
print("=" * 100)
print(json.dumps(summary, indent=2))

print()
print("Selected class distribution:")
display(
    final_manifest.groupby("diagnosis").agg(
        recordings=("recording_id", "count"),
        subjects=("subject_id", "nunique"),
        mean_duration_seconds=("duration_seconds", "mean"),
        median_duration_seconds=("duration_seconds", "median"),
        mean_word_count=("word_count", "mean"),
        median_word_count=("word_count", "median"),
        manual_review_count=("manual_review", "sum"),
    ).reset_index()
)

print()
print("Review flags among selected recordings:")
review_flag_counts = (
    final_manifest.loc[
        final_manifest["review_flags"].fillna("") != "",
        "review_flags",
    ]
    .str.split(";")
    .explode()
    .value_counts()
    .reset_index()
)

review_flag_counts.columns = ["review_flag", "count"]

display(review_flag_counts)


# ============================================================
# CREATE FILE MANIFEST
# ============================================================

file_manifest_rows = []

for path in sorted(FINAL_ROOT.rglob("*")):
    if not path.is_file():
        continue

    file_manifest_rows.append({
        "relative_path": str(
            path.relative_to(FINAL_ROOT)
        ),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

file_manifest = pd.DataFrame(file_manifest_rows)

file_manifest_path = FINAL_ROOT / "file_manifest.csv"

file_manifest.to_csv(
    file_manifest_path,
    index=False,
)


# ============================================================
# CREATE ZIP BACKUP
# ============================================================

zip_base = LOCAL_ROOT / "Delaware_Recall_Final"

final_zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=FINAL_ROOT.parent,
    base_dir=FINAL_ROOT.name,
)

print()
print("Created final ZIP:")
print(final_zip_path)


# ============================================================
# UPLOAD FINAL OUTPUT TO LINKED DRIVE FOLDER
# ============================================================

output_drive_folder_id = get_or_create_folder(
    DELAWARE_FOLDER_ID,
    "Delaware_Recall_Final",
)

upload_paths = [
    Path(final_zip_path),
    final_manifest_path,
    summary_path,
    quality_report_path,
    excluded_path,
    manual_review_path,
    failed_extraction_path,
    file_manifest_path,
]

print()
print("=" * 100)
print("UPLOADING FINAL OUTPUT")
print("=" * 100)

for local_path in upload_paths:
    result = upload_file(
        local_path,
        output_drive_folder_id,
    )

    print("Uploaded:", result["name"])


# ============================================================
# FINAL VERIFICATION
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)
print("Final local folder:")
print(FINAL_ROOT)

print()
print("Final Drive folder:")
print("Delaware_Recall_Final")

print()
print("Main files:")
print("- Delaware_Recall_Final.zip")
print("- recall_final_manifest.csv")
print("- quality_summary.json")
print("- quality_report_all_recordings.csv")
print("- excluded_recordings.csv")
print("- selected_manual_review.csv")
print("- failed_cinderella_extractions.csv")
print("- file_manifest.csv")

In [ ]:
# ============================================================
# FIX CROSS-DIAGNOSIS SUBJECT DUPLICATION
# ONE RECORDING TOTAL PER PERSON
# Uses only the Drive folder ID
# ============================================================

!pip -q install google-api-python-client

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

from pathlib import Path
import hashlib
import json
import re
import shutil
import zipfile
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DELAWARE_FOLDER_ID = "1RV-tYijUIOBV4O7jb8RcvnWC3RdV8U5J"

LOCAL_ROOT = Path("/content/delaware_recall_subject_fix")
INPUT_ROOT = LOCAL_ROOT / "input"
EXTRACT_ROOT = INPUT_ROOT / "extracted"

CORRECTED_ROOT = (
    LOCAL_ROOT
    / "Delaware_Recall_Final_Subject_Unique"
)

CORRECTED_AUDIO_ROOT = CORRECTED_ROOT / "audio"
CORRECTED_TRANSCRIPT_ROOT = CORRECTED_ROOT / "transcripts"
AUDIT_ROOT = CORRECTED_ROOT / "audit"

for path in [
    INPUT_ROOT,
    CORRECTED_ROOT,
    CORRECTED_AUDIO_ROOT,
    CORRECTED_TRANSCRIPT_ROOT,
    AUDIT_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

drive_service = build("drive", "v3")


# ============================================================
# DRIVE UTILITIES
# ============================================================

def list_folder(folder_id):
    items = []
    page_token = None

    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields=(
                "nextPageToken,"
                "files(id,name,mimeType,size,modifiedTime)"
            ),
            pageToken=page_token,
            pageSize=1000,
        ).execute()

        items.extend(response.get("files", []))
        page_token = response.get("nextPageToken")

        if not page_token:
            break

    return items


def walk_drive_folder(folder_id, relative_path=Path()):
    records = []

    for item in list_folder(folder_id):
        current_path = relative_path / item["name"]

        if item["mimeType"] == "application/vnd.google-apps.folder":
            records.extend(
                walk_drive_folder(
                    item["id"],
                    current_path,
                )
            )
        else:
            records.append({
                "file_id": item["id"],
                "name": item["name"],
                "relative_path": str(current_path),
                "mime_type": item["mimeType"],
                "modified_time": item.get("modifiedTime"),
            })

    return records


def download_drive_file(file_id, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    request = drive_service.files().get_media(
        fileId=file_id
    )

    with destination.open("wb") as output_file:
        downloader = MediaIoBaseDownload(
            output_file,
            request,
            chunksize=16 * 1024 * 1024,
        )

        done = False

        while not done:
            _, done = downloader.next_chunk()

    return destination


def get_or_create_folder(parent_id, folder_name):
    safe_name = folder_name.replace("'", "\\'")

    response = drive_service.files().list(
        q=(
            f"'{parent_id}' in parents and "
            f"name = '{safe_name}' and "
            "mimeType = 'application/vnd.google-apps.folder' and "
            "trashed = false"
        ),
        fields="files(id,name)",
    ).execute()

    matches = response.get("files", [])

    if matches:
        return matches[0]["id"]

    created = drive_service.files().create(
        body={
            "name": folder_name,
            "mimeType": "application/vnd.google-apps.folder",
            "parents": [parent_id],
        },
        fields="id",
    ).execute()

    return created["id"]


def upload_file(local_path, parent_id):
    local_path = Path(local_path)

    result = drive_service.files().create(
        body={
            "name": local_path.name,
            "parents": [parent_id],
        },
        media_body=MediaFileUpload(
            str(local_path),
            resumable=True,
        ),
        fields="id,name",
    ).execute()

    return result


# ============================================================
# FILE HELPERS
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def parse_visit_number(recording_id):
    match = re.search(
        r"-(\d+)$",
        str(recording_id),
    )

    if match:
        return int(match.group(1))

    return 999999


# ============================================================
# DOWNLOAD PREVIOUS FINAL ZIP
# ============================================================

print("=" * 100)
print("LOCATING PREVIOUS FINAL DATASET")
print("=" * 100)

drive_records = walk_drive_folder(
    DELAWARE_FOLDER_ID
)

zip_candidates = [
    record
    for record in drive_records
    if record["name"].lower()
    == "delaware_recall_final.zip"
]

if not zip_candidates:
    raise FileNotFoundError(
        "Could not find Delaware_Recall_Final.zip."
    )

zip_candidates = sorted(
    zip_candidates,
    key=lambda item: item.get("modified_time") or "",
    reverse=True,
)

selected_zip = zip_candidates[0]

print("Using:")
print(selected_zip["relative_path"])

local_zip = INPUT_ROOT / "Delaware_Recall_Final.zip"

download_drive_file(
    selected_zip["file_id"],
    local_zip,
)

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(local_zip, "r") as archive:
    archive.extractall(EXTRACT_ROOT)

manifest_candidates = list(
    EXTRACT_ROOT.rglob(
        "recall_final_manifest.csv"
    )
)

if not manifest_candidates:
    raise FileNotFoundError(
        "recall_final_manifest.csv was not found."
    )

OLD_ROOT = manifest_candidates[0].parent
old_manifest_path = manifest_candidates[0]

manifest = pd.read_csv(
    old_manifest_path
)

manifest["subject_id"] = (
    manifest["subject_id"]
    .astype(str)
)

manifest["recording_id"] = (
    manifest["recording_id"]
    .astype(str)
)

manifest["visit_number"] = (
    manifest["recording_id"]
    .map(parse_visit_number)
)

print("Rows loaded:", len(manifest))


# ============================================================
# FIND SUBJECTS PRESENT IN BOTH CLASSES
# ============================================================

diagnoses_per_subject = (
    manifest.groupby("subject_id")["diagnosis"]
    .nunique()
)

cross_diagnosis_subjects = sorted(
    diagnoses_per_subject[
        diagnoses_per_subject > 1
    ].index.tolist()
)

cross_diagnosis_table = (
    manifest[
        manifest["subject_id"].isin(
            cross_diagnosis_subjects
        )
    ]
    .sort_values(
        [
            "subject_id",
            "visit_number",
            "recording_id",
        ]
    )
)

cross_diagnosis_path = (
    AUDIT_ROOT
    / "cross_diagnosis_subjects.csv"
)

cross_diagnosis_table.to_csv(
    cross_diagnosis_path,
    index=False,
)

print()
print(
    "Subjects appearing in both classes:",
    len(cross_diagnosis_subjects),
)

display(
    cross_diagnosis_table[
        [
            "subject_id",
            "recording_id",
            "visit_number",
            "diagnosis",
            "duration_seconds",
            "word_count",
            "manual_review",
        ]
    ]
)


# ============================================================
# SELECT EXACTLY ONE RECORDING PER PERSON
# ============================================================

# Rule:
# 1. Prefer a recording without a manual-review flag.
# 2. Select the earliest visit.
# 3. Use recording ID as a deterministic tie-breaker.
#
# Diagnosis remains the diagnosis assigned to that selected
# visit. A person is never included twice.

manifest["review_priority"] = (
    manifest["manual_review"]
    .fillna(False)
    .astype(bool)
    .astype(int)
)

ordered = manifest.sort_values(
    [
        "subject_id",
        "review_priority",
        "visit_number",
        "recording_id",
    ]
)

selected = (
    ordered
    .groupby(
        "subject_id",
        as_index=False,
        sort=False,
    )
    .first()
)

selected_keys = set(
    zip(
        selected["diagnosis"],
        selected["recording_id"],
    )
)

not_selected = manifest[
    ~manifest.apply(
        lambda row: (
            row["diagnosis"],
            row["recording_id"],
        ) in selected_keys,
        axis=1,
    )
].copy()

not_selected["exclusion_reason"] = (
    "additional_visit_or_cross_diagnosis_duplicate"
)

not_selected_path = (
    AUDIT_ROOT
    / "subject_level_excluded_recordings.csv"
)

not_selected.to_csv(
    not_selected_path,
    index=False,
)


# ============================================================
# REBUILD CORRECTED DATASET
# ============================================================

print()
print("=" * 100)
print("BUILDING SUBJECT-UNIQUE DATASET")
print("=" * 100)

corrected_rows = []

for index, row in enumerate(
    selected.itertuples(index=False),
    start=1,
):
    diagnosis = str(row.diagnosis)
    recording_id = str(row.recording_id)
    subject_id = str(row.subject_id)

    source_audio = (
        OLD_ROOT
        / str(row.audio_path)
    )

    source_transcript = (
        OLD_ROOT
        / str(row.transcript_path)
    )

    if not source_audio.exists():
        raise FileNotFoundError(
            f"Missing audio: {source_audio}"
        )

    if not source_transcript.exists():
        raise FileNotFoundError(
            f"Missing transcript: {source_transcript}"
        )

    destination_audio = (
        CORRECTED_AUDIO_ROOT
        / diagnosis
        / source_audio.name
    )

    destination_transcript = (
        CORRECTED_TRANSCRIPT_ROOT
        / diagnosis
        / source_transcript.name
    )

    destination_audio.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    destination_transcript.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        source_audio,
        destination_audio,
    )

    shutil.copy2(
        source_transcript,
        destination_transcript,
    )

    corrected_rows.append({
        "dataset": "Delaware",
        "task": "Cinderella_recall",
        "diagnosis": diagnosis,
        "binary_label": (
            0 if diagnosis == "Control"
            else 1
        ),
        "subject_id": subject_id,
        "recording_id": recording_id,
        "visit_number": int(row.visit_number),
        "audio_path": str(
            destination_audio.relative_to(
                CORRECTED_ROOT
            )
        ),
        "transcript_path": str(
            destination_transcript.relative_to(
                CORRECTED_ROOT
            )
        ),
        "duration_seconds": row.duration_seconds,
        "word_count": row.word_count,
        "words_per_minute": row.words_per_minute,
        "dbfs": row.dbfs,
        "silence_proportion": row.silence_proportion,
        "manual_review": row.manual_review,
        "review_flags": row.review_flags,
        "audio_sha256": sha256_file(
            destination_audio
        ),
        "transcript_sha256": sha256_file(
            destination_transcript
        ),
    })

    if index % 50 == 0 or index == len(selected):
        print(
            f"Copied {index}/{len(selected)}"
        )


corrected_manifest = pd.DataFrame(
    corrected_rows
)

corrected_manifest_path = (
    CORRECTED_ROOT
    / "recall_subject_unique_manifest.csv"
)

corrected_manifest.to_csv(
    corrected_manifest_path,
    index=False,
)


# ============================================================
# VERIFY NO SUBJECT LEAKAGE REMAINS
# ============================================================

duplicate_subject_count = int(
    corrected_manifest[
        "subject_id"
    ].duplicated().sum()
)

cross_label_after_fix = int(
    (
        corrected_manifest
        .groupby("subject_id")["diagnosis"]
        .nunique()
        > 1
    ).sum()
)

if duplicate_subject_count != 0:
    raise RuntimeError(
        "Duplicate subjects remain after correction."
    )

if cross_label_after_fix != 0:
    raise RuntimeError(
        "Cross-label subjects remain after correction."
    )


# ============================================================
# SUMMARY
# ============================================================

summary = {
    "dataset": "Delaware",
    "task": "Cinderella recall",
    "classification_target": "MCI versus Control",
    "selection_rule": (
        "Exactly one recording per person. Prefer an "
        "unflagged recording, then select the earliest visit."
    ),
    "previous_selected_recordings": int(
        len(manifest)
    ),
    "cross_diagnosis_subjects_found": int(
        len(cross_diagnosis_subjects)
    ),
    "corrected_selected_recordings": int(
        len(corrected_manifest)
    ),
    "corrected_unique_subjects": int(
        corrected_manifest[
            "subject_id"
        ].nunique()
    ),
    "duplicate_subjects_after_fix": (
        duplicate_subject_count
    ),
    "cross_label_subjects_after_fix": (
        cross_label_after_fix
    ),
    "class_counts": {
        str(key): int(value)
        for key, value in (
            corrected_manifest[
                "diagnosis"
            ]
            .value_counts()
            .to_dict()
            .items()
        )
    },
    "manual_review_count": int(
        corrected_manifest[
            "manual_review"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    ),
}

summary_path = (
    CORRECTED_ROOT
    / "subject_unique_summary.json"
)

summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 100)
print("CORRECTED SUMMARY")
print("=" * 100)
print(
    json.dumps(
        summary,
        indent=2,
    )
)

display(
    corrected_manifest.groupby(
        "diagnosis"
    ).agg(
        recordings=("recording_id", "count"),
        subjects=("subject_id", "nunique"),
        mean_duration_seconds=(
            "duration_seconds",
            "mean",
        ),
        median_duration_seconds=(
            "duration_seconds",
            "median",
        ),
        mean_word_count=("word_count", "mean"),
        median_word_count=(
            "word_count",
            "median",
        ),
        manual_review_count=(
            "manual_review",
            "sum",
        ),
    ).reset_index()
)


# ============================================================
# FILE MANIFEST
# ============================================================

file_rows = []

for path in sorted(
    CORRECTED_ROOT.rglob("*")
):
    if not path.is_file():
        continue

    file_rows.append({
        "relative_path": str(
            path.relative_to(
                CORRECTED_ROOT
            )
        ),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

file_manifest = pd.DataFrame(file_rows)

file_manifest_path = (
    CORRECTED_ROOT
    / "file_manifest.csv"
)

file_manifest.to_csv(
    file_manifest_path,
    index=False,
)


# ============================================================
# ZIP AND UPLOAD
# ============================================================

corrected_zip_base = (
    LOCAL_ROOT
    / "Delaware_Recall_Final_Subject_Unique"
)

corrected_zip = shutil.make_archive(
    str(corrected_zip_base),
    "zip",
    root_dir=CORRECTED_ROOT.parent,
    base_dir=CORRECTED_ROOT.name,
)

print()
print("Created ZIP:")
print(corrected_zip)

drive_output_folder_id = get_or_create_folder(
    DELAWARE_FOLDER_ID,
    "Delaware_Recall_Final_Subject_Unique",
)

upload_paths = [
    Path(corrected_zip),
    corrected_manifest_path,
    summary_path,
    cross_diagnosis_path,
    not_selected_path,
    file_manifest_path,
]

print()
print("=" * 100)
print("UPLOADING CORRECTED DATASET")
print("=" * 100)

for path in upload_paths:
    result = upload_file(
        path,
        drive_output_folder_id,
    )

    print(
        "Uploaded:",
        result["name"],
    )


print()
print("=" * 100)
print("DONE")
print("=" * 100)
print(
    "Use this dataset for all later modeling:"
)
print(
    "Delaware_Recall_Final_Subject_Unique.zip"
)

In [ ]:
# ============================================================
# DELAWARE RECALL: FINAL LEAKAGE-FREE MODEL SPLITS
# Conservative version: flagged recordings are held out
# Uses only the Google Drive folder ID
# ============================================================

!pip -q install google-api-python-client scikit-learn

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

from sklearn.model_selection import train_test_split

from pathlib import Path

import hashlib
import json
import shutil
import zipfile
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DELAWARE_FOLDER_ID = "1RV-tYijUIOBV4O7jb8RcvnWC3RdV8U5J"

RANDOM_SEED = 42

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

# Conservative first benchmark:
# do not include recordings requiring manual listening.
EXCLUDE_MANUAL_REVIEW = True

LOCAL_ROOT = Path(
    "/content/delaware_recall_model_splits"
)

INPUT_ROOT = LOCAL_ROOT / "input"
EXTRACT_ROOT = INPUT_ROOT / "extracted"

FINAL_ROOT = (
    LOCAL_ROOT
    / "Delaware_Recall_Model_Splits"
)

DATA_ROOT = FINAL_ROOT / "data"
AUDIT_ROOT = FINAL_ROOT / "audit"

for path in [
    INPUT_ROOT,
    EXTRACT_ROOT,
    FINAL_ROOT,
    DATA_ROOT,
    AUDIT_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

drive_service = build(
    "drive",
    "v3",
)


# ============================================================
# GOOGLE DRIVE UTILITIES
# ============================================================

def list_folder(folder_id):
    items = []
    page_token = None

    while True:
        response = drive_service.files().list(
            q=(
                f"'{folder_id}' in parents "
                "and trashed = false"
            ),
            fields=(
                "nextPageToken,"
                "files("
                "id,name,mimeType,size,"
                "modifiedTime,parents"
                ")"
            ),
            pageToken=page_token,
            pageSize=1000,
        ).execute()

        items.extend(
            response.get("files", [])
        )

        page_token = response.get(
            "nextPageToken"
        )

        if not page_token:
            break

    return items


def walk_drive_folder(
    folder_id,
    relative_path=Path(),
):
    records = []

    for item in list_folder(folder_id):
        current_relative_path = (
            relative_path / item["name"]
        )

        if (
            item["mimeType"]
            == "application/vnd.google-apps.folder"
        ):
            records.extend(
                walk_drive_folder(
                    item["id"],
                    current_relative_path,
                )
            )

        else:
            records.append({
                "file_id": item["id"],
                "name": item["name"],
                "relative_path": str(
                    current_relative_path
                ),
                "mime_type": item["mimeType"],
                "size": int(
                    item.get("size", 0)
                ),
                "modified_time": item.get(
                    "modifiedTime"
                ),
            })

    return records


def download_drive_file(
    file_id,
    destination,
):
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    request = (
        drive_service.files()
        .get_media(fileId=file_id)
    )

    with destination.open("wb") as output_file:
        downloader = MediaIoBaseDownload(
            output_file,
            request,
            chunksize=16 * 1024 * 1024,
        )

        done = False

        while not done:
            _, done = downloader.next_chunk()

    return destination


def get_or_create_folder(
    parent_id,
    folder_name,
):
    safe_name = folder_name.replace(
        "'",
        "\\'",
    )

    response = (
        drive_service.files()
        .list(
            q=(
                f"'{parent_id}' in parents and "
                f"name = '{safe_name}' and "
                "mimeType = "
                "'application/vnd.google-apps.folder' "
                "and trashed = false"
            ),
            fields="files(id,name)",
        )
        .execute()
    )

    matches = response.get(
        "files",
        [],
    )

    if matches:
        return matches[0]["id"]

    created = (
        drive_service.files()
        .create(
            body={
                "name": folder_name,
                "mimeType": (
                    "application/"
                    "vnd.google-apps.folder"
                ),
                "parents": [parent_id],
            },
            fields="id",
        )
        .execute()
    )

    return created["id"]


def upload_file(
    local_path,
    parent_id,
):
    local_path = Path(local_path)

    media = MediaFileUpload(
        str(local_path),
        resumable=True,
    )

    result = (
        drive_service.files()
        .create(
            body={
                "name": local_path.name,
                "parents": [parent_id],
            },
            media_body=media,
            fields="id,name",
        )
        .execute()
    )

    return result


# ============================================================
# FILE UTILITIES
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def normalize_boolean(value):
    if pd.isna(value):
        return False

    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
        "y",
    }


# ============================================================
# LOCATE CORRECTED SUBJECT-UNIQUE ZIP
# ============================================================

print("=" * 100)
print("LOCATING SUBJECT-UNIQUE DATASET")
print("=" * 100)

drive_records = walk_drive_folder(
    DELAWARE_FOLDER_ID
)

zip_candidates = [
    record
    for record in drive_records
    if (
        record["name"].lower()
        == (
            "delaware_recall_final_"
            "subject_unique.zip"
        )
    )
]

if not zip_candidates:
    raise FileNotFoundError(
        "Could not find "
        "Delaware_Recall_Final_Subject_Unique.zip "
        "inside the linked Drive folder."
    )

zip_candidates = sorted(
    zip_candidates,
    key=lambda item: (
        item.get("modified_time") or ""
    ),
    reverse=True,
)

selected_zip_record = zip_candidates[0]

print("Using:")
print(
    selected_zip_record[
        "relative_path"
    ]
)

local_zip_path = (
    INPUT_ROOT
    / (
        "Delaware_Recall_Final_"
        "Subject_Unique.zip"
    )
)

download_drive_file(
    selected_zip_record["file_id"],
    local_zip_path,
)

print("Downloaded:")
print(local_zip_path)


# ============================================================
# EXTRACT DATASET
# ============================================================

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(
    local_zip_path,
    "r",
) as archive:
    archive.extractall(
        EXTRACT_ROOT
    )

manifest_candidates = list(
    EXTRACT_ROOT.rglob(
        "recall_subject_unique_manifest.csv"
    )
)

if not manifest_candidates:
    raise FileNotFoundError(
        "The ZIP does not contain "
        "recall_subject_unique_manifest.csv."
    )

source_manifest_path = (
    manifest_candidates[0]
)

SOURCE_ROOT = (
    source_manifest_path.parent
)

print()
print("Dataset root:")
print(SOURCE_ROOT)


# ============================================================
# LOAD AND VALIDATE MANIFEST
# ============================================================

manifest = pd.read_csv(
    source_manifest_path
)

required_columns = {
    "diagnosis",
    "binary_label",
    "subject_id",
    "recording_id",
    "audio_path",
    "transcript_path",
    "manual_review",
}

missing_columns = (
    required_columns
    - set(manifest.columns)
)

if missing_columns:
    raise ValueError(
        "Missing required manifest columns: "
        f"{sorted(missing_columns)}"
    )

manifest["subject_id"] = (
    manifest["subject_id"]
    .astype(str)
)

manifest["recording_id"] = (
    manifest["recording_id"]
    .astype(str)
)

manifest["manual_review"] = (
    manifest["manual_review"]
    .map(normalize_boolean)
)

print()
print("Original subject-unique rows:")
print(len(manifest))

print()
print("Original class distribution:")
print(
    manifest["diagnosis"]
    .value_counts()
)


# ============================================================
# VERIFY SUBJECT UNIQUENESS
# ============================================================

duplicate_subject_rows = manifest[
    manifest["subject_id"].duplicated(
        keep=False
    )
].copy()

if len(duplicate_subject_rows) > 0:
    duplicate_subject_rows.to_csv(
        AUDIT_ROOT
        / "unexpected_duplicate_subjects.csv",
        index=False,
    )

    raise RuntimeError(
        "The source dataset unexpectedly contains "
        "duplicate subject IDs."
    )

cross_label_subject_count = int(
    (
        manifest
        .groupby("subject_id")[
            "diagnosis"
        ]
        .nunique()
        > 1
    ).sum()
)

if cross_label_subject_count != 0:
    raise RuntimeError(
        "Cross-label subjects remain in the "
        "subject-unique dataset."
    )


# ============================================================
# HOLD OUT MANUAL-REVIEW RECORDINGS
# ============================================================

if EXCLUDE_MANUAL_REVIEW:
    modeling_manifest = manifest[
        ~manifest["manual_review"]
    ].copy()

    held_out_review = manifest[
        manifest["manual_review"]
    ].copy()

else:
    modeling_manifest = manifest.copy()

    held_out_review = manifest.iloc[
        0:0
    ].copy()

held_out_review_path = (
    AUDIT_ROOT
    / "manual_review_holdout.csv"
)

held_out_review.to_csv(
    held_out_review_path,
    index=False,
)

print()
print(
    "Manual-review recordings held out:",
    len(held_out_review),
)

if len(held_out_review) > 0:
    display(
        held_out_review[
            [
                "diagnosis",
                "subject_id",
                "recording_id",
                "duration_seconds",
                "word_count",
                "review_flags",
            ]
        ]
    )

print()
print(
    "Subjects available for splitting:",
    len(modeling_manifest),
)

print(
    modeling_manifest["diagnosis"]
    .value_counts()
)


# ============================================================
# VERIFY ALL SOURCE FILES BEFORE SPLITTING
# ============================================================

integrity_rows = []

for row in modeling_manifest.itertuples(
    index=False
):
    source_audio = (
        SOURCE_ROOT
        / str(row.audio_path)
    )

    source_transcript = (
        SOURCE_ROOT
        / str(row.transcript_path)
    )

    integrity_rows.append({
        "subject_id": row.subject_id,
        "recording_id": row.recording_id,
        "diagnosis": row.diagnosis,
        "audio_exists": (
            source_audio.exists()
        ),
        "transcript_exists": (
            source_transcript.exists()
        ),
        "audio_size_bytes": (
            source_audio.stat().st_size
            if source_audio.exists()
            else None
        ),
        "transcript_size_bytes": (
            source_transcript.stat().st_size
            if source_transcript.exists()
            else None
        ),
    })

integrity_before_split = pd.DataFrame(
    integrity_rows
)

integrity_before_split_path = (
    AUDIT_ROOT
    / "source_integrity_before_split.csv"
)

integrity_before_split.to_csv(
    integrity_before_split_path,
    index=False,
)

missing_file_rows = integrity_before_split[
    ~(
        integrity_before_split[
            "audio_exists"
        ]
        & integrity_before_split[
            "transcript_exists"
        ]
    )
]

if len(missing_file_rows) > 0:
    display(missing_file_rows)

    raise FileNotFoundError(
        "One or more source audio or transcript "
        "files are missing."
    )


# ============================================================
# CREATE STRATIFIED TRAIN / VALIDATION / TEST SPLITS
# ============================================================

# First separate 70% train from 30% temporary.
train_df, temporary_df = train_test_split(
    modeling_manifest,
    test_size=(
        VALIDATION_FRACTION
        + TEST_FRACTION
    ),
    stratify=modeling_manifest[
        "diagnosis"
    ],
    random_state=RANDOM_SEED,
)

# Divide the temporary set equally between validation and test.
relative_test_fraction = (
    TEST_FRACTION
    / (
        VALIDATION_FRACTION
        + TEST_FRACTION
    )
)

validation_df, test_df = train_test_split(
    temporary_df,
    test_size=relative_test_fraction,
    stratify=temporary_df[
        "diagnosis"
    ],
    random_state=RANDOM_SEED,
)

split_frames = {
    "train": train_df.copy(),
    "validation": validation_df.copy(),
    "test": test_df.copy(),
}

for split_name, split_df in split_frames.items():
    split_df["split"] = split_name

combined_splits = pd.concat(
    split_frames.values(),
    ignore_index=True,
)

print()
print("=" * 100)
print("SPLIT DISTRIBUTION")
print("=" * 100)

split_distribution = (
    combined_splits.groupby(
        ["split", "diagnosis"]
    )
    .size()
    .reset_index(
        name="recordings"
    )
)

display(split_distribution)


# ============================================================
# VERIFY SPLIT LEAKAGE
# ============================================================

subject_split_counts = (
    combined_splits.groupby(
        "subject_id"
    )["split"]
    .nunique()
)

leaking_subjects = (
    subject_split_counts[
        subject_split_counts > 1
    ]
    .index.tolist()
)

if leaking_subjects:
    leakage_table = combined_splits[
        combined_splits[
            "subject_id"
        ].isin(leaking_subjects)
    ].sort_values(
        [
            "subject_id",
            "split",
        ]
    )

    leakage_table.to_csv(
        AUDIT_ROOT
        / "subject_split_leakage.csv",
        index=False,
    )

    raise RuntimeError(
        f"Subject leakage detected for "
        f"{len(leaking_subjects)} subjects."
    )

if (
    combined_splits[
        "recording_id"
    ].duplicated().any()
):
    raise RuntimeError(
        "A recording appears in multiple splits."
    )

if len(combined_splits) != len(
    modeling_manifest
):
    raise RuntimeError(
        "Split row count does not equal "
        "the modeling dataset row count."
    )


# ============================================================
# COPY FILES INTO SPLIT STRUCTURE
# ============================================================

print()
print("=" * 100)
print("COPYING SPLIT FILES")
print("=" * 100)

final_manifest_rows = []

for row_number, row in enumerate(
    combined_splits.itertuples(
        index=False
    ),
    start=1,
):
    split_name = str(row.split)
    diagnosis = str(row.diagnosis)
    recording_id = str(
        row.recording_id
    )

    source_audio = (
        SOURCE_ROOT
        / str(row.audio_path)
    )

    source_transcript = (
        SOURCE_ROOT
        / str(row.transcript_path)
    )

    destination_audio = (
        DATA_ROOT
        / split_name
        / "audio"
        / diagnosis
        / source_audio.name
    )

    destination_transcript = (
        DATA_ROOT
        / split_name
        / "transcripts"
        / diagnosis
        / source_transcript.name
    )

    destination_audio.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    destination_transcript.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        source_audio,
        destination_audio,
    )

    shutil.copy2(
        source_transcript,
        destination_transcript,
    )

    final_manifest_rows.append({
        "dataset": "Delaware",
        "task": "Cinderella_recall",
        "classification_target": (
            "MCI_vs_Control"
        ),
        "split": split_name,
        "diagnosis": diagnosis,
        "binary_label": int(
            row.binary_label
        ),
        "subject_id": str(
            row.subject_id
        ),
        "recording_id": recording_id,
        "visit_number": int(
            row.visit_number
        ),
        "audio_path": str(
            destination_audio.relative_to(
                FINAL_ROOT
            )
        ),
        "transcript_path": str(
            destination_transcript.relative_to(
                FINAL_ROOT
            )
        ),
        "duration_seconds": float(
            row.duration_seconds
        ),
        "word_count": int(
            row.word_count
        ),
        "words_per_minute": float(
            row.words_per_minute
        ),
        "dbfs": float(row.dbfs),
        "silence_proportion": float(
            row.silence_proportion
        ),
        "manual_review": bool(
            row.manual_review
        ),
        "review_flags": (
            ""
            if pd.isna(row.review_flags)
            else str(row.review_flags)
        ),
        "audio_sha256": sha256_file(
            destination_audio
        ),
        "transcript_sha256": sha256_file(
            destination_transcript
        ),
    })

    if (
        row_number % 50 == 0
        or row_number
        == len(combined_splits)
    ):
        print(
            f"Copied {row_number}/"
            f"{len(combined_splits)}"
        )


final_manifest = pd.DataFrame(
    final_manifest_rows
)

final_manifest_path = (
    FINAL_ROOT
    / "recall_model_split_manifest.csv"
)

final_manifest.to_csv(
    final_manifest_path,
    index=False,
)


# ============================================================
# SAVE INDIVIDUAL SPLIT MANIFESTS
# ============================================================

split_manifest_paths = []

for split_name in [
    "train",
    "validation",
    "test",
]:
    split_manifest = final_manifest[
        final_manifest["split"]
        == split_name
    ].copy()

    split_manifest_path = (
        FINAL_ROOT
        / f"{split_name}_manifest.csv"
    )

    split_manifest.to_csv(
        split_manifest_path,
        index=False,
    )

    split_manifest_paths.append(
        split_manifest_path
    )


# ============================================================
# FINAL INTEGRITY CHECK
# ============================================================

final_integrity_rows = []

for row in final_manifest.itertuples(
    index=False
):
    audio_path = (
        FINAL_ROOT
        / row.audio_path
    )

    transcript_path = (
        FINAL_ROOT
        / row.transcript_path
    )

    audio_exists = (
        audio_path.exists()
    )

    transcript_exists = (
        transcript_path.exists()
    )

    audio_checksum_matches = (
        audio_exists
        and sha256_file(audio_path)
        == row.audio_sha256
    )

    transcript_checksum_matches = (
        transcript_exists
        and sha256_file(
            transcript_path
        )
        == row.transcript_sha256
    )

    final_integrity_rows.append({
        "split": row.split,
        "diagnosis": row.diagnosis,
        "subject_id": row.subject_id,
        "recording_id": row.recording_id,
        "audio_exists": audio_exists,
        "transcript_exists": (
            transcript_exists
        ),
        "audio_checksum_matches": (
            audio_checksum_matches
        ),
        "transcript_checksum_matches": (
            transcript_checksum_matches
        ),
    })

final_integrity = pd.DataFrame(
    final_integrity_rows
)

final_integrity_path = (
    AUDIT_ROOT
    / "final_split_integrity.csv"
)

final_integrity.to_csv(
    final_integrity_path,
    index=False,
)

integrity_columns = [
    "audio_exists",
    "transcript_exists",
    "audio_checksum_matches",
    "transcript_checksum_matches",
]

if not final_integrity[
    integrity_columns
].all().all():
    failed_integrity = (
        final_integrity[
            ~final_integrity[
                integrity_columns
            ].all(axis=1)
        ]
    )

    display(failed_integrity)

    raise RuntimeError(
        "Final file-integrity verification failed."
    )


# ============================================================
# SPLIT STATISTICS
# ============================================================

split_summary_table = (
    final_manifest.groupby(
        ["split", "diagnosis"]
    )
    .agg(
        recordings=(
            "recording_id",
            "count",
        ),
        unique_subjects=(
            "subject_id",
            "nunique",
        ),
        mean_duration_seconds=(
            "duration_seconds",
            "mean",
        ),
        median_duration_seconds=(
            "duration_seconds",
            "median",
        ),
        mean_word_count=(
            "word_count",
            "mean",
        ),
        median_word_count=(
            "word_count",
            "median",
        ),
    )
    .reset_index()
)

split_summary_path = (
    AUDIT_ROOT
    / "split_summary.csv"
)

split_summary_table.to_csv(
    split_summary_path,
    index=False,
)

display(split_summary_table)


# ============================================================
# CREATE SUMMARY JSON
# ============================================================

summary = {
    "dataset": "Delaware",
    "task": "Cinderella recall",
    "classification_target": (
        "MCI versus Control"
    ),
    "random_seed": RANDOM_SEED,
    "requested_split_fractions": {
        "train": TRAIN_FRACTION,
        "validation": (
            VALIDATION_FRACTION
        ),
        "test": TEST_FRACTION,
    },
    "source_subject_unique_rows": int(
        len(manifest)
    ),
    "manual_review_recordings_held_out": int(
        len(held_out_review)
    ),
    "modeling_subjects": int(
        len(final_manifest)
    ),
    "modeling_unique_subjects": int(
        final_manifest[
            "subject_id"
        ].nunique()
    ),
    "duplicate_subjects": int(
        final_manifest[
            "subject_id"
        ].duplicated().sum()
    ),
    "subjects_in_multiple_splits": int(
        (
            final_manifest.groupby(
                "subject_id"
            )["split"].nunique()
            > 1
        ).sum()
    ),
    "class_counts": {
        str(key): int(value)
        for key, value in (
            final_manifest[
                "diagnosis"
            ]
            .value_counts()
            .to_dict()
            .items()
        )
    },
    "split_counts": {
        str(split_name): {
            str(key): int(value)
            for key, value in (
                split_df[
                    "diagnosis"
                ]
                .value_counts()
                .to_dict()
                .items()
            )
        }
        for split_name, split_df
        in final_manifest.groupby(
            "split"
        )
    },
    "integrity": {
        "all_audio_files_exist": bool(
            final_integrity[
                "audio_exists"
            ].all()
        ),
        "all_transcripts_exist": bool(
            final_integrity[
                "transcript_exists"
            ].all()
        ),
        "all_audio_checksums_match": bool(
            final_integrity[
                "audio_checksum_matches"
            ].all()
        ),
        "all_transcript_checksums_match": bool(
            final_integrity[
                "transcript_checksum_matches"
            ].all()
        ),
    },
}

summary_path = (
    FINAL_ROOT
    / "split_summary.json"
)

summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print("=" * 100)
print("FINAL SPLIT SUMMARY")
print("=" * 100)

print(
    json.dumps(
        summary,
        indent=2,
    )
)


# ============================================================
# CREATE COMPLETE FILE MANIFEST
# ============================================================

file_manifest_rows = []

for path in sorted(
    FINAL_ROOT.rglob("*")
):
    if not path.is_file():
        continue

    file_manifest_rows.append({
        "relative_path": str(
            path.relative_to(
                FINAL_ROOT
            )
        ),
        "size_bytes": (
            path.stat().st_size
        ),
        "sha256": sha256_file(path),
    })

file_manifest = pd.DataFrame(
    file_manifest_rows
)

file_manifest_path = (
    FINAL_ROOT
    / "file_manifest.csv"
)

file_manifest.to_csv(
    file_manifest_path,
    index=False,
)


# ============================================================
# CREATE ZIP BACKUP
# ============================================================

zip_base = (
    LOCAL_ROOT
    / "Delaware_Recall_Model_Splits"
)

final_zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=FINAL_ROOT.parent,
    base_dir=FINAL_ROOT.name,
)

print()
print("Created ZIP:")
print(final_zip_path)


# ============================================================
# UPLOAD TO LINKED DRIVE FOLDER
# ============================================================

output_drive_folder_id = (
    get_or_create_folder(
        DELAWARE_FOLDER_ID,
        "Delaware_Recall_Model_Splits",
    )
)

upload_paths = [
    Path(final_zip_path),
    final_manifest_path,
    *split_manifest_paths,
    summary_path,
    held_out_review_path,
    integrity_before_split_path,
    final_integrity_path,
    split_summary_path,
    file_manifest_path,
]

print()
print("=" * 100)
print("UPLOADING MODEL SPLITS")
print("=" * 100)

for local_path in upload_paths:
    result = upload_file(
        local_path,
        output_drive_folder_id,
    )

    print(
        "Uploaded:",
        result["name"],
    )


# ============================================================
# DONE
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Use this ZIP for baseline modeling:"
)

print(
    "Delaware_Recall_Model_Splits.zip"
)

print()
print(
    "The three flagged recordings remain in:"
)

print(
    "manual_review_holdout.csv"
)

In [ ]:
# ============================================================
# DELAWARE RECALL TRANSCRIPT-ONLY BASELINE
# Word TF-IDF + character TF-IDF + logistic regression
# Uses only the Google Drive folder ID
# ============================================================

!pip -q install google-api-python-client scikit-learn joblib

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    matthews_corrcoef,
)
from sklearn.pipeline import FeatureUnion, Pipeline

from pathlib import Path

import hashlib
import json
import math
import shutil
import zipfile
import joblib
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DELAWARE_FOLDER_ID = "1RV-tYijUIOBV4O7jb8RcvnWC3RdV8U5J"

RANDOM_SEED = 42
BOOTSTRAP_RUNS = 2000

LOCAL_ROOT = Path(
    "/content/delaware_recall_transcript_baseline"
)

INPUT_ROOT = LOCAL_ROOT / "input"
EXTRACT_ROOT = INPUT_ROOT / "extracted"

OUTPUT_ROOT = (
    LOCAL_ROOT
    / "Delaware_Recall_Transcript_Baseline"
)

MODEL_ROOT = OUTPUT_ROOT / "model"
AUDIT_ROOT = OUTPUT_ROOT / "audit"
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"

for path in [
    INPUT_ROOT,
    EXTRACT_ROOT,
    OUTPUT_ROOT,
    MODEL_ROOT,
    AUDIT_ROOT,
    PREDICTION_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

drive_service = build(
    "drive",
    "v3",
)


# ============================================================
# GOOGLE DRIVE UTILITIES
# ============================================================

def list_folder(folder_id):
    items = []
    page_token = None

    while True:
        response = (
            drive_service.files()
            .list(
                q=(
                    f"'{folder_id}' in parents "
                    "and trashed = false"
                ),
                fields=(
                    "nextPageToken,"
                    "files("
                    "id,name,mimeType,size,"
                    "modifiedTime,parents"
                    ")"
                ),
                pageToken=page_token,
                pageSize=1000,
            )
            .execute()
        )

        items.extend(
            response.get("files", [])
        )

        page_token = response.get(
            "nextPageToken"
        )

        if not page_token:
            break

    return items


def walk_drive_folder(
    folder_id,
    relative_path=Path(),
):
    records = []

    for item in list_folder(folder_id):
        current_path = (
            relative_path / item["name"]
        )

        if (
            item["mimeType"]
            == "application/vnd.google-apps.folder"
        ):
            records.extend(
                walk_drive_folder(
                    item["id"],
                    current_path,
                )
            )

        else:
            records.append({
                "file_id": item["id"],
                "name": item["name"],
                "relative_path": str(
                    current_path
                ),
                "mime_type": item["mimeType"],
                "size": int(
                    item.get("size", 0)
                ),
                "modified_time": item.get(
                    "modifiedTime"
                ),
            })

    return records


def download_drive_file(
    file_id,
    destination,
):
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    request = (
        drive_service.files()
        .get_media(fileId=file_id)
    )

    with destination.open("wb") as output_file:
        downloader = MediaIoBaseDownload(
            output_file,
            request,
            chunksize=16 * 1024 * 1024,
        )

        done = False

        while not done:
            _, done = downloader.next_chunk()

    return destination


def get_or_create_folder(
    parent_id,
    folder_name,
):
    safe_name = folder_name.replace(
        "'",
        "\\'",
    )

    response = (
        drive_service.files()
        .list(
            q=(
                f"'{parent_id}' in parents and "
                f"name = '{safe_name}' and "
                "mimeType = "
                "'application/vnd.google-apps.folder' "
                "and trashed = false"
            ),
            fields="files(id,name)",
        )
        .execute()
    )

    matches = response.get(
        "files",
        [],
    )

    if matches:
        return matches[0]["id"]

    created = (
        drive_service.files()
        .create(
            body={
                "name": folder_name,
                "mimeType": (
                    "application/"
                    "vnd.google-apps.folder"
                ),
                "parents": [parent_id],
            },
            fields="id",
        )
        .execute()
    )

    return created["id"]


def upload_file(
    local_path,
    parent_id,
):
    local_path = Path(local_path)

    result = (
        drive_service.files()
        .create(
            body={
                "name": local_path.name,
                "parents": [parent_id],
            },
            media_body=MediaFileUpload(
                str(local_path),
                resumable=True,
            ),
            fields="id,name",
        )
        .execute()
    )

    return result


# ============================================================
# FILE UTILITIES
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_transcript(path):
    return Path(path).read_text(
        encoding="utf-8",
        errors="replace",
    ).strip()


# ============================================================
# LOCATE MODEL-SPLIT ZIP
# ============================================================

print("=" * 100)
print("LOCATING DELAWARE MODEL SPLITS")
print("=" * 100)

drive_records = walk_drive_folder(
    DELAWARE_FOLDER_ID
)

zip_candidates = [
    record
    for record in drive_records
    if (
        record["name"].lower()
        == "delaware_recall_model_splits.zip"
    )
]

if not zip_candidates:
    raise FileNotFoundError(
        "Could not find "
        "Delaware_Recall_Model_Splits.zip "
        "inside the linked Drive folder."
    )

zip_candidates = sorted(
    zip_candidates,
    key=lambda item: (
        item.get("modified_time") or ""
    ),
    reverse=True,
)

selected_zip_record = zip_candidates[0]

print("Using:")
print(
    selected_zip_record[
        "relative_path"
    ]
)

local_zip_path = (
    INPUT_ROOT
    / "Delaware_Recall_Model_Splits.zip"
)

download_drive_file(
    selected_zip_record["file_id"],
    local_zip_path,
)

print("Downloaded:")
print(local_zip_path)


# ============================================================
# EXTRACT MODEL SPLITS
# ============================================================

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(
    local_zip_path,
    "r",
) as archive:
    archive.extractall(
        EXTRACT_ROOT
    )

manifest_candidates = list(
    EXTRACT_ROOT.rglob(
        "recall_model_split_manifest.csv"
    )
)

if not manifest_candidates:
    raise FileNotFoundError(
        "The ZIP does not contain "
        "recall_model_split_manifest.csv."
    )

SOURCE_ROOT = (
    manifest_candidates[0].parent
)

manifest = pd.read_csv(
    manifest_candidates[0]
)

manifest["subject_id"] = (
    manifest["subject_id"]
    .astype(str)
)

manifest["recording_id"] = (
    manifest["recording_id"]
    .astype(str)
)

print()
print("Rows loaded:", len(manifest))
print(
    manifest.groupby(
        ["split", "diagnosis"]
    ).size()
)


# ============================================================
# LOAD TRANSCRIPT TEXT
# ============================================================

print()
print("=" * 100)
print("LOADING TRANSCRIPTS")
print("=" * 100)

text_rows = []

for index, row in enumerate(
    manifest.itertuples(index=False),
    start=1,
):
    transcript_path = (
        SOURCE_ROOT
        / str(row.transcript_path)
    )

    if not transcript_path.exists():
        raise FileNotFoundError(
            f"Missing transcript: {transcript_path}"
        )

    transcript_text = read_transcript(
        transcript_path
    )

    text_rows.append({
        "split": row.split,
        "diagnosis": row.diagnosis,
        "binary_label": int(
            row.binary_label
        ),
        "subject_id": row.subject_id,
        "recording_id": row.recording_id,
        "transcript_path": str(
            row.transcript_path
        ),
        "text": transcript_text,
        "character_count": len(
            transcript_text
        ),
        "word_count_loaded": len(
            transcript_text.split()
        ),
    })

    if (
        index % 50 == 0
        or index == len(manifest)
    ):
        print(
            f"Loaded {index}/{len(manifest)}"
        )

text_manifest = pd.DataFrame(
    text_rows
)

empty_transcripts = text_manifest[
    text_manifest["text"].str.len() == 0
]

if len(empty_transcripts) > 0:
    empty_transcripts.to_csv(
        AUDIT_ROOT
        / "empty_transcripts.csv",
        index=False,
    )

    raise RuntimeError(
        f"Found {len(empty_transcripts)} "
        "empty transcripts."
    )

text_manifest.to_csv(
    AUDIT_ROOT
    / "transcript_loading_audit.csv",
    index=False,
)


# ============================================================
# SPLIT DATA
# ============================================================

train_df = text_manifest[
    text_manifest["split"] == "train"
].copy()

validation_df = text_manifest[
    text_manifest["split"] == "validation"
].copy()

test_df = text_manifest[
    text_manifest["split"] == "test"
].copy()

X_train = train_df["text"]
y_train = train_df[
    "binary_label"
].to_numpy()

X_validation = validation_df["text"]
y_validation = validation_df[
    "binary_label"
].to_numpy()

X_test = test_df["text"]
y_test = test_df[
    "binary_label"
].to_numpy()

print()
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))


# ============================================================
# METRIC FUNCTIONS
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "recall_sensitivity": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "specificity": float(
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        ),
        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "average_precision": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                predictions,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_threshold(
    y_true,
    probabilities,
):
    threshold_rows = []

    for threshold in np.linspace(
        0.05,
        0.95,
        181,
    ):
        metrics = calculate_metrics(
            y_true,
            probabilities,
            threshold,
        )

        threshold_rows.append(
            metrics
        )

    threshold_table = pd.DataFrame(
        threshold_rows
    )

    # Primary criterion:
    # highest balanced accuracy.
    #
    # Tie-breakers:
    # highest F1, then threshold nearest 0.50.
    threshold_table[
        "distance_from_half"
    ] = (
        threshold_table["threshold"]
        - 0.5
    ).abs()

    threshold_table = (
        threshold_table
        .sort_values(
            [
                "balanced_accuracy",
                "f1",
                "distance_from_half",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    selected_threshold = float(
        threshold_table.loc[
            0,
            "threshold",
        ]
    )

    return (
        selected_threshold,
        threshold_table,
    )


# ============================================================
# MODEL CANDIDATES
# ============================================================

def build_word_model(C):
    return Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.98,
                sublinear_tf=True,
                max_features=30000,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                C=C,
                class_weight="balanced",
                solver="liblinear",
                max_iter=5000,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


def build_character_model(C):
    return Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                analyzer="char_wb",
                lowercase=True,
                ngram_range=(3, 5),
                min_df=2,
                sublinear_tf=True,
                max_features=40000,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                C=C,
                class_weight="balanced",
                solver="liblinear",
                max_iter=5000,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


def build_combined_model(C):
    features = FeatureUnion([
        (
            "word",
            TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.98,
                sublinear_tf=True,
                max_features=30000,
            ),
        ),
        (
            "character",
            TfidfVectorizer(
                analyzer="char_wb",
                lowercase=True,
                ngram_range=(3, 5),
                min_df=2,
                sublinear_tf=True,
                max_features=40000,
            ),
        ),
    ])

    return Pipeline([
        (
            "features",
            features,
        ),
        (
            "classifier",
            LogisticRegression(
                C=C,
                class_weight="balanced",
                solver="liblinear",
                max_iter=5000,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


candidate_builders = {
    "word_tfidf": build_word_model,
    "character_tfidf": (
        build_character_model
    ),
    "word_character_tfidf": (
        build_combined_model
    ),
}

C_VALUES = [
    0.1,
    0.25,
    0.5,
    1.0,
    2.0,
    4.0,
]


# ============================================================
# TRAIN AND SELECT USING VALIDATION ONLY
# ============================================================

print()
print("=" * 100)
print("MODEL SELECTION ON VALIDATION SET")
print("=" * 100)

candidate_rows = []
candidate_objects = {}
threshold_tables = {}

for model_name, builder in (
    candidate_builders.items()
):
    for C in C_VALUES:
        candidate_id = (
            f"{model_name}_C_{C}"
        )

        print("Training:", candidate_id)

        model = builder(C)

        model.fit(
            X_train,
            y_train,
        )

        validation_probabilities = (
            model.predict_proba(
                X_validation
            )[:, 1]
        )

        (
            selected_threshold,
            threshold_table,
        ) = select_threshold(
            y_validation,
            validation_probabilities,
        )

        validation_metrics = (
            calculate_metrics(
                y_validation,
                validation_probabilities,
                selected_threshold,
            )
        )

        candidate_rows.append({
            "candidate_id": candidate_id,
            "model_name": model_name,
            "C": C,
            **validation_metrics,
        })

        candidate_objects[
            candidate_id
        ] = model

        threshold_tables[
            candidate_id
        ] = threshold_table

candidate_results = pd.DataFrame(
    candidate_rows
)

candidate_results[
    "distance_from_half"
] = (
    candidate_results["threshold"]
    - 0.5
).abs()

candidate_results = (
    candidate_results
    .sort_values(
        [
            "balanced_accuracy",
            "roc_auc",
            "f1",
            "distance_from_half",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

candidate_results_path = (
    AUDIT_ROOT
    / "validation_model_selection.csv"
)

candidate_results.to_csv(
    candidate_results_path,
    index=False,
)

display(
    candidate_results[
        [
            "candidate_id",
            "threshold",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall_sensitivity",
            "specificity",
            "f1",
            "roc_auc",
            "average_precision",
            "mcc",
        ]
    ]
)

best_candidate_id = str(
    candidate_results.loc[
        0,
        "candidate_id",
    ]
)

best_threshold = float(
    candidate_results.loc[
        0,
        "threshold",
    ]
)

best_model = candidate_objects[
    best_candidate_id
]

best_threshold_table = (
    threshold_tables[
        best_candidate_id
    ]
)

best_threshold_table.to_csv(
    AUDIT_ROOT
    / "best_validation_threshold_search.csv",
    index=False,
)

print()
print("Best candidate:")
print(best_candidate_id)

print("Selected threshold:")
print(best_threshold)


# ============================================================
# SAVE VALIDATION PREDICTIONS
# ============================================================

validation_probabilities = (
    best_model.predict_proba(
        X_validation
    )[:, 1]
)

validation_predictions = (
    validation_probabilities
    >= best_threshold
).astype(int)

validation_predictions_table = (
    validation_df[
        [
            "subject_id",
            "recording_id",
            "diagnosis",
            "binary_label",
            "transcript_path",
        ]
    ]
    .copy()
)

validation_predictions_table[
    "predicted_probability_MCI"
] = validation_probabilities

validation_predictions_table[
    "predicted_label"
] = validation_predictions

validation_predictions_table.to_csv(
    PREDICTION_ROOT
    / "validation_predictions.csv",
    index=False,
)


# ============================================================
# FINAL TEST EVALUATION
# ============================================================

print()
print("=" * 100)
print("FINAL UNTOUCHED TEST EVALUATION")
print("=" * 100)

test_probabilities = (
    best_model.predict_proba(
        X_test
    )[:, 1]
)

test_predictions = (
    test_probabilities
    >= best_threshold
).astype(int)

test_metrics = calculate_metrics(
    y_test,
    test_probabilities,
    best_threshold,
)

print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)

test_predictions_table = (
    test_df[
        [
            "subject_id",
            "recording_id",
            "diagnosis",
            "binary_label",
            "transcript_path",
        ]
    ]
    .copy()
)

test_predictions_table[
    "predicted_probability_MCI"
] = test_probabilities

test_predictions_table[
    "predicted_label"
] = test_predictions

test_predictions_table[
    "correct"
] = (
    test_predictions_table[
        "binary_label"
    ]
    == test_predictions_table[
        "predicted_label"
    ]
)

test_predictions_path = (
    PREDICTION_ROOT
    / "test_predictions.csv"
)

test_predictions_table.to_csv(
    test_predictions_path,
    index=False,
)


# ============================================================
# BOOTSTRAP TEST CONFIDENCE INTERVALS
# ============================================================

print()
print("=" * 100)
print("BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 100)

rng = np.random.default_rng(
    RANDOM_SEED
)

bootstrap_rows = []

attempts = 0
max_attempts = (
    BOOTSTRAP_RUNS * 20
)

while (
    len(bootstrap_rows)
    < BOOTSTRAP_RUNS
    and attempts < max_attempts
):
    attempts += 1

    indices = rng.integers(
        0,
        len(y_test),
        size=len(y_test),
    )

    sampled_y = y_test[indices]

    # AUROC requires both classes.
    if len(np.unique(sampled_y)) < 2:
        continue

    sampled_probabilities = (
        test_probabilities[indices]
    )

    sampled_metrics = calculate_metrics(
        sampled_y,
        sampled_probabilities,
        best_threshold,
    )

    bootstrap_rows.append(
        sampled_metrics
    )

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

if len(bootstrap_results) < BOOTSTRAP_RUNS:
    print(
        "Warning: obtained only",
        len(bootstrap_results),
        "valid bootstrap samples."
    )

confidence_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1",
    "roc_auc",
    "average_precision",
    "mcc",
]

confidence_interval_rows = []

for metric_name in confidence_metrics:
    values = (
        bootstrap_results[
            metric_name
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
        .to_numpy()
    )

    confidence_interval_rows.append({
        "metric": metric_name,
        "point_estimate": (
            test_metrics[
                metric_name
            ]
        ),
        "bootstrap_mean": float(
            np.mean(values)
        ),
        "ci_95_lower": float(
            np.percentile(
                values,
                2.5,
            )
        ),
        "ci_95_upper": float(
            np.percentile(
                values,
                97.5,
            )
        ),
        "valid_bootstrap_runs": int(
            len(values)
        ),
    })

confidence_intervals = pd.DataFrame(
    confidence_interval_rows
)

confidence_intervals_path = (
    OUTPUT_ROOT
    / "test_bootstrap_confidence_intervals.csv"
)

confidence_intervals.to_csv(
    confidence_intervals_path,
    index=False,
)

display(confidence_intervals)


# ============================================================
# ERROR ANALYSIS
# ============================================================

false_positives = (
    test_predictions_table[
        (
            test_predictions_table[
                "binary_label"
            ] == 0
        )
        & (
            test_predictions_table[
                "predicted_label"
            ] == 1
        )
    ]
    .copy()
)

false_negatives = (
    test_predictions_table[
        (
            test_predictions_table[
                "binary_label"
            ] == 1
        )
        & (
            test_predictions_table[
                "predicted_label"
            ] == 0
        )
    ]
    .copy()
)

false_positives.to_csv(
    AUDIT_ROOT
    / "test_false_positives.csv",
    index=False,
)

false_negatives.to_csv(
    AUDIT_ROOT
    / "test_false_negatives.csv",
    index=False,
)


# ============================================================
# SAVE FINAL MODEL
# ============================================================

model_path = (
    MODEL_ROOT
    / "transcript_baseline_model.joblib"
)

joblib.dump(
    best_model,
    model_path,
)

model_metadata = {
    "dataset": "Delaware",
    "task": "Cinderella recall",
    "classification_target": (
        "MCI versus Control"
    ),
    "input_modality": "transcript only",
    "best_candidate_id": (
        best_candidate_id
    ),
    "selected_threshold": (
        best_threshold
    ),
    "random_seed": RANDOM_SEED,
    "train_subjects": int(
        len(train_df)
    ),
    "validation_subjects": int(
        len(validation_df)
    ),
    "test_subjects": int(
        len(test_df)
    ),
    "validation_selection_metric": (
        "balanced_accuracy"
    ),
    "test_metrics": test_metrics,
    "limitations": [
        (
            "Single-corpus MCI versus Control "
            "research classifier."
        ),
        (
            "Not clinically validated and not "
            "a diagnostic test."
        ),
        (
            "The test set contains only 43 subjects, "
            "so confidence intervals may be wide."
        ),
    ],
}

metadata_path = (
    MODEL_ROOT
    / "model_metadata.json"
)

metadata_path.write_text(
    json.dumps(
        model_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# SAVE METRICS
# ============================================================

test_metrics_table = pd.DataFrame([
    {
        "split": "test",
        "candidate_id": (
            best_candidate_id
        ),
        **test_metrics,
    }
])

test_metrics_path = (
    OUTPUT_ROOT
    / "test_metrics.csv"
)

test_metrics_table.to_csv(
    test_metrics_path,
    index=False,
)


# ============================================================
# CREATE README
# ============================================================

readme_text = f"""
DELAWARE RECALL TRANSCRIPT-ONLY BASELINE
========================================

Task
----
Cinderella story recall classification.

Target
------
MCI versus healthy Control.

Data
----
Train subjects: {len(train_df)}
Validation subjects: {len(validation_df)}
Test subjects: {len(test_df)}

Model selection
---------------
Candidate models:
- Word TF-IDF logistic regression
- Character TF-IDF logistic regression
- Combined word + character TF-IDF logistic regression

Regularization values tested:
{C_VALUES}

Best model:
{best_candidate_id}

Validation-selected threshold:
{best_threshold:.3f}

Untouched test result
---------------------
Accuracy: {test_metrics['accuracy']:.4f}
Balanced accuracy: {test_metrics['balanced_accuracy']:.4f}
Precision: {test_metrics['precision']:.4f}
Sensitivity: {test_metrics['recall_sensitivity']:.4f}
Specificity: {test_metrics['specificity']:.4f}
F1: {test_metrics['f1']:.4f}
ROC-AUC: {test_metrics['roc_auc']:.4f}
Average precision: {test_metrics['average_precision']:.4f}
MCC: {test_metrics['mcc']:.4f}

Confusion matrix
----------------
TN: {test_metrics['tn']}
FP: {test_metrics['fp']}
FN: {test_metrics['fn']}
TP: {test_metrics['tp']}

Important limitation
--------------------
This is a research baseline for MCI versus Control within the
Delaware corpus. It is not a clinically validated diagnostic test.

The classification threshold was selected using validation data only.
The test set was not used for model selection.
"""

readme_path = (
    OUTPUT_ROOT
    / "README.txt"
)

readme_path.write_text(
    readme_text.strip() + "\n",
    encoding="utf-8",
)


# ============================================================
# CREATE FILE MANIFEST
# ============================================================

file_manifest_rows = []

for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if not path.is_file():
        continue

    file_manifest_rows.append({
        "relative_path": str(
            path.relative_to(
                OUTPUT_ROOT
            )
        ),
        "size_bytes": (
            path.stat().st_size
        ),
        "sha256": sha256_file(path),
    })

file_manifest = pd.DataFrame(
    file_manifest_rows
)

file_manifest_path = (
    OUTPUT_ROOT
    / "file_manifest.csv"
)

file_manifest.to_csv(
    file_manifest_path,
    index=False,
)


# ============================================================
# ZIP BACKUP
# ============================================================

zip_base = (
    LOCAL_ROOT
    / "Delaware_Recall_Transcript_Baseline"
)

final_zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUTPUT_ROOT.parent,
    base_dir=OUTPUT_ROOT.name,
)

print()
print("Created ZIP:")
print(final_zip_path)


# ============================================================
# UPLOAD RESULTS TO DRIVE
# ============================================================

output_drive_folder_id = (
    get_or_create_folder(
        DELAWARE_FOLDER_ID,
        "Delaware_Recall_Transcript_Baseline",
    )
)

upload_paths = [
    Path(final_zip_path),
    model_path,
    metadata_path,
    test_metrics_path,
    confidence_intervals_path,
    candidate_results_path,
    test_predictions_path,
    PREDICTION_ROOT
    / "validation_predictions.csv",
    AUDIT_ROOT
    / "best_validation_threshold_search.csv",
    AUDIT_ROOT
    / "test_false_positives.csv",
    AUDIT_ROOT
    / "test_false_negatives.csv",
    readme_path,
    file_manifest_path,
]

print()
print("=" * 100)
print("UPLOADING TRANSCRIPT BASELINE")
print("=" * 100)

for local_path in upload_paths:
    result = upload_file(
        local_path,
        output_drive_folder_id,
    )

    print(
        "Uploaded:",
        result["name"],
    )


# ============================================================
# DONE
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print("Best model:")
print(best_candidate_id)

print()
print("Validation-selected threshold:")
print(best_threshold)

print()
print("Final untouched test metrics:")
print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)

print()
print("Use this ZIP:")
print(
    "Delaware_Recall_Transcript_Baseline.zip"
)

In [ ]:
# ============================================================
# DELAWARE RECALL AUDIO-ONLY BASELINE
# openSMILE eGeMAPSv02 acoustic features
# Uses only the Google Drive folder ID
# ============================================================

!pip -q install \
    google-api-python-client \
    opensmile \
    scikit-learn \
    joblib \
    soundfile

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import (
    MediaIoBaseDownload,
    MediaFileUpload,
)

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

from pathlib import Path

import hashlib
import json
import shutil
import warnings
import zipfile

import joblib
import numpy as np
import opensmile
import pandas as pd
import soundfile as sf


# ============================================================
# CONFIGURATION
# ============================================================

DELAWARE_FOLDER_ID = "1RV-tYijUIOBV4O7jb8RcvnWC3RdV8U5J"

RANDOM_SEED = 42
BOOTSTRAP_RUNS = 2000

LOCAL_ROOT = Path(
    "/content/delaware_recall_audio_baseline"
)

INPUT_ROOT = LOCAL_ROOT / "input"
EXTRACT_ROOT = INPUT_ROOT / "extracted"

OUTPUT_ROOT = (
    LOCAL_ROOT
    / "Delaware_Recall_Audio_Baseline"
)

FEATURE_ROOT = OUTPUT_ROOT / "features"
MODEL_ROOT = OUTPUT_ROOT / "model"
AUDIT_ROOT = OUTPUT_ROOT / "audit"
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"

for path in [
    INPUT_ROOT,
    EXTRACT_ROOT,
    OUTPUT_ROOT,
    FEATURE_ROOT,
    MODEL_ROOT,
    AUDIT_ROOT,
    PREDICTION_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
)

drive_service = build(
    "drive",
    "v3",
)


# ============================================================
# GOOGLE DRIVE UTILITIES
# ============================================================

def list_folder(folder_id):
    items = []
    page_token = None

    while True:
        response = (
            drive_service.files()
            .list(
                q=(
                    f"'{folder_id}' in parents "
                    "and trashed = false"
                ),
                fields=(
                    "nextPageToken,"
                    "files("
                    "id,name,mimeType,size,"
                    "modifiedTime,parents"
                    ")"
                ),
                pageToken=page_token,
                pageSize=1000,
            )
            .execute()
        )

        items.extend(
            response.get("files", [])
        )

        page_token = response.get(
            "nextPageToken"
        )

        if not page_token:
            break

    return items


def walk_drive_folder(
    folder_id,
    relative_path=Path(),
):
    records = []

    for item in list_folder(folder_id):
        current_path = (
            relative_path / item["name"]
        )

        if (
            item["mimeType"]
            == "application/vnd.google-apps.folder"
        ):
            records.extend(
                walk_drive_folder(
                    item["id"],
                    current_path,
                )
            )

        else:
            records.append({
                "file_id": item["id"],
                "name": item["name"],
                "relative_path": str(
                    current_path
                ),
                "mime_type": item["mimeType"],
                "size": int(
                    item.get("size", 0)
                ),
                "modified_time": item.get(
                    "modifiedTime"
                ),
            })

    return records


def download_drive_file(
    file_id,
    destination,
):
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    request = (
        drive_service.files()
        .get_media(fileId=file_id)
    )

    with destination.open("wb") as output_file:
        downloader = MediaIoBaseDownload(
            output_file,
            request,
            chunksize=16 * 1024 * 1024,
        )

        done = False

        while not done:
            _, done = downloader.next_chunk()

    return destination


def get_or_create_folder(
    parent_id,
    folder_name,
):
    safe_name = folder_name.replace(
        "'",
        "\\'",
    )

    response = (
        drive_service.files()
        .list(
            q=(
                f"'{parent_id}' in parents and "
                f"name = '{safe_name}' and "
                "mimeType = "
                "'application/vnd.google-apps.folder' "
                "and trashed = false"
            ),
            fields="files(id,name)",
        )
        .execute()
    )

    matches = response.get(
        "files",
        [],
    )

    if matches:
        return matches[0]["id"]

    created = (
        drive_service.files()
        .create(
            body={
                "name": folder_name,
                "mimeType": (
                    "application/"
                    "vnd.google-apps.folder"
                ),
                "parents": [parent_id],
            },
            fields="id",
        )
        .execute()
    )

    return created["id"]


def upload_file(
    local_path,
    parent_id,
):
    local_path = Path(local_path)

    result = (
        drive_service.files()
        .create(
            body={
                "name": local_path.name,
                "parents": [parent_id],
            },
            media_body=MediaFileUpload(
                str(local_path),
                resumable=True,
            ),
            fields="id,name",
        )
        .execute()
    )

    return result


# ============================================================
# FILE UTILITIES
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ============================================================
# LOCATE MODEL-SPLIT ZIP
# ============================================================

print("=" * 100)
print("LOCATING DELAWARE MODEL SPLITS")
print("=" * 100)

drive_records = walk_drive_folder(
    DELAWARE_FOLDER_ID
)

zip_candidates = [
    record
    for record in drive_records
    if (
        record["name"].lower()
        == "delaware_recall_model_splits.zip"
    )
]

if not zip_candidates:
    raise FileNotFoundError(
        "Could not find "
        "Delaware_Recall_Model_Splits.zip."
    )

zip_candidates = sorted(
    zip_candidates,
    key=lambda item: (
        item.get("modified_time") or ""
    ),
    reverse=True,
)

selected_zip_record = zip_candidates[0]

print("Using:")
print(
    selected_zip_record[
        "relative_path"
    ]
)

local_zip_path = (
    INPUT_ROOT
    / "Delaware_Recall_Model_Splits.zip"
)

download_drive_file(
    selected_zip_record["file_id"],
    local_zip_path,
)

print("Downloaded:")
print(local_zip_path)


# ============================================================
# EXTRACT MODEL SPLITS
# ============================================================

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(
    local_zip_path,
    "r",
) as archive:
    archive.extractall(
        EXTRACT_ROOT
    )

manifest_candidates = list(
    EXTRACT_ROOT.rglob(
        "recall_model_split_manifest.csv"
    )
)

if not manifest_candidates:
    raise FileNotFoundError(
        "Could not find "
        "recall_model_split_manifest.csv."
    )

SOURCE_ROOT = (
    manifest_candidates[0].parent
)

manifest = pd.read_csv(
    manifest_candidates[0]
)

manifest["subject_id"] = (
    manifest["subject_id"]
    .astype(str)
)

manifest["recording_id"] = (
    manifest["recording_id"]
    .astype(str)
)

print()
print("Rows loaded:", len(manifest))

display(
    manifest.groupby(
        ["split", "diagnosis"]
    )
    .size()
    .reset_index(name="recordings")
)


# ============================================================
# OPENSMILE FEATURE EXTRACTOR
# ============================================================

smile = opensmile.Smile(
    feature_set=(
        opensmile.FeatureSet.eGeMAPSv02
    ),
    feature_level=(
        opensmile.FeatureLevel.Functionals
    ),
)


# ============================================================
# EXTRACT AUDIO FEATURES
# ============================================================

print()
print("=" * 100)
print("EXTRACTING OPENSMILE eGeMAPS FEATURES")
print("=" * 100)

feature_rows = []
failed_rows = []

for index, row in enumerate(
    manifest.itertuples(index=False),
    start=1,
):
    audio_path = (
        SOURCE_ROOT
        / str(row.audio_path)
    )

    try:
        if not audio_path.exists():
            raise FileNotFoundError(
                str(audio_path)
            )

        audio_info = sf.info(
            str(audio_path)
        )

        feature_frame = (
            smile.process_file(
                str(audio_path)
            )
        )

        if len(feature_frame) != 1:
            raise ValueError(
                "Expected exactly one functional "
                "feature row."
            )

        feature_values = (
            feature_frame
            .reset_index(drop=True)
            .iloc[0]
            .to_dict()
        )

        record = {
            "split": row.split,
            "diagnosis": row.diagnosis,
            "binary_label": int(
                row.binary_label
            ),
            "subject_id": row.subject_id,
            "recording_id": (
                row.recording_id
            ),
            "audio_path": str(
                row.audio_path
            ),
            "duration_seconds_manifest": float(
                row.duration_seconds
            ),
            "duration_seconds_file": float(
                audio_info.duration
            ),
            "sample_rate_hz": int(
                audio_info.samplerate
            ),
            "channels": int(
                audio_info.channels
            ),
        }

        record.update(
            {
                str(key): float(value)
                if pd.notna(value)
                else np.nan
                for key, value
                in feature_values.items()
            }
        )

        feature_rows.append(record)

    except Exception as exception:
        failed_rows.append({
            "split": row.split,
            "diagnosis": row.diagnosis,
            "subject_id": row.subject_id,
            "recording_id": (
                row.recording_id
            ),
            "audio_path": str(
                row.audio_path
            ),
            "error_type": (
                type(exception).__name__
            ),
            "error": str(exception),
        })

    if (
        index % 25 == 0
        or index == len(manifest)
    ):
        print(
            f"Processed {index}/{len(manifest)}"
        )


features = pd.DataFrame(
    feature_rows
)

failed_features = pd.DataFrame(
    failed_rows
)

failed_features_path = (
    AUDIT_ROOT
    / "failed_feature_extraction.csv"
)

failed_features.to_csv(
    failed_features_path,
    index=False,
)

if len(failed_features) > 0:
    display(failed_features)

    raise RuntimeError(
        f"Feature extraction failed for "
        f"{len(failed_features)} recordings."
    )

feature_metadata_columns = [
    "split",
    "diagnosis",
    "binary_label",
    "subject_id",
    "recording_id",
    "audio_path",
    "duration_seconds_manifest",
    "duration_seconds_file",
    "sample_rate_hz",
    "channels",
]

acoustic_feature_columns = [
    column
    for column in features.columns
    if column not in feature_metadata_columns
]

print()
print(
    "Acoustic features:",
    len(acoustic_feature_columns),
)

print(
    "Recordings extracted:",
    len(features),
)

feature_table_path = (
    FEATURE_ROOT
    / "egemaps_features_all.csv"
)

features.to_csv(
    feature_table_path,
    index=False,
)


# ============================================================
# AUDIO FEATURE QUALITY AUDIT
# ============================================================

quality_rows = []

for row in features.itertuples(
    index=False
):
    quality_rows.append({
        "split": row.split,
        "diagnosis": row.diagnosis,
        "subject_id": row.subject_id,
        "recording_id": row.recording_id,
        "sample_rate_hz": row.sample_rate_hz,
        "channels": row.channels,
        "duration_seconds_manifest": (
            row.duration_seconds_manifest
        ),
        "duration_seconds_file": (
            row.duration_seconds_file
        ),
        "duration_difference_seconds": abs(
            row.duration_seconds_manifest
            - row.duration_seconds_file
        ),
    })

audio_quality_audit = pd.DataFrame(
    quality_rows
)

audio_quality_audit_path = (
    AUDIT_ROOT
    / "audio_feature_quality_audit.csv"
)

audio_quality_audit.to_csv(
    audio_quality_audit_path,
    index=False,
)

if (
    audio_quality_audit[
        "duration_difference_seconds"
    ].max()
    > 1.0
):
    print(
        "Warning: at least one audio duration "
        "differs from the manifest by over one second."
    )


# ============================================================
# SPLIT FEATURE MATRICES
# ============================================================

train_df = features[
    features["split"] == "train"
].copy()

validation_df = features[
    features["split"] == "validation"
].copy()

test_df = features[
    features["split"] == "test"
].copy()

X_train = train_df[
    acoustic_feature_columns
].to_numpy(dtype=np.float64)

y_train = train_df[
    "binary_label"
].to_numpy(dtype=int)

X_validation = validation_df[
    acoustic_feature_columns
].to_numpy(dtype=np.float64)

y_validation = validation_df[
    "binary_label"
].to_numpy(dtype=int)

X_test = test_df[
    acoustic_feature_columns
].to_numpy(dtype=np.float64)

y_test = test_df[
    "binary_label"
].to_numpy(dtype=int)

print()
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))


# ============================================================
# METRIC FUNCTIONS
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "recall_sensitivity": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "specificity": float(
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        ),
        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "average_precision": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                predictions,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_threshold(
    y_true,
    probabilities,
):
    threshold_rows = []

    for threshold in np.linspace(
        0.05,
        0.95,
        181,
    ):
        metrics = calculate_metrics(
            y_true,
            probabilities,
            threshold,
        )

        threshold_rows.append(
            metrics
        )

    threshold_table = pd.DataFrame(
        threshold_rows
    )

    threshold_table[
        "distance_from_half"
    ] = (
        threshold_table["threshold"]
        - 0.5
    ).abs()

    threshold_table = (
        threshold_table
        .sort_values(
            [
                "balanced_accuracy",
                "f1",
                "roc_auc",
                "distance_from_half",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    selected_threshold = float(
        threshold_table.loc[
            0,
            "threshold",
        ]
    )

    return (
        selected_threshold,
        threshold_table,
    )


# ============================================================
# MODEL BUILDERS
# ============================================================

def build_logistic_model(C):
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                C=C,
                class_weight="balanced",
                solver="liblinear",
                max_iter=10000,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


def build_linear_svm(C):
    base_model = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LinearSVC(
                C=C,
                class_weight="balanced",
                max_iter=20000,
                random_state=RANDOM_SEED,
            ),
        ),
    ])

    return CalibratedClassifierCV(
        estimator=base_model,
        method="sigmoid",
        cv=5,
    )


def build_rbf_svm(C):
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            SVC(
                C=C,
                kernel="rbf",
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


def build_random_forest(
    max_depth,
    min_samples_leaf,
):
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=500,
                max_depth=max_depth,
                min_samples_leaf=(
                    min_samples_leaf
                ),
                max_features="sqrt",
                class_weight="balanced",
                random_state=RANDOM_SEED,
                n_jobs=-1,
            ),
        ),
    ])


candidate_definitions = []

for C in [
    0.01,
    0.05,
    0.1,
    0.25,
    0.5,
    1.0,
    2.0,
    4.0,
]:
    candidate_definitions.append({
        "candidate_id": (
            f"logistic_C_{C}"
        ),
        "model_family": "logistic",
        "parameters": {
            "C": C,
        },
        "builder": lambda C=C: (
            build_logistic_model(C)
        ),
    })

for C in [
    0.01,
    0.05,
    0.1,
    0.25,
    0.5,
    1.0,
]:
    candidate_definitions.append({
        "candidate_id": (
            f"linear_svm_C_{C}"
        ),
        "model_family": "linear_svm",
        "parameters": {
            "C": C,
        },
        "builder": lambda C=C: (
            build_linear_svm(C)
        ),
    })

for C in [
    0.1,
    0.25,
    0.5,
    1.0,
    2.0,
    4.0,
    8.0,
]:
    candidate_definitions.append({
        "candidate_id": (
            f"rbf_svm_C_{C}"
        ),
        "model_family": "rbf_svm",
        "parameters": {
            "C": C,
        },
        "builder": lambda C=C: (
            build_rbf_svm(C)
        ),
    })

for max_depth in [
    None,
    4,
    8,
    12,
]:
    for min_samples_leaf in [
        2,
        4,
        8,
    ]:
        depth_name = (
            "none"
            if max_depth is None
            else str(max_depth)
        )

        candidate_definitions.append({
            "candidate_id": (
                f"random_forest_depth_"
                f"{depth_name}_leaf_"
                f"{min_samples_leaf}"
            ),
            "model_family": (
                "random_forest"
            ),
            "parameters": {
                "max_depth": max_depth,
                "min_samples_leaf": (
                    min_samples_leaf
                ),
            },
            "builder": (
                lambda
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf:
                build_random_forest(
                    max_depth,
                    min_samples_leaf,
                )
            ),
        })


# ============================================================
# TRAIN AND SELECT ON VALIDATION ONLY
# ============================================================

print()
print("=" * 100)
print("MODEL SELECTION ON VALIDATION SET")
print("=" * 100)

candidate_rows = []
candidate_objects = {}
threshold_tables = {}

for candidate in candidate_definitions:
    candidate_id = candidate[
        "candidate_id"
    ]

    print("Training:", candidate_id)

    model = candidate["builder"]()

    model.fit(
        X_train,
        y_train,
    )

    validation_probabilities = (
        model.predict_proba(
            X_validation
        )[:, 1]
    )

    (
        selected_threshold,
        threshold_table,
    ) = select_threshold(
        y_validation,
        validation_probabilities,
    )

    validation_metrics = (
        calculate_metrics(
            y_validation,
            validation_probabilities,
            selected_threshold,
        )
    )

    candidate_rows.append({
        "candidate_id": candidate_id,
        "model_family": candidate[
            "model_family"
        ],
        "parameters": json.dumps(
            candidate["parameters"],
            sort_keys=True,
        ),
        **validation_metrics,
    })

    candidate_objects[
        candidate_id
    ] = model

    threshold_tables[
        candidate_id
    ] = threshold_table


candidate_results = pd.DataFrame(
    candidate_rows
)

candidate_results[
    "distance_from_half"
] = (
    candidate_results["threshold"]
    - 0.5
).abs()

candidate_results = (
    candidate_results
    .sort_values(
        [
            "balanced_accuracy",
            "roc_auc",
            "f1",
            "mcc",
            "distance_from_half",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

candidate_results_path = (
    AUDIT_ROOT
    / "validation_model_selection.csv"
)

candidate_results.to_csv(
    candidate_results_path,
    index=False,
)

display(
    candidate_results[
        [
            "candidate_id",
            "threshold",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall_sensitivity",
            "specificity",
            "f1",
            "roc_auc",
            "average_precision",
            "mcc",
        ]
    ]
)

best_candidate_id = str(
    candidate_results.loc[
        0,
        "candidate_id",
    ]
)

best_threshold = float(
    candidate_results.loc[
        0,
        "threshold",
    ]
)

best_model = candidate_objects[
    best_candidate_id
]

best_threshold_table = (
    threshold_tables[
        best_candidate_id
    ]
)

best_threshold_table.to_csv(
    AUDIT_ROOT
    / "best_validation_threshold_search.csv",
    index=False,
)

print()
print("Best candidate:")
print(best_candidate_id)

print("Selected threshold:")
print(best_threshold)


# ============================================================
# VALIDATION PREDICTIONS
# ============================================================

validation_probabilities = (
    best_model.predict_proba(
        X_validation
    )[:, 1]
)

validation_predictions = (
    validation_probabilities
    >= best_threshold
).astype(int)

validation_predictions_table = (
    validation_df[
        [
            "subject_id",
            "recording_id",
            "diagnosis",
            "binary_label",
            "audio_path",
        ]
    ]
    .copy()
)

validation_predictions_table[
    "predicted_probability_MCI"
] = validation_probabilities

validation_predictions_table[
    "predicted_label"
] = validation_predictions

validation_predictions_path = (
    PREDICTION_ROOT
    / "validation_predictions.csv"
)

validation_predictions_table.to_csv(
    validation_predictions_path,
    index=False,
)


# ============================================================
# UNTOUCHED TEST EVALUATION
# ============================================================

print()
print("=" * 100)
print("FINAL UNTOUCHED TEST EVALUATION")
print("=" * 100)

test_probabilities = (
    best_model.predict_proba(
        X_test
    )[:, 1]
)

test_predictions = (
    test_probabilities
    >= best_threshold
).astype(int)

test_metrics = calculate_metrics(
    y_test,
    test_probabilities,
    best_threshold,
)

print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)

test_predictions_table = (
    test_df[
        [
            "subject_id",
            "recording_id",
            "diagnosis",
            "binary_label",
            "audio_path",
        ]
    ]
    .copy()
)

test_predictions_table[
    "predicted_probability_MCI"
] = test_probabilities

test_predictions_table[
    "predicted_label"
] = test_predictions

test_predictions_table[
    "correct"
] = (
    test_predictions_table[
        "binary_label"
    ]
    == test_predictions_table[
        "predicted_label"
    ]
)

test_predictions_path = (
    PREDICTION_ROOT
    / "test_predictions.csv"
)

test_predictions_table.to_csv(
    test_predictions_path,
    index=False,
)


# ============================================================
# BOOTSTRAP TEST CONFIDENCE INTERVALS
# ============================================================

print()
print("=" * 100)
print("BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 100)

rng = np.random.default_rng(
    RANDOM_SEED
)

bootstrap_rows = []

attempts = 0
max_attempts = (
    BOOTSTRAP_RUNS * 20
)

while (
    len(bootstrap_rows)
    < BOOTSTRAP_RUNS
    and attempts < max_attempts
):
    attempts += 1

    indices = rng.integers(
        0,
        len(y_test),
        size=len(y_test),
    )

    sampled_y = y_test[indices]

    if len(np.unique(sampled_y)) < 2:
        continue

    sampled_probabilities = (
        test_probabilities[indices]
    )

    sampled_metrics = (
        calculate_metrics(
            sampled_y,
            sampled_probabilities,
            best_threshold,
        )
    )

    bootstrap_rows.append(
        sampled_metrics
    )

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

confidence_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1",
    "roc_auc",
    "average_precision",
    "mcc",
]

confidence_interval_rows = []

for metric_name in confidence_metrics:
    values = (
        bootstrap_results[
            metric_name
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
        .to_numpy()
    )

    confidence_interval_rows.append({
        "metric": metric_name,
        "point_estimate": (
            test_metrics[
                metric_name
            ]
        ),
        "bootstrap_mean": float(
            np.mean(values)
        ),
        "ci_95_lower": float(
            np.percentile(
                values,
                2.5,
            )
        ),
        "ci_95_upper": float(
            np.percentile(
                values,
                97.5,
            )
        ),
        "valid_bootstrap_runs": int(
            len(values)
        ),
    })

confidence_intervals = pd.DataFrame(
    confidence_interval_rows
)

confidence_intervals_path = (
    OUTPUT_ROOT
    / "test_bootstrap_confidence_intervals.csv"
)

confidence_intervals.to_csv(
    confidence_intervals_path,
    index=False,
)

display(confidence_intervals)


# ============================================================
# ERROR ANALYSIS
# ============================================================

false_positives = (
    test_predictions_table[
        (
            test_predictions_table[
                "binary_label"
            ] == 0
        )
        & (
            test_predictions_table[
                "predicted_label"
            ] == 1
        )
    ]
    .copy()
)

false_negatives = (
    test_predictions_table[
        (
            test_predictions_table[
                "binary_label"
            ] == 1
        )
        & (
            test_predictions_table[
                "predicted_label"
            ] == 0
        )
    ]
    .copy()
)

false_positives.to_csv(
    AUDIT_ROOT
    / "test_false_positives.csv",
    index=False,
)

false_negatives.to_csv(
    AUDIT_ROOT
    / "test_false_negatives.csv",
    index=False,
)


# ============================================================
# FEATURE IMPORTANCE FOR LINEAR MODELS
# ============================================================

feature_importance_path = (
    AUDIT_ROOT
    / "linear_feature_coefficients.csv"
)

feature_importance_table = pd.DataFrame()

try:
    if isinstance(
        best_model,
        Pipeline,
    ):
        classifier = (
            best_model.named_steps[
                "classifier"
            ]
        )

        if hasattr(
            classifier,
            "coef_",
        ):
            coefficients = (
                classifier.coef_[0]
            )

            feature_importance_table = (
                pd.DataFrame({
                    "feature": (
                        acoustic_feature_columns
                    ),
                    "coefficient": (
                        coefficients
                    ),
                    "absolute_coefficient": (
                        np.abs(coefficients)
                    ),
                })
                .sort_values(
                    "absolute_coefficient",
                    ascending=False,
                )
            )

            feature_importance_table.to_csv(
                feature_importance_path,
                index=False,
            )

except Exception as exception:
    print(
        "Could not export linear coefficients:",
        exception,
    )


# ============================================================
# SAVE MODEL
# ============================================================

model_path = (
    MODEL_ROOT
    / "audio_baseline_model.joblib"
)

joblib.dump(
    best_model,
    model_path,
)

feature_columns_path = (
    MODEL_ROOT
    / "acoustic_feature_columns.json"
)

feature_columns_path.write_text(
    json.dumps(
        acoustic_feature_columns,
        indent=2,
    ),
    encoding="utf-8",
)

model_metadata = {
    "dataset": "Delaware",
    "task": "Cinderella recall",
    "classification_target": (
        "MCI versus Control"
    ),
    "input_modality": "audio only",
    "feature_set": (
        "openSMILE eGeMAPSv02 Functionals"
    ),
    "number_of_acoustic_features": int(
        len(acoustic_feature_columns)
    ),
    "best_candidate_id": (
        best_candidate_id
    ),
    "selected_threshold": (
        best_threshold
    ),
    "random_seed": RANDOM_SEED,
    "train_subjects": int(
        len(train_df)
    ),
    "validation_subjects": int(
        len(validation_df)
    ),
    "test_subjects": int(
        len(test_df)
    ),
    "test_metrics": test_metrics,
    "limitations": [
        (
            "Single-corpus MCI versus Control "
            "research classifier."
        ),
        (
            "Not clinically validated and not "
            "a diagnostic test."
        ),
        (
            "Audio consists of concatenated "
            "participant-only utterances."
        ),
        (
            "The untouched test set contains "
            "43 subjects."
        ),
    ],
}

metadata_path = (
    MODEL_ROOT
    / "model_metadata.json"
)

metadata_path.write_text(
    json.dumps(
        model_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# SAVE TEST METRICS
# ============================================================

test_metrics_table = pd.DataFrame([
    {
        "split": "test",
        "candidate_id": (
            best_candidate_id
        ),
        **test_metrics,
    }
])

test_metrics_path = (
    OUTPUT_ROOT
    / "test_metrics.csv"
)

test_metrics_table.to_csv(
    test_metrics_path,
    index=False,
)


# ============================================================
# CREATE README
# ============================================================

readme_text = f"""
DELAWARE RECALL AUDIO-ONLY BASELINE
===================================

Task
----
Cinderella story recall classification.

Target
------
MCI versus healthy Control.

Input
-----
Participant-only recall audio.

Acoustic features
-----------------
openSMILE eGeMAPSv02 Functionals.

Number of acoustic features:
{len(acoustic_feature_columns)}

Data
----
Train subjects: {len(train_df)}
Validation subjects: {len(validation_df)}
Test subjects: {len(test_df)}

Best model
----------
{best_candidate_id}

Validation-selected threshold:
{best_threshold:.3f}

Untouched test result
---------------------
Accuracy: {test_metrics['accuracy']:.4f}
Balanced accuracy: {test_metrics['balanced_accuracy']:.4f}
Precision: {test_metrics['precision']:.4f}
Sensitivity: {test_metrics['recall_sensitivity']:.4f}
Specificity: {test_metrics['specificity']:.4f}
F1: {test_metrics['f1']:.4f}
ROC-AUC: {test_metrics['roc_auc']:.4f}
Average precision: {test_metrics['average_precision']:.4f}
MCC: {test_metrics['mcc']:.4f}

Confusion matrix
----------------
TN: {test_metrics['tn']}
FP: {test_metrics['fp']}
FN: {test_metrics['fn']}
TP: {test_metrics['tp']}

Important limitation
--------------------
This is a research baseline for MCI versus Control within the
Delaware corpus. It is not a clinically validated diagnostic test.

Model and threshold selection used validation data only.
The test set was used once for final evaluation.
"""

readme_path = (
    OUTPUT_ROOT
    / "README.txt"
)

readme_path.write_text(
    readme_text.strip() + "\n",
    encoding="utf-8",
)


# ============================================================
# CREATE FILE MANIFEST
# ============================================================

file_manifest_rows = []

for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if not path.is_file():
        continue

    file_manifest_rows.append({
        "relative_path": str(
            path.relative_to(
                OUTPUT_ROOT
            )
        ),
        "size_bytes": (
            path.stat().st_size
        ),
        "sha256": sha256_file(path),
    })

file_manifest = pd.DataFrame(
    file_manifest_rows
)

file_manifest_path = (
    OUTPUT_ROOT
    / "file_manifest.csv"
)

file_manifest.to_csv(
    file_manifest_path,
    index=False,
)


# ============================================================
# ZIP BACKUP
# ============================================================

zip_base = (
    LOCAL_ROOT
    / "Delaware_Recall_Audio_Baseline"
)

final_zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUTPUT_ROOT.parent,
    base_dir=OUTPUT_ROOT.name,
)

print()
print("Created ZIP:")
print(final_zip_path)


# ============================================================
# UPLOAD TO DRIVE
# ============================================================

output_drive_folder_id = (
    get_or_create_folder(
        DELAWARE_FOLDER_ID,
        "Delaware_Recall_Audio_Baseline",
    )
)

upload_paths = [
    Path(final_zip_path),
    model_path,
    metadata_path,
    feature_columns_path,
    feature_table_path,
    test_metrics_path,
    confidence_intervals_path,
    candidate_results_path,
    validation_predictions_path,
    test_predictions_path,
    AUDIT_ROOT
    / "best_validation_threshold_search.csv",
    AUDIT_ROOT
    / "test_false_positives.csv",
    AUDIT_ROOT
    / "test_false_negatives.csv",
    audio_quality_audit_path,
    failed_features_path,
    readme_path,
    file_manifest_path,
]

if feature_importance_path.exists():
    upload_paths.append(
        feature_importance_path
    )

print()
print("=" * 100)
print("UPLOADING AUDIO BASELINE")
print("=" * 100)

for local_path in upload_paths:
    result = upload_file(
        local_path,
        output_drive_folder_id,
    )

    print(
        "Uploaded:",
        result["name"],
    )


# ============================================================
# DONE
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print("Best model:")
print(best_candidate_id)

print()
print("Validation-selected threshold:")
print(best_threshold)

print()
print("Final untouched test metrics:")
print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)

print()
print("Transcript baseline benchmark:")
print(
    "Balanced accuracy: 0.6489"
)
print(
    "ROC-AUC: 0.6356"
)

print()
print("Use this ZIP:")
print(
    "Delaware_Recall_Audio_Baseline.zip"
)

In [ ]:
# ============================================================
# DELAWARE RECALL LATE FUSION + SAFE DRIVE CLEANUP
# Uses only the linked Google Drive folder ID
# Works after a runtime restart
# ============================================================

!pip -q install google-api-python-client scikit-learn

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

from pathlib import Path

import hashlib
import json
import shutil
import zipfile

import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DELAWARE_FOLDER_ID = "1RV-tYijUIOBV4O7jb8RcvnWC3RdV8U5J"

RANDOM_SEED = 42
BOOTSTRAP_RUNS = 2000

# The user requested removal of obsolete intermediate folders.
DELETE_OLD_INTERMEDIATES = True

LOCAL_ROOT = Path(
    "/content/delaware_recall_fusion"
)

INPUT_ROOT = LOCAL_ROOT / "input"

OUTPUT_ROOT = (
    LOCAL_ROOT
    / "Delaware_Recall_Multimodal_Fusion"
)

AUDIT_ROOT = OUTPUT_ROOT / "audit"
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"

for path in [
    INPUT_ROOT,
    OUTPUT_ROOT,
    AUDIT_ROOT,
    PREDICTION_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

drive_service = build(
    "drive",
    "v3",
)


# ============================================================
# DRIVE UTILITIES
# ============================================================

def list_folder(folder_id):
    items = []
    page_token = None

    while True:
        response = (
            drive_service.files()
            .list(
                q=(
                    f"'{folder_id}' in parents "
                    "and trashed = false"
                ),
                fields=(
                    "nextPageToken,"
                    "files("
                    "id,name,mimeType,size,"
                    "modifiedTime,parents"
                    ")"
                ),
                pageToken=page_token,
                pageSize=1000,
            )
            .execute()
        )

        items.extend(
            response.get("files", [])
        )

        page_token = response.get(
            "nextPageToken"
        )

        if not page_token:
            break

    return items


def walk_drive_folder(
    folder_id,
    relative_path=Path(),
):
    records = []

    for item in list_folder(folder_id):
        current_path = (
            relative_path / item["name"]
        )

        record = {
            "file_id": item["id"],
            "name": item["name"],
            "relative_path": str(
                current_path
            ),
            "mime_type": item["mimeType"],
            "size": int(
                item.get("size", 0)
            ),
            "modified_time": item.get(
                "modifiedTime"
            ),
            "parent_id": folder_id,
        }

        records.append(record)

        if (
            item["mimeType"]
            == "application/vnd.google-apps.folder"
        ):
            records.extend(
                walk_drive_folder(
                    item["id"],
                    current_path,
                )
            )

    return records


def download_drive_file(
    file_id,
    destination,
):
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    request = (
        drive_service.files()
        .get_media(fileId=file_id)
    )

    with destination.open("wb") as output_file:
        downloader = MediaIoBaseDownload(
            output_file,
            request,
            chunksize=16 * 1024 * 1024,
        )

        done = False

        while not done:
            _, done = downloader.next_chunk()

    return destination


def get_or_create_folder(
    parent_id,
    folder_name,
):
    safe_name = folder_name.replace(
        "'",
        "\\'",
    )

    response = (
        drive_service.files()
        .list(
            q=(
                f"'{parent_id}' in parents and "
                f"name = '{safe_name}' and "
                "mimeType = "
                "'application/vnd.google-apps.folder' "
                "and trashed = false"
            ),
            fields="files(id,name)",
        )
        .execute()
    )

    matches = response.get(
        "files",
        [],
    )

    if matches:
        return matches[0]["id"]

    created = (
        drive_service.files()
        .create(
            body={
                "name": folder_name,
                "mimeType": (
                    "application/"
                    "vnd.google-apps.folder"
                ),
                "parents": [parent_id],
            },
            fields="id",
        )
        .execute()
    )

    return created["id"]


def upload_file(
    local_path,
    parent_id,
):
    local_path = Path(local_path)

    result = (
        drive_service.files()
        .create(
            body={
                "name": local_path.name,
                "parents": [parent_id],
            },
            media_body=MediaFileUpload(
                str(local_path),
                resumable=True,
            ),
            fields="id,name",
        )
        .execute()
    )

    return result


def move_to_trash(file_id):
    return (
        drive_service.files()
        .update(
            fileId=file_id,
            body={"trashed": True},
            fields="id,name,trashed",
        )
        .execute()
    )


# ============================================================
# FILE UTILITIES
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def find_latest_file(
    records,
    required_parent_folder,
    filename,
):
    expected_suffix = (
        f"{required_parent_folder}/{filename}"
    ).lower()

    candidates = [
        record
        for record in records
        if (
            record["mime_type"]
            != "application/vnd.google-apps.folder"
            and record["relative_path"]
            .replace("\\", "/")
            .lower()
            .endswith(expected_suffix)
        )
    ]

    if not candidates:
        raise FileNotFoundError(
            f"Could not find {expected_suffix}"
        )

    candidates = sorted(
        candidates,
        key=lambda item: (
            item.get("modified_time") or ""
        ),
        reverse=True,
    )

    return candidates[0]


# ============================================================
# INDEX LINKED FOLDER
# ============================================================

print("=" * 100)
print("INDEXING LINKED DRIVE FOLDER")
print("=" * 100)

drive_records = walk_drive_folder(
    DELAWARE_FOLDER_ID
)

print(
    "Drive entries indexed:",
    len(drive_records),
)


# ============================================================
# LOCATE SAVED PREDICTIONS
# ============================================================

required_files = {
    "transcript_validation": (
        "Delaware_Recall_Transcript_Baseline",
        "validation_predictions.csv",
    ),
    "transcript_test": (
        "Delaware_Recall_Transcript_Baseline",
        "test_predictions.csv",
    ),
    "audio_validation": (
        "Delaware_Recall_Audio_Baseline",
        "validation_predictions.csv",
    ),
    "audio_test": (
        "Delaware_Recall_Audio_Baseline",
        "test_predictions.csv",
    ),
}

located_files = {}

for key, (
    parent_folder,
    filename,
) in required_files.items():

    record = find_latest_file(
        drive_records,
        parent_folder,
        filename,
    )

    local_path = (
        INPUT_ROOT
        / key
        / filename
    )

    download_drive_file(
        record["file_id"],
        local_path,
    )

    located_files[key] = local_path

    print(
        "Downloaded:",
        record["relative_path"],
    )


# ============================================================
# LOAD AND STANDARDIZE PREDICTIONS
# ============================================================

transcript_validation = pd.read_csv(
    located_files["transcript_validation"]
)

transcript_test = pd.read_csv(
    located_files["transcript_test"]
)

audio_validation = pd.read_csv(
    located_files["audio_validation"]
)

audio_test = pd.read_csv(
    located_files["audio_test"]
)

for dataframe in [
    transcript_validation,
    transcript_test,
    audio_validation,
    audio_test,
]:
    dataframe["subject_id"] = (
        dataframe["subject_id"]
        .astype(str)
    )

    dataframe["recording_id"] = (
        dataframe["recording_id"]
        .astype(str)
    )


def prepare_modality(
    dataframe,
    probability_name,
):
    required_columns = {
        "subject_id",
        "recording_id",
        "diagnosis",
        "binary_label",
        "predicted_probability_MCI",
    }

    missing = (
        required_columns
        - set(dataframe.columns)
    )

    if missing:
        raise ValueError(
            f"Missing columns: {sorted(missing)}"
        )

    result = dataframe[
        [
            "subject_id",
            "recording_id",
            "diagnosis",
            "binary_label",
            "predicted_probability_MCI",
        ]
    ].copy()

    result = result.rename(
        columns={
            "predicted_probability_MCI": (
                probability_name
            )
        }
    )

    return result


transcript_validation = prepare_modality(
    transcript_validation,
    "transcript_probability",
)

transcript_test = prepare_modality(
    transcript_test,
    "transcript_probability",
)

audio_validation = prepare_modality(
    audio_validation,
    "audio_probability",
)

audio_test = prepare_modality(
    audio_test,
    "audio_probability",
)


# ============================================================
# MERGE MODALITIES
# ============================================================

merge_keys = [
    "subject_id",
    "recording_id",
    "diagnosis",
    "binary_label",
]

validation = transcript_validation.merge(
    audio_validation,
    on=merge_keys,
    how="inner",
    validate="one_to_one",
)

test = transcript_test.merge(
    audio_test,
    on=merge_keys,
    how="inner",
    validate="one_to_one",
)

if len(validation) != len(
    transcript_validation
):
    raise RuntimeError(
        "Validation predictions did not match "
        "one-to-one across modalities."
    )

if len(test) != len(
    transcript_test
):
    raise RuntimeError(
        "Test predictions did not match "
        "one-to-one across modalities."
    )

print()
print("Validation subjects merged:", len(validation))
print("Test subjects merged:", len(test))


# ============================================================
# METRIC FUNCTIONS
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "recall_sensitivity": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "specificity": float(
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        ),
        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "average_precision": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                predictions,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


# ============================================================
# SELECT FUSION WEIGHT AND THRESHOLD ON VALIDATION ONLY
# ============================================================

print()
print("=" * 100)
print("VALIDATION-ONLY FUSION SEARCH")
print("=" * 100)

y_validation = (
    validation["binary_label"]
    .to_numpy(dtype=int)
)

search_rows = []

# alpha = transcript weight
# 1 - alpha = audio weight
for alpha in np.linspace(
    0.0,
    1.0,
    101,
):
    fused_probabilities = (
        alpha
        * validation[
            "transcript_probability"
        ].to_numpy()
        +
        (1.0 - alpha)
        * validation[
            "audio_probability"
        ].to_numpy()
    )

    for threshold in np.linspace(
        0.05,
        0.95,
        181,
    ):
        metrics = calculate_metrics(
            y_validation,
            fused_probabilities,
            threshold,
        )

        search_rows.append({
            "transcript_weight": float(alpha),
            "audio_weight": float(
                1.0 - alpha
            ),
            **metrics,
        })


fusion_search = pd.DataFrame(
    search_rows
)

fusion_search[
    "distance_from_half"
] = (
    fusion_search["threshold"]
    - 0.5
).abs()

# Primary selection:
# 1. validation balanced accuracy
# 2. validation ROC-AUC
# 3. F1
# 4. MCC
# 5. simpler preference toward transcript-only,
#    because the audio baseline was weak
# 6. threshold nearest 0.5
fusion_search = (
    fusion_search
    .sort_values(
        [
            "balanced_accuracy",
            "roc_auc",
            "f1",
            "mcc",
            "transcript_weight",
            "distance_from_half",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

fusion_search_path = (
    AUDIT_ROOT
    / "validation_fusion_search.csv"
)

fusion_search.to_csv(
    fusion_search_path,
    index=False,
)

best_row = fusion_search.iloc[0]

best_transcript_weight = float(
    best_row["transcript_weight"]
)

best_audio_weight = float(
    best_row["audio_weight"]
)

best_threshold = float(
    best_row["threshold"]
)

print(
    "Selected transcript weight:",
    best_transcript_weight,
)

print(
    "Selected audio weight:",
    best_audio_weight,
)

print(
    "Selected threshold:",
    best_threshold,
)

print()
print("Best validation metrics:")

print(
    json.dumps(
        {
            key: (
                float(best_row[key])
                if key not in {
                    "tn",
                    "fp",
                    "fn",
                    "tp",
                }
                else int(best_row[key])
            )
            for key in [
                "accuracy",
                "balanced_accuracy",
                "precision",
                "recall_sensitivity",
                "specificity",
                "f1",
                "roc_auc",
                "average_precision",
                "mcc",
                "tn",
                "fp",
                "fn",
                "tp",
            ]
        },
        indent=2,
    )
)


# ============================================================
# SAVE VALIDATION PREDICTIONS
# ============================================================

validation[
    "fused_probability_MCI"
] = (
    best_transcript_weight
    * validation[
        "transcript_probability"
    ]
    +
    best_audio_weight
    * validation[
        "audio_probability"
    ]
)

validation[
    "predicted_label"
] = (
    validation[
        "fused_probability_MCI"
    ]
    >= best_threshold
).astype(int)

validation[
    "correct"
] = (
    validation["binary_label"]
    == validation["predicted_label"]
)

validation_predictions_path = (
    PREDICTION_ROOT
    / "validation_fusion_predictions.csv"
)

validation.to_csv(
    validation_predictions_path,
    index=False,
)


# ============================================================
# FINAL TEST EVALUATION
# ============================================================

print()
print("=" * 100)
print("FINAL UNTOUCHED FUSION TEST")
print("=" * 100)

y_test = (
    test["binary_label"]
    .to_numpy(dtype=int)
)

test[
    "fused_probability_MCI"
] = (
    best_transcript_weight
    * test[
        "transcript_probability"
    ]
    +
    best_audio_weight
    * test[
        "audio_probability"
    ]
)

test_probabilities = (
    test["fused_probability_MCI"]
    .to_numpy()
)

test_metrics = calculate_metrics(
    y_test,
    test_probabilities,
    best_threshold,
)

test["predicted_label"] = (
    test_probabilities
    >= best_threshold
).astype(int)

test["correct"] = (
    test["binary_label"]
    == test["predicted_label"]
)

print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)

test_predictions_path = (
    PREDICTION_ROOT
    / "test_fusion_predictions.csv"
)

test.to_csv(
    test_predictions_path,
    index=False,
)


# ============================================================
# COMPARE ALL THREE MODELS ON THE SAME TEST SUBJECTS
# ============================================================

transcript_test_metrics = calculate_metrics(
    y_test,
    test[
        "transcript_probability"
    ].to_numpy(),
    0.495,
)

audio_test_metrics = calculate_metrics(
    y_test,
    test[
        "audio_probability"
    ].to_numpy(),
    0.35,
)

comparison_rows = []

for model_name, metrics in [
    (
        "transcript_only",
        transcript_test_metrics,
    ),
    (
        "audio_only",
        audio_test_metrics,
    ),
    (
        "late_fusion",
        test_metrics,
    ),
]:
    comparison_rows.append({
        "model": model_name,
        **metrics,
    })

model_comparison = pd.DataFrame(
    comparison_rows
)

comparison_path = (
    OUTPUT_ROOT
    / "test_model_comparison.csv"
)

model_comparison.to_csv(
    comparison_path,
    index=False,
)

display(
    model_comparison[
        [
            "model",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall_sensitivity",
            "specificity",
            "f1",
            "roc_auc",
            "average_precision",
            "mcc",
        ]
    ]
)


# ============================================================
# BOOTSTRAP FUSION CONFIDENCE INTERVALS
# ============================================================

print()
print("=" * 100)
print("BOOTSTRAP FUSION CONFIDENCE INTERVALS")
print("=" * 100)

rng = np.random.default_rng(
    RANDOM_SEED
)

bootstrap_rows = []

attempts = 0
maximum_attempts = (
    BOOTSTRAP_RUNS * 20
)

while (
    len(bootstrap_rows)
    < BOOTSTRAP_RUNS
    and attempts < maximum_attempts
):
    attempts += 1

    sampled_indices = rng.integers(
        0,
        len(y_test),
        size=len(y_test),
    )

    sampled_y = y_test[
        sampled_indices
    ]

    if len(np.unique(sampled_y)) < 2:
        continue

    sampled_probabilities = (
        test_probabilities[
            sampled_indices
        ]
    )

    bootstrap_rows.append(
        calculate_metrics(
            sampled_y,
            sampled_probabilities,
            best_threshold,
        )
    )

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

confidence_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1",
    "roc_auc",
    "average_precision",
    "mcc",
]

confidence_rows = []

for metric_name in confidence_metrics:
    values = (
        bootstrap_results[
            metric_name
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
        .to_numpy()
    )

    confidence_rows.append({
        "metric": metric_name,
        "point_estimate": (
            test_metrics[
                metric_name
            ]
        ),
        "bootstrap_mean": float(
            np.mean(values)
        ),
        "ci_95_lower": float(
            np.percentile(
                values,
                2.5,
            )
        ),
        "ci_95_upper": float(
            np.percentile(
                values,
                97.5,
            )
        ),
        "valid_bootstrap_runs": int(
            len(values)
        ),
    })

confidence_intervals = pd.DataFrame(
    confidence_rows
)

confidence_intervals_path = (
    OUTPUT_ROOT
    / "test_bootstrap_confidence_intervals.csv"
)

confidence_intervals.to_csv(
    confidence_intervals_path,
    index=False,
)

display(confidence_intervals)


# ============================================================
# SAVE FUSION CONFIGURATION
# ============================================================

fusion_configuration = {
    "dataset": "Delaware",
    "task": "Cinderella recall",
    "classification_target": (
        "MCI versus Control"
    ),
    "fusion_type": (
        "validation-selected weighted "
        "probability late fusion"
    ),
    "transcript_weight": (
        best_transcript_weight
    ),
    "audio_weight": (
        best_audio_weight
    ),
    "classification_threshold": (
        best_threshold
    ),
    "random_seed": RANDOM_SEED,
    "validation_subjects": int(
        len(validation)
    ),
    "test_subjects": int(
        len(test)
    ),
    "test_metrics": test_metrics,
    "note": (
        "Weights and threshold were selected "
        "using validation data only."
    ),
}

configuration_path = (
    OUTPUT_ROOT
    / "fusion_configuration.json"
)

configuration_path.write_text(
    json.dumps(
        fusion_configuration,
        indent=2,
    ),
    encoding="utf-8",
)

test_metrics_path = (
    OUTPUT_ROOT
    / "test_metrics.csv"
)

pd.DataFrame([
    {
        "model": "late_fusion",
        "transcript_weight": (
            best_transcript_weight
        ),
        "audio_weight": (
            best_audio_weight
        ),
        **test_metrics,
    }
]).to_csv(
    test_metrics_path,
    index=False,
)


# ============================================================
# README
# ============================================================

readme_text = f"""
DELAWARE RECALL MULTIMODAL LATE FUSION
======================================

Task
----
Cinderella recall: MCI versus Control.

Fusion
------
Transcript probability weight:
{best_transcript_weight:.3f}

Audio probability weight:
{best_audio_weight:.3f}

Validation-selected threshold:
{best_threshold:.3f}

Untouched fusion test result
----------------------------
Accuracy: {test_metrics['accuracy']:.4f}
Balanced accuracy: {test_metrics['balanced_accuracy']:.4f}
Precision: {test_metrics['precision']:.4f}
Sensitivity: {test_metrics['recall_sensitivity']:.4f}
Specificity: {test_metrics['specificity']:.4f}
F1: {test_metrics['f1']:.4f}
ROC-AUC: {test_metrics['roc_auc']:.4f}
Average precision: {test_metrics['average_precision']:.4f}
MCC: {test_metrics['mcc']:.4f}

Confusion matrix
----------------
TN: {test_metrics['tn']}
FP: {test_metrics['fp']}
FN: {test_metrics['fn']}
TP: {test_metrics['tp']}

Methodological note
-------------------
Fusion weights and threshold were selected on validation data only.
The test set was used for final evaluation.

This is a research classifier, not a clinical diagnostic system.
"""

readme_path = (
    OUTPUT_ROOT
    / "README.txt"
)

readme_path.write_text(
    readme_text.strip() + "\n",
    encoding="utf-8",
)


# ============================================================
# FILE MANIFEST AND ZIP
# ============================================================

file_manifest_rows = []

for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if not path.is_file():
        continue

    file_manifest_rows.append({
        "relative_path": str(
            path.relative_to(
                OUTPUT_ROOT
            )
        ),
        "size_bytes": (
            path.stat().st_size
        ),
        "sha256": sha256_file(path),
    })

file_manifest = pd.DataFrame(
    file_manifest_rows
)

file_manifest_path = (
    OUTPUT_ROOT
    / "file_manifest.csv"
)

file_manifest.to_csv(
    file_manifest_path,
    index=False,
)

zip_base = (
    LOCAL_ROOT
    / "Delaware_Recall_Multimodal_Fusion"
)

fusion_zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUTPUT_ROOT.parent,
    base_dir=OUTPUT_ROOT.name,
)

print()
print("Created fusion ZIP:")
print(fusion_zip_path)


# ============================================================
# UPLOAD FUSION RESULTS
# ============================================================

output_drive_folder_id = (
    get_or_create_folder(
        DELAWARE_FOLDER_ID,
        "Delaware_Recall_Multimodal_Fusion",
    )
)

upload_paths = [
    Path(fusion_zip_path),
    configuration_path,
    test_metrics_path,
    comparison_path,
    confidence_intervals_path,
    validation_predictions_path,
    test_predictions_path,
    fusion_search_path,
    readme_path,
    file_manifest_path,
]

print()
print("=" * 100)
print("UPLOADING FUSION RESULTS")
print("=" * 100)

for local_path in upload_paths:
    result = upload_file(
        local_path,
        output_drive_folder_id,
    )

    print(
        "Uploaded:",
        result["name"],
    )


# ============================================================
# DELETE OBSOLETE INTERMEDIATE DRIVE FOLDERS
# ============================================================

# These are redundant after the clean subject-unique dataset,
# fixed splits, and baseline results were created.
#
# Raw Delaware data is intentionally preserved.
# Model splits and all model outputs are preserved.

obsolete_folder_names = {
    "Delaware_Recall_Processed",
    "Delaware_Recall_Final",
}

protected_folder_names = {
    "Delaware",
    "Delaware_Audio",
    "Delaware_Recall_Final_Subject_Unique",
    "Delaware_Recall_Model_Splits",
    "Delaware_Recall_Transcript_Baseline",
    "Delaware_Recall_Audio_Baseline",
    "Delaware_Recall_Multimodal_Fusion",
}

root_children = list_folder(
    DELAWARE_FOLDER_ID
)

cleanup_rows = []

print()
print("=" * 100)
print("DRIVE CLEANUP")
print("=" * 100)

for item in root_children:
    item_name = item["name"]

    if item_name in protected_folder_names:
        cleanup_rows.append({
            "name": item_name,
            "action": "preserved",
            "reason": "protected_required_folder",
        })

        continue

    if item_name in obsolete_folder_names:
        if DELETE_OLD_INTERMEDIATES:
            move_to_trash(
                item["id"]
            )

            cleanup_rows.append({
                "name": item_name,
                "action": "moved_to_trash",
                "reason": "obsolete_intermediate",
            })

            print(
                "Moved to Trash:",
                item_name,
            )

        else:
            cleanup_rows.append({
                "name": item_name,
                "action": "would_move_to_trash",
                "reason": "cleanup_disabled",
            })

            print(
                "Would move to Trash:",
                item_name,
            )

cleanup_report = pd.DataFrame(
    cleanup_rows
)

cleanup_report_path = (
    OUTPUT_ROOT
    / "drive_cleanup_report.csv"
)

cleanup_report.to_csv(
    cleanup_report_path,
    index=False,
)

upload_file(
    cleanup_report_path,
    output_drive_folder_id,
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Transcript weight:",
    best_transcript_weight,
)

print(
    "Audio weight:",
    best_audio_weight,
)

print(
    "Classification threshold:",
    best_threshold,
)

print()
print("Final fusion test metrics:")

print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)

print()
print("Saved:")
print(
    "Delaware_Recall_Multimodal_Fusion.zip"
)

print()
print("Obsolete folders moved to Trash:")
print(
    sorted(obsolete_folder_names)
)

In [ ]:
# ============================================================
# DELAWARE CINDERELLA RECALL-CONTENT MODEL
# Restart-safe: reloads everything from the linked Drive folder
#
# Models:
# 1. Recall-content features only
# 2. Character TF-IDF baseline
# 3. Recall-content + character TF-IDF
#
# Uses validation data for model/threshold selection.
# ============================================================

!pip -q install \
    google-api-python-client \
    sentence-transformers \
    scikit-learn \
    joblib

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import (
    MediaIoBaseDownload,
    MediaFileUpload,
)

from sentence_transformers import SentenceTransformer

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from pathlib import Path

import hashlib
import json
import math
import re
import shutil
import warnings
import zipfile

import joblib
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DELAWARE_FOLDER_ID = "1RV-tYijUIOBV4O7jb8RcvnWC3RdV8U5J"

RANDOM_SEED = 42
BOOTSTRAP_RUNS = 2000

LOCAL_ROOT = Path(
    "/content/delaware_recall_content_model"
)

INPUT_ROOT = LOCAL_ROOT / "input"
EXTRACT_ROOT = INPUT_ROOT / "extracted"

OUTPUT_ROOT = (
    LOCAL_ROOT
    / "Delaware_Recall_Content_Model"
)

FEATURE_ROOT = OUTPUT_ROOT / "features"
MODEL_ROOT = OUTPUT_ROOT / "model"
AUDIT_ROOT = OUTPUT_ROOT / "audit"
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"

for path in [
    INPUT_ROOT,
    EXTRACT_ROOT,
    OUTPUT_ROOT,
    FEATURE_ROOT,
    MODEL_ROOT,
    AUDIT_ROOT,
    PREDICTION_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

warnings.filterwarnings("ignore")

drive_service = build(
    "drive",
    "v3",
)


# ============================================================
# DRIVE UTILITIES
# ============================================================

def list_folder(folder_id):
    items = []
    page_token = None

    while True:
        response = (
            drive_service.files()
            .list(
                q=(
                    f"'{folder_id}' in parents "
                    "and trashed = false"
                ),
                fields=(
                    "nextPageToken,"
                    "files("
                    "id,name,mimeType,size,"
                    "modifiedTime,parents"
                    ")"
                ),
                pageToken=page_token,
                pageSize=1000,
            )
            .execute()
        )

        items.extend(
            response.get("files", [])
        )

        page_token = response.get(
            "nextPageToken"
        )

        if not page_token:
            break

    return items


def walk_drive_folder(
    folder_id,
    relative_path=Path(),
):
    records = []

    for item in list_folder(folder_id):
        current_path = (
            relative_path / item["name"]
        )

        if (
            item["mimeType"]
            == "application/vnd.google-apps.folder"
        ):
            records.extend(
                walk_drive_folder(
                    item["id"],
                    current_path,
                )
            )

        else:
            records.append({
                "file_id": item["id"],
                "name": item["name"],
                "relative_path": str(
                    current_path
                ),
                "mime_type": item["mimeType"],
                "size": int(
                    item.get("size", 0)
                ),
                "modified_time": item.get(
                    "modifiedTime"
                ),
            })

    return records


def download_drive_file(
    file_id,
    destination,
):
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    request = (
        drive_service.files()
        .get_media(fileId=file_id)
    )

    with destination.open("wb") as output_file:
        downloader = MediaIoBaseDownload(
            output_file,
            request,
            chunksize=16 * 1024 * 1024,
        )

        done = False

        while not done:
            _, done = downloader.next_chunk()

    return destination


def get_or_create_folder(
    parent_id,
    folder_name,
):
    safe_name = folder_name.replace(
        "'",
        "\\'",
    )

    response = (
        drive_service.files()
        .list(
            q=(
                f"'{parent_id}' in parents and "
                f"name = '{safe_name}' and "
                "mimeType = "
                "'application/vnd.google-apps.folder' "
                "and trashed = false"
            ),
            fields="files(id,name)",
        )
        .execute()
    )

    matches = response.get(
        "files",
        [],
    )

    if matches:
        return matches[0]["id"]

    created = (
        drive_service.files()
        .create(
            body={
                "name": folder_name,
                "mimeType": (
                    "application/"
                    "vnd.google-apps.folder"
                ),
                "parents": [parent_id],
            },
            fields="id",
        )
        .execute()
    )

    return created["id"]


def upload_file(
    local_path,
    parent_id,
):
    local_path = Path(local_path)

    result = (
        drive_service.files()
        .create(
            body={
                "name": local_path.name,
                "parents": [parent_id],
            },
            media_body=MediaFileUpload(
                str(local_path),
                resumable=True,
            ),
            fields="id,name",
        )
        .execute()
    )

    return result


# ============================================================
# FILE UTILITIES
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_text(path):
    return Path(path).read_text(
        encoding="utf-8",
        errors="replace",
    ).strip()


# ============================================================
# LOCATE MODEL-SPLIT ZIP
# ============================================================

print("=" * 100)
print("LOCATING DELAWARE MODEL SPLITS")
print("=" * 100)

drive_records = walk_drive_folder(
    DELAWARE_FOLDER_ID
)

zip_candidates = [
    record
    for record in drive_records
    if (
        record["name"].lower()
        == "delaware_recall_model_splits.zip"
    )
]

if not zip_candidates:
    raise FileNotFoundError(
        "Could not find "
        "Delaware_Recall_Model_Splits.zip."
    )

zip_candidates = sorted(
    zip_candidates,
    key=lambda item: (
        item.get("modified_time") or ""
    ),
    reverse=True,
)

selected_zip = zip_candidates[0]

print("Using:")
print(selected_zip["relative_path"])

local_zip = (
    INPUT_ROOT
    / "Delaware_Recall_Model_Splits.zip"
)

download_drive_file(
    selected_zip["file_id"],
    local_zip,
)

print("Downloaded:")
print(local_zip)


# ============================================================
# EXTRACT MODEL SPLITS
# ============================================================

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(
    local_zip,
    "r",
) as archive:
    archive.extractall(
        EXTRACT_ROOT
    )

manifest_candidates = list(
    EXTRACT_ROOT.rglob(
        "recall_model_split_manifest.csv"
    )
)

if not manifest_candidates:
    raise FileNotFoundError(
        "Missing recall_model_split_manifest.csv."
    )

SOURCE_ROOT = (
    manifest_candidates[0].parent
)

manifest = pd.read_csv(
    manifest_candidates[0]
)

manifest["subject_id"] = (
    manifest["subject_id"]
    .astype(str)
)

manifest["recording_id"] = (
    manifest["recording_id"]
    .astype(str)
)

print()
print("Rows loaded:", len(manifest))

display(
    manifest.groupby(
        ["split", "diagnosis"]
    )
    .size()
    .reset_index(name="recordings")
)


# ============================================================
# LOAD TRANSCRIPTS
# ============================================================

print()
print("=" * 100)
print("LOADING TRANSCRIPTS")
print("=" * 100)

rows = []

for index, row in enumerate(
    manifest.itertuples(index=False),
    start=1,
):
    transcript_path = (
        SOURCE_ROOT
        / str(row.transcript_path)
    )

    if not transcript_path.exists():
        raise FileNotFoundError(
            str(transcript_path)
        )

    text = read_text(
        transcript_path
    )

    if not text:
        raise ValueError(
            f"Empty transcript: {row.recording_id}"
        )

    rows.append({
        "split": row.split,
        "diagnosis": row.diagnosis,
        "binary_label": int(
            row.binary_label
        ),
        "subject_id": row.subject_id,
        "recording_id": row.recording_id,
        "transcript_path": str(
            row.transcript_path
        ),
        "duration_seconds": float(
            row.duration_seconds
        ),
        "text": text,
    })

    if (
        index % 50 == 0
        or index == len(manifest)
    ):
        print(
            f"Loaded {index}/{len(manifest)}"
        )

data = pd.DataFrame(rows)


# ============================================================
# CINDERELLA CONTENT UNITS
# ============================================================

# These are short semantic descriptions, not a full story text.
CONTENT_UNITS = {
    "cinderella_main_character":
        "Cinderella is the main girl in the story.",

    "father_remarries":
        "Cinderella's father marries another woman.",

    "cruel_stepmother":
        "The stepmother treats Cinderella badly.",

    "stepsisters":
        "Cinderella has two stepsisters.",

    "housework":
        "Cinderella is forced to clean or do household work.",

    "invitation_ball":
        "There is an invitation to a royal ball or palace event.",

    "prevented_from_going":
        "Cinderella is prevented from attending the ball.",

    "fairy_godmother":
        "A fairy godmother appears to help Cinderella.",

    "magic_transformation":
        "Magic changes Cinderella's clothes or appearance.",

    "pumpkin_carriage":
        "A pumpkin is transformed into a carriage.",

    "animals_transformed":
        "Mice or other animals are transformed into horses or helpers.",

    "glass_slippers":
        "Cinderella receives or wears glass slippers.",

    "midnight_deadline":
        "The magic will end at midnight.",

    "meets_prince":
        "Cinderella meets the prince.",

    "dances_with_prince":
        "Cinderella dances with the prince.",

    "runs_away_midnight":
        "Cinderella runs away when midnight arrives.",

    "loses_slipper":
        "Cinderella loses one slipper while leaving.",

    "prince_searches":
        "The prince searches for the woman who wore the slipper.",

    "slipper_does_not_fit_sisters":
        "The slipper does not fit the stepsisters.",

    "slipper_fits_cinderella":
        "The slipper fits Cinderella.",

    "recognition_reunion":
        "The prince recognizes or reunites with Cinderella.",

    "marriage_happy_ending":
        "Cinderella and the prince marry or live happily.",
}

CONTENT_NAMES = list(
    CONTENT_UNITS.keys()
)

CONTENT_DESCRIPTIONS = [
    CONTENT_UNITS[name]
    for name in CONTENT_NAMES
]


# ============================================================
# TEXT FEATURE HELPERS
# ============================================================

FILLERS = {
    "uh",
    "um",
    "erm",
    "hmm",
    "hm",
}

SEQUENCE_WORDS = {
    "first",
    "then",
    "next",
    "after",
    "afterward",
    "later",
    "finally",
    "before",
    "when",
    "while",
}

UNCERTAINTY_PHRASES = [
    "i don't know",
    "i do not know",
    "i can't remember",
    "i cannot remember",
    "i forgot",
    "maybe",
    "i think",
    "something like",
    "or something",
]

CORRECTION_PHRASES = [
    "i mean",
    "no wait",
    "or rather",
    "actually",
    "sorry",
]


def tokenize(text):
    return re.findall(
        r"\b[a-zA-Z']+\b",
        text.lower(),
    )


def split_sentences(text):
    sentences = re.split(
        r"(?<=[.!?])\s+|\n+",
        text,
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


def moving_average_ttr(
    tokens,
    window_size=50,
):
    if not tokens:
        return 0.0

    if len(tokens) <= window_size:
        return (
            len(set(tokens))
            / len(tokens)
        )

    ratios = []

    for start in range(
        0,
        len(tokens) - window_size + 1
    ):
        window = tokens[
            start:start + window_size
        ]

        ratios.append(
            len(set(window))
            / window_size
        )

    return float(
        np.mean(ratios)
    )


def adjacent_repetition_count(tokens):
    return sum(
        1
        for first, second
        in zip(tokens, tokens[1:])
        if first == second
    )


def repeated_ngram_count(
    tokens,
    n,
):
    if len(tokens) < n:
        return 0

    ngrams = [
        tuple(tokens[index:index+n])
        for index in range(
            len(tokens) - n + 1
        )
    ]

    counts = pd.Series(
        ngrams
    ).value_counts()

    return int(
        (counts - 1)
        .clip(lower=0)
        .sum()
    )


def count_phrase_occurrences(
    lower_text,
    phrases,
):
    return sum(
        lower_text.count(phrase)
        for phrase in phrases
    )


# ============================================================
# SEMANTIC CONTENT-UNIT MODEL
# ============================================================

print()
print("=" * 100)
print("LOADING SEMANTIC EMBEDDING MODEL")
print("=" * 100)

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

content_embeddings = (
    embedding_model.encode(
        CONTENT_DESCRIPTIONS,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
)

print(
    "Content units:",
    len(CONTENT_NAMES),
)


# ============================================================
# EXTRACT RECALL-CONTENT FEATURES
# ============================================================

print()
print("=" * 100)
print("EXTRACTING RECALL-CONTENT FEATURES")
print("=" * 100)

feature_rows = []

for index, row in enumerate(
    data.itertuples(index=False),
    start=1,
):
    text = row.text
    lower_text = text.lower()

    tokens = tokenize(text)
    sentences = split_sentences(text)

    if not sentences:
        sentences = [text]

    sentence_embeddings = (
        embedding_model.encode(
            sentences,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    )

    similarity_matrix = (
        sentence_embeddings
        @ content_embeddings.T
    )

    maximum_unit_similarities = (
        similarity_matrix.max(axis=0)
    )

    word_count = len(tokens)
    unique_word_count = len(
        set(tokens)
    )

    sentence_count = len(sentences)

    duration_minutes = max(
        row.duration_seconds / 60,
        1e-6,
    )

    record = {
        "split": row.split,
        "diagnosis": row.diagnosis,
        "binary_label": row.binary_label,
        "subject_id": row.subject_id,
        "recording_id": row.recording_id,
        "transcript_path": row.transcript_path,
        "text": text,

        "duration_seconds":
            row.duration_seconds,

        "word_count":
            word_count,

        "unique_word_count":
            unique_word_count,

        "type_token_ratio":
            (
                unique_word_count / word_count
                if word_count > 0
                else 0
            ),

        "mattr_50":
            moving_average_ttr(
                tokens,
                window_size=50,
            ),

        "sentence_count":
            sentence_count,

        "mean_sentence_words":
            (
                word_count / sentence_count
                if sentence_count > 0
                else 0
            ),

        "words_per_minute":
            word_count / duration_minutes,

        "filler_count":
            sum(
                token in FILLERS
                for token in tokens
            ),

        "filler_rate":
            (
                sum(
                    token in FILLERS
                    for token in tokens
                )
                / max(word_count, 1)
            ),

        "uncertainty_count":
            count_phrase_occurrences(
                lower_text,
                UNCERTAINTY_PHRASES,
            ),

        "correction_count":
            count_phrase_occurrences(
                lower_text,
                CORRECTION_PHRASES,
            ),

        "sequence_word_count":
            sum(
                token in SEQUENCE_WORDS
                for token in tokens
            ),

        "adjacent_repetition_count":
            adjacent_repetition_count(
                tokens
            ),

        "repeated_bigram_count":
            repeated_ngram_count(
                tokens,
                2,
            ),

        "repeated_trigram_count":
            repeated_ngram_count(
                tokens,
                3,
            ),

        "content_mean_similarity":
            float(
                np.mean(
                    maximum_unit_similarities
                )
            ),

        "content_max_similarity":
            float(
                np.max(
                    maximum_unit_similarities
                )
            ),

        "content_units_above_030":
            int(
                np.sum(
                    maximum_unit_similarities
                    >= 0.30
                )
            ),

        "content_units_above_035":
            int(
                np.sum(
                    maximum_unit_similarities
                    >= 0.35
                )
            ),

        "content_units_above_040":
            int(
                np.sum(
                    maximum_unit_similarities
                    >= 0.40
                )
            ),

        "content_units_above_045":
            int(
                np.sum(
                    maximum_unit_similarities
                    >= 0.45
                )
            ),

        "content_recall_efficiency":
            float(
                np.sum(
                    maximum_unit_similarities
                    >= 0.40
                )
                / duration_minutes
            ),
    }

    for content_name, similarity in zip(
        CONTENT_NAMES,
        maximum_unit_similarities,
    ):
        record[
            f"semantic_{content_name}"
        ] = float(similarity)

    feature_rows.append(record)

    if (
        index % 25 == 0
        or index == len(data)
    ):
        print(
            f"Processed {index}/{len(data)}"
        )


features = pd.DataFrame(
    feature_rows
)

feature_table_path = (
    FEATURE_ROOT
    / "recall_content_features.csv"
)

features.to_csv(
    feature_table_path,
    index=False,
)

metadata_columns = [
    "split",
    "diagnosis",
    "binary_label",
    "subject_id",
    "recording_id",
    "transcript_path",
    "text",
]

numeric_feature_columns = [
    column
    for column in features.columns
    if column not in metadata_columns
]

print()
print(
    "Numeric recall-content features:",
    len(numeric_feature_columns),
)


# ============================================================
# SPLIT DATA
# ============================================================

train_df = features[
    features["split"] == "train"
].copy()

validation_df = features[
    features["split"] == "validation"
].copy()

test_df = features[
    features["split"] == "test"
].copy()

y_train = (
    train_df["binary_label"]
    .to_numpy(dtype=int)
)

y_validation = (
    validation_df["binary_label"]
    .to_numpy(dtype=int)
)

y_test = (
    test_df["binary_label"]
    .to_numpy(dtype=int)
)

print()
print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))


# ============================================================
# METRIC FUNCTIONS
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "threshold": float(threshold),

        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),

        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),

        "recall_sensitivity": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),

        "specificity": float(
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        ),

        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),

        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),

        "average_precision": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),

        "mcc": float(
            matthews_corrcoef(
                y_true,
                predictions,
            )
        ),

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_threshold(
    y_true,
    probabilities,
):
    rows = []

    for threshold in np.linspace(
        0.05,
        0.95,
        181,
    ):
        rows.append(
            calculate_metrics(
                y_true,
                probabilities,
                threshold,
            )
        )

    table = pd.DataFrame(rows)

    table[
        "distance_from_half"
    ] = (
        table["threshold"]
        - 0.5
    ).abs()

    table = (
        table
        .sort_values(
            [
                "balanced_accuracy",
                "roc_auc",
                "f1",
                "mcc",
                "distance_from_half",
            ],
            ascending=[
                False,
                False,
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    return (
        float(
            table.loc[
                0,
                "threshold",
            ]
        ),
        table,
    )


# ============================================================
# MODEL BUILDERS
# ============================================================

def build_content_logistic(C):
    return Pipeline([
        (
            "features",
            ColumnTransformer(
                transformers=[
                    (
                        "numeric",
                        Pipeline([
                            (
                                "imputer",
                                SimpleImputer(
                                    strategy="median"
                                ),
                            ),
                            (
                                "scaler",
                                StandardScaler(),
                            ),
                        ]),
                        numeric_feature_columns,
                    ),
                ],
                remainder="drop",
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                C=C,
                class_weight="balanced",
                solver="liblinear",
                max_iter=10000,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


def build_content_rbf(C):
    return Pipeline([
        (
            "features",
            ColumnTransformer(
                transformers=[
                    (
                        "numeric",
                        Pipeline([
                            (
                                "imputer",
                                SimpleImputer(
                                    strategy="median"
                                ),
                            ),
                            (
                                "scaler",
                                StandardScaler(),
                            ),
                        ]),
                        numeric_feature_columns,
                    ),
                ],
                remainder="drop",
            ),
        ),
        (
            "classifier",
            SVC(
                C=C,
                kernel="rbf",
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


def build_content_forest(
    max_depth,
    min_samples_leaf,
):
    return Pipeline([
        (
            "features",
            ColumnTransformer(
                transformers=[
                    (
                        "numeric",
                        SimpleImputer(
                            strategy="median"
                        ),
                        numeric_feature_columns,
                    ),
                ],
                remainder="drop",
            ),
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=700,
                max_depth=max_depth,
                min_samples_leaf=(
                    min_samples_leaf
                ),
                class_weight="balanced",
                max_features="sqrt",
                random_state=RANDOM_SEED,
                n_jobs=-1,
            ),
        ),
    ])


def build_character_model(C):
    return Pipeline([
        (
            "features",
            ColumnTransformer(
                transformers=[
                    (
                        "character",
                        TfidfVectorizer(
                            analyzer="char_wb",
                            lowercase=True,
                            ngram_range=(3, 5),
                            min_df=2,
                            sublinear_tf=True,
                            max_features=40000,
                        ),
                        "text",
                    ),
                ],
                remainder="drop",
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                C=C,
                class_weight="balanced",
                solver="liblinear",
                max_iter=10000,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


def build_combined_model(
    C,
    numeric_weight,
):
    transformer = ColumnTransformer(
        transformers=[
            (
                "character",
                TfidfVectorizer(
                    analyzer="char_wb",
                    lowercase=True,
                    ngram_range=(3, 5),
                    min_df=2,
                    sublinear_tf=True,
                    max_features=40000,
                ),
                "text",
            ),
            (
                "numeric",
                Pipeline([
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="median"
                        ),
                    ),
                    (
                        "scaler",
                        StandardScaler(),
                    ),
                ]),
                numeric_feature_columns,
            ),
        ],
        transformer_weights={
            "character": 1.0,
            "numeric": numeric_weight,
        },
        remainder="drop",
    )

    return Pipeline([
        (
            "features",
            transformer,
        ),
        (
            "classifier",
            LogisticRegression(
                C=C,
                class_weight="balanced",
                solver="liblinear",
                max_iter=10000,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


# ============================================================
# CANDIDATE DEFINITIONS
# ============================================================

candidate_definitions = []

for C in [
    0.01,
    0.05,
    0.1,
    0.25,
    0.5,
    1.0,
    2.0,
    4.0,
]:
    candidate_definitions.append({
        "candidate_id":
            f"content_logistic_C_{C}",

        "family":
            "content_logistic",

        "builder":
            lambda C=C:
            build_content_logistic(C),
    })

for C in [
    0.1,
    0.25,
    0.5,
    1.0,
    2.0,
    4.0,
    8.0,
]:
    candidate_definitions.append({
        "candidate_id":
            f"content_rbf_C_{C}",

        "family":
            "content_rbf",

        "builder":
            lambda C=C:
            build_content_rbf(C),
    })

for max_depth in [
    None,
    4,
    8,
    12,
]:
    for min_samples_leaf in [
        2,
        4,
        8,
    ]:
        depth_name = (
            "none"
            if max_depth is None
            else str(max_depth)
        )

        candidate_definitions.append({
            "candidate_id": (
                f"content_forest_depth_"
                f"{depth_name}_leaf_"
                f"{min_samples_leaf}"
            ),

            "family":
                "content_forest",

            "builder": (
                lambda
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf:
                build_content_forest(
                    max_depth,
                    min_samples_leaf,
                )
            ),
        })

for C in [
    0.1,
    0.25,
    0.5,
    1.0,
]:
    candidate_definitions.append({
        "candidate_id":
            f"character_baseline_C_{C}",

        "family":
            "character_baseline",

        "builder":
            lambda C=C:
            build_character_model(C),
    })

for C in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0,
    2.0,
]:
    for numeric_weight in [
        0.1,
        0.25,
        0.5,
        1.0,
        2.0,
    ]:
        candidate_definitions.append({
            "candidate_id": (
                f"combined_C_{C}_"
                f"numeric_{numeric_weight}"
            ),

            "family":
                "combined_content_character",

            "builder": (
                lambda
                C=C,
                numeric_weight=numeric_weight:
                build_combined_model(
                    C,
                    numeric_weight,
                )
            ),
        })


# ============================================================
# MODEL SELECTION ON VALIDATION ONLY
# ============================================================

print()
print("=" * 100)
print("MODEL SELECTION ON VALIDATION SET")
print("=" * 100)

candidate_rows = []
candidate_models = {}
threshold_tables = {}

for candidate in candidate_definitions:
    candidate_id = candidate[
        "candidate_id"
    ]

    print("Training:", candidate_id)

    model = candidate["builder"]()

    model.fit(
        train_df,
        y_train,
    )

    validation_probabilities = (
        model.predict_proba(
            validation_df
        )[:, 1]
    )

    (
        selected_threshold,
        threshold_table,
    ) = select_threshold(
        y_validation,
        validation_probabilities,
    )

    metrics = calculate_metrics(
        y_validation,
        validation_probabilities,
        selected_threshold,
    )

    candidate_rows.append({
        "candidate_id": candidate_id,
        "family": candidate["family"],
        **metrics,
    })

    candidate_models[
        candidate_id
    ] = model

    threshold_tables[
        candidate_id
    ] = threshold_table


candidate_results = pd.DataFrame(
    candidate_rows
)

candidate_results[
    "distance_from_half"
] = (
    candidate_results["threshold"]
    - 0.5
).abs()

candidate_results = (
    candidate_results
    .sort_values(
        [
            "balanced_accuracy",
            "roc_auc",
            "f1",
            "mcc",
            "distance_from_half",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

candidate_results_path = (
    AUDIT_ROOT
    / "validation_model_selection.csv"
)

candidate_results.to_csv(
    candidate_results_path,
    index=False,
)

display(
    candidate_results.head(30)[
        [
            "candidate_id",
            "family",
            "threshold",
            "accuracy",
            "balanced_accuracy",
            "precision",
            "recall_sensitivity",
            "specificity",
            "f1",
            "roc_auc",
            "average_precision",
            "mcc",
        ]
    ]
)

best_candidate_id = str(
    candidate_results.loc[
        0,
        "candidate_id",
    ]
)

best_family = str(
    candidate_results.loc[
        0,
        "family",
    ]
)

best_threshold = float(
    candidate_results.loc[
        0,
        "threshold",
    ]
)

best_model = candidate_models[
    best_candidate_id
]

threshold_tables[
    best_candidate_id
].to_csv(
    AUDIT_ROOT
    / "best_validation_threshold_search.csv",
    index=False,
)

print()
print("Best candidate:")
print(best_candidate_id)

print("Best family:")
print(best_family)

print("Selected threshold:")
print(best_threshold)


# ============================================================
# VALIDATION PREDICTIONS
# ============================================================

validation_probabilities = (
    best_model.predict_proba(
        validation_df
    )[:, 1]
)

validation_predictions = (
    validation_probabilities
    >= best_threshold
).astype(int)

validation_output = validation_df[
    [
        "subject_id",
        "recording_id",
        "diagnosis",
        "binary_label",
        "transcript_path",
    ]
].copy()

validation_output[
    "predicted_probability_MCI"
] = validation_probabilities

validation_output[
    "predicted_label"
] = validation_predictions

validation_output[
    "correct"
] = (
    validation_output["binary_label"]
    == validation_output["predicted_label"]
)

validation_predictions_path = (
    PREDICTION_ROOT
    / "validation_predictions.csv"
)

validation_output.to_csv(
    validation_predictions_path,
    index=False,
)


# ============================================================
# FINAL TEST EVALUATION
# ============================================================

print()
print("=" * 100)
print("FINAL TEST EVALUATION")
print("=" * 100)

test_probabilities = (
    best_model.predict_proba(
        test_df
    )[:, 1]
)

test_predictions = (
    test_probabilities
    >= best_threshold
).astype(int)

test_metrics = calculate_metrics(
    y_test,
    test_probabilities,
    best_threshold,
)

print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)

test_output = test_df[
    [
        "subject_id",
        "recording_id",
        "diagnosis",
        "binary_label",
        "transcript_path",
    ]
].copy()

test_output[
    "predicted_probability_MCI"
] = test_probabilities

test_output[
    "predicted_label"
] = test_predictions

test_output[
    "correct"
] = (
    test_output["binary_label"]
    == test_output["predicted_label"]
)

test_predictions_path = (
    PREDICTION_ROOT
    / "test_predictions.csv"
)

test_output.to_csv(
    test_predictions_path,
    index=False,
)


# ============================================================
# BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

print()
print("=" * 100)
print("BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 100)

rng = np.random.default_rng(
    RANDOM_SEED
)

bootstrap_rows = []

attempts = 0
max_attempts = (
    BOOTSTRAP_RUNS * 20
)

while (
    len(bootstrap_rows)
    < BOOTSTRAP_RUNS
    and attempts < max_attempts
):
    attempts += 1

    indices = rng.integers(
        0,
        len(y_test),
        size=len(y_test),
    )

    sampled_y = y_test[indices]

    if len(np.unique(sampled_y)) < 2:
        continue

    sampled_probabilities = (
        test_probabilities[indices]
    )

    bootstrap_rows.append(
        calculate_metrics(
            sampled_y,
            sampled_probabilities,
            best_threshold,
        )
    )

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

confidence_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1",
    "roc_auc",
    "average_precision",
    "mcc",
]

confidence_rows = []

for metric_name in confidence_metrics:
    values = (
        bootstrap_results[
            metric_name
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
        .to_numpy()
    )

    confidence_rows.append({
        "metric": metric_name,

        "point_estimate":
            test_metrics[
                metric_name
            ],

        "bootstrap_mean":
            float(
                np.mean(values)
            ),

        "ci_95_lower":
            float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

        "ci_95_upper":
            float(
                np.percentile(
                    values,
                    97.5,
                )
            ),

        "valid_bootstrap_runs":
            int(len(values)),
    })

confidence_intervals = pd.DataFrame(
    confidence_rows
)

confidence_intervals_path = (
    OUTPUT_ROOT
    / "test_bootstrap_confidence_intervals.csv"
)

confidence_intervals.to_csv(
    confidence_intervals_path,
    index=False,
)

display(confidence_intervals)


# ============================================================
# FEATURE GROUP COMPARISON
# ============================================================

family_summary = (
    candidate_results
    .groupby("family")
    .first()
    .reset_index()
    [
        [
            "family",
            "candidate_id",
            "balanced_accuracy",
            "roc_auc",
            "f1",
            "mcc",
        ]
    ]
    .sort_values(
        "balanced_accuracy",
        ascending=False,
    )
)

family_summary_path = (
    OUTPUT_ROOT
    / "validation_family_summary.csv"
)

family_summary.to_csv(
    family_summary_path,
    index=False,
)

display(family_summary)


# ============================================================
# ERROR ANALYSIS
# ============================================================

false_positives = test_output[
    (
        test_output["binary_label"] == 0
    )
    &
    (
        test_output["predicted_label"] == 1
    )
].copy()

false_negatives = test_output[
    (
        test_output["binary_label"] == 1
    )
    &
    (
        test_output["predicted_label"] == 0
    )
].copy()

false_positives.to_csv(
    AUDIT_ROOT
    / "test_false_positives.csv",
    index=False,
)

false_negatives.to_csv(
    AUDIT_ROOT
    / "test_false_negatives.csv",
    index=False,
)


# ============================================================
# SAVE MODEL AND METADATA
# ============================================================

model_path = (
    MODEL_ROOT
    / "recall_content_model.joblib"
)

joblib.dump(
    best_model,
    model_path,
)

content_units_path = (
    MODEL_ROOT
    / "cinderella_content_units.json"
)

content_units_path.write_text(
    json.dumps(
        CONTENT_UNITS,
        indent=2,
    ),
    encoding="utf-8",
)

numeric_columns_path = (
    MODEL_ROOT
    / "numeric_feature_columns.json"
)

numeric_columns_path.write_text(
    json.dumps(
        numeric_feature_columns,
        indent=2,
    ),
    encoding="utf-8",
)

model_metadata = {
    "dataset": "Delaware",
    "task": "Cinderella recall",
    "classification_target":
        "MCI versus Control",

    "best_candidate_id":
        best_candidate_id,

    "best_model_family":
        best_family,

    "selected_threshold":
        best_threshold,

    "embedding_model":
        "sentence-transformers/all-MiniLM-L6-v2",

    "number_of_content_units":
        len(CONTENT_UNITS),

    "number_of_numeric_features":
        len(numeric_feature_columns),

    "train_subjects":
        int(len(train_df)),

    "validation_subjects":
        int(len(validation_df)),

    "test_subjects":
        int(len(test_df)),

    "test_metrics":
        test_metrics,

    "previous_baselines": {
        "character_tfidf_balanced_accuracy":
            0.6488888888888888,

        "character_tfidf_roc_auc":
            0.6355555555555555,

        "audio_egemaps_balanced_accuracy":
            0.4577777777777778,

        "audio_egemaps_roc_auc":
            0.5266666666666666,
    },

    "limitations": [
        (
            "The test set has already been evaluated "
            "by earlier baselines, so it should not be "
            "used for further model tuning."
        ),
        (
            "Semantic content-unit features are based "
            "on automated similarity rather than manual "
            "clinical scoring."
        ),
        (
            "This is a research classifier, not a "
            "clinical diagnostic test."
        ),
    ],
}

metadata_path = (
    MODEL_ROOT
    / "model_metadata.json"
)

metadata_path.write_text(
    json.dumps(
        model_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# SAVE TEST METRICS
# ============================================================

test_metrics_path = (
    OUTPUT_ROOT
    / "test_metrics.csv"
)

pd.DataFrame([
    {
        "candidate_id":
            best_candidate_id,

        "family":
            best_family,

        **test_metrics,
    }
]).to_csv(
    test_metrics_path,
    index=False,
)


# ============================================================
# README
# ============================================================

readme_text = f"""
DELAWARE CINDERELLA RECALL-CONTENT MODEL
========================================

Target
------
MCI versus healthy Control.

Inputs
------
1. Cinderella-specific semantic content-unit features
2. Recall structure and lexical features
3. Character TF-IDF, when selected by validation performance

Content features
----------------
- Story details recalled
- Semantic similarity to {len(CONTENT_UNITS)} Cinderella content units
- Recall-unit counts at multiple similarity thresholds
- Recall efficiency
- Sequencing language
- Repetitions
- Self-corrections
- Uncertainty expressions
- Lexical diversity
- Speech rate

Best model
----------
{best_candidate_id}

Best family:
{best_family}

Validation-selected threshold:
{best_threshold:.3f}

Test result
-----------
Accuracy: {test_metrics['accuracy']:.4f}
Balanced accuracy: {test_metrics['balanced_accuracy']:.4f}
Precision: {test_metrics['precision']:.4f}
Sensitivity: {test_metrics['recall_sensitivity']:.4f}
Specificity: {test_metrics['specificity']:.4f}
F1: {test_metrics['f1']:.4f}
ROC-AUC: {test_metrics['roc_auc']:.4f}
Average precision: {test_metrics['average_precision']:.4f}
MCC: {test_metrics['mcc']:.4f}

Confusion matrix
----------------
TN: {test_metrics['tn']}
FP: {test_metrics['fp']}
FN: {test_metrics['fn']}
TP: {test_metrics['tp']}

Previous transcript baseline
----------------------------
Balanced accuracy: 0.6489
ROC-AUC: 0.6356

Important limitation
--------------------
The same test set was evaluated by earlier baselines. It should not
be used for additional hyperparameter decisions after this analysis.

This is a research classifier, not a clinical diagnostic test.
"""

readme_path = (
    OUTPUT_ROOT
    / "README.txt"
)

readme_path.write_text(
    readme_text.strip() + "\n",
    encoding="utf-8",
)


# ============================================================
# FILE MANIFEST
# ============================================================

file_manifest_rows = []

for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if not path.is_file():
        continue

    file_manifest_rows.append({
        "relative_path": str(
            path.relative_to(
                OUTPUT_ROOT
            )
        ),

        "size_bytes":
            path.stat().st_size,

        "sha256":
            sha256_file(path),
    })

file_manifest = pd.DataFrame(
    file_manifest_rows
)

file_manifest_path = (
    OUTPUT_ROOT
    / "file_manifest.csv"
)

file_manifest.to_csv(
    file_manifest_path,
    index=False,
)


# ============================================================
# ZIP BACKUP
# ============================================================

zip_base = (
    LOCAL_ROOT
    / "Delaware_Recall_Content_Model"
)

final_zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUTPUT_ROOT.parent,
    base_dir=OUTPUT_ROOT.name,
)

print()
print("Created ZIP:")
print(final_zip_path)


# ============================================================
# UPLOAD TO DRIVE
# ============================================================

output_drive_folder_id = (
    get_or_create_folder(
        DELAWARE_FOLDER_ID,
        "Delaware_Recall_Content_Model",
    )
)

upload_paths = [
    Path(final_zip_path),
    model_path,
    metadata_path,
    content_units_path,
    numeric_columns_path,
    feature_table_path,
    test_metrics_path,
    confidence_intervals_path,
    candidate_results_path,
    family_summary_path,
    validation_predictions_path,
    test_predictions_path,
    AUDIT_ROOT
    / "best_validation_threshold_search.csv",
    AUDIT_ROOT
    / "test_false_positives.csv",
    AUDIT_ROOT
    / "test_false_negatives.csv",
    readme_path,
    file_manifest_path,
]

print()
print("=" * 100)
print("UPLOADING RECALL-CONTENT MODEL")
print("=" * 100)

for local_path in upload_paths:
    result = upload_file(
        local_path,
        output_drive_folder_id,
    )

    print(
        "Uploaded:",
        result["name"],
    )


# ============================================================
# REMOVE TEMPORARY LOCAL FILES
# ============================================================

# Everything important is already uploaded.
# This removes downloaded/extracted temporary copies from Colab.

if INPUT_ROOT.exists():
    shutil.rmtree(INPUT_ROOT)

print()
print("Deleted temporary local input files.")


# ============================================================
# DONE
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print("Best candidate:")
print(best_candidate_id)

print()
print("Best family:")
print(best_family)

print()
print("Validation-selected threshold:")
print(best_threshold)

print()
print("Final test metrics:")
print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)

print()
print("Previous transcript baseline:")
print("Balanced accuracy: 0.6489")
print("ROC-AUC: 0.6356")

print()
print("Use this ZIP:")
print(
    "Delaware_Recall_Content_Model.zip"
)

In [ ]:
# ============================================================
# DELAWARE RECALL:
# REPEATED NESTED SUBJECT-LEVEL CROSS-VALIDATION
#
# Models compared:
# 1. Character TF-IDF
# 2. Recall-content features
# 3. Audio eGeMAPS
# 4. Character TF-IDF + recall-content features
#
# Outer evaluation:
#   5 folds x 3 repetitions = 15 outer test folds
#
# Inner model selection:
#   4-fold stratified CV
#
# No fixed test-set tuning is performed.
# Uses only the linked Google Drive folder ID.
# ============================================================

!pip -q install \
    google-api-python-client \
    scikit-learn \
    joblib

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import (
    MediaIoBaseDownload,
    MediaFileUpload,
)

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from pathlib import Path

import hashlib
import json
import shutil
import warnings
import zipfile

import joblib
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DELAWARE_FOLDER_ID = "1RV-tYijUIOBV4O7jb8RcvnWC3RdV8U5J"

RANDOM_SEED = 42

OUTER_FOLDS = 5
OUTER_REPEATS = 3
INNER_FOLDS = 4

BOOTSTRAP_RUNS = 2000

LOCAL_ROOT = Path(
    "/content/delaware_recall_nested_cv"
)

INPUT_ROOT = LOCAL_ROOT / "input"
EXTRACT_ROOT = INPUT_ROOT / "model_splits"

OUTPUT_ROOT = (
    LOCAL_ROOT
    / "Delaware_Recall_Repeated_Nested_CV"
)

AUDIT_ROOT = OUTPUT_ROOT / "audit"
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"
MODEL_ROOT = OUTPUT_ROOT / "models"

for path in [
    INPUT_ROOT,
    EXTRACT_ROOT,
    OUTPUT_ROOT,
    AUDIT_ROOT,
    PREDICTION_ROOT,
    MODEL_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

warnings.filterwarnings("ignore")

drive_service = build(
    "drive",
    "v3",
)


# ============================================================
# GOOGLE DRIVE UTILITIES
# ============================================================

def list_folder(folder_id):
    items = []
    page_token = None

    while True:
        response = (
            drive_service.files()
            .list(
                q=(
                    f"'{folder_id}' in parents "
                    "and trashed = false"
                ),
                fields=(
                    "nextPageToken,"
                    "files("
                    "id,name,mimeType,size,"
                    "modifiedTime,parents"
                    ")"
                ),
                pageToken=page_token,
                pageSize=1000,
            )
            .execute()
        )

        items.extend(
            response.get("files", [])
        )

        page_token = response.get(
            "nextPageToken"
        )

        if not page_token:
            break

    return items


def walk_drive_folder(
    folder_id,
    relative_path=Path(),
):
    records = []

    for item in list_folder(folder_id):
        current_path = (
            relative_path / item["name"]
        )

        if (
            item["mimeType"]
            == "application/vnd.google-apps.folder"
        ):
            records.extend(
                walk_drive_folder(
                    item["id"],
                    current_path,
                )
            )

        else:
            records.append({
                "file_id": item["id"],
                "name": item["name"],
                "relative_path": str(
                    current_path
                ),
                "mime_type": item["mimeType"],
                "size": int(
                    item.get("size", 0)
                ),
                "modified_time": item.get(
                    "modifiedTime"
                ),
            })

    return records


def download_drive_file(
    file_id,
    destination,
):
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    request = (
        drive_service.files()
        .get_media(fileId=file_id)
    )

    with destination.open("wb") as output_file:
        downloader = MediaIoBaseDownload(
            output_file,
            request,
            chunksize=16 * 1024 * 1024,
        )

        done = False

        while not done:
            _, done = downloader.next_chunk()

    return destination


def get_or_create_folder(
    parent_id,
    folder_name,
):
    safe_name = folder_name.replace(
        "'",
        "\\'",
    )

    response = (
        drive_service.files()
        .list(
            q=(
                f"'{parent_id}' in parents and "
                f"name = '{safe_name}' and "
                "mimeType = "
                "'application/vnd.google-apps.folder' "
                "and trashed = false"
            ),
            fields="files(id,name)",
        )
        .execute()
    )

    matches = response.get(
        "files",
        [],
    )

    if matches:
        return matches[0]["id"]

    created = (
        drive_service.files()
        .create(
            body={
                "name": folder_name,
                "mimeType": (
                    "application/"
                    "vnd.google-apps.folder"
                ),
                "parents": [parent_id],
            },
            fields="id",
        )
        .execute()
    )

    return created["id"]


def upload_file(
    local_path,
    parent_id,
):
    local_path = Path(local_path)

    result = (
        drive_service.files()
        .create(
            body={
                "name": local_path.name,
                "parents": [parent_id],
            },
            media_body=MediaFileUpload(
                str(local_path),
                resumable=True,
            ),
            fields="id,name",
        )
        .execute()
    )

    return result


# ============================================================
# FILE UTILITIES
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def find_latest_file(
    records,
    folder_name,
    filename,
):
    expected_suffix = (
        f"{folder_name}/{filename}"
    ).lower()

    matches = [
        record
        for record in records
        if (
            record["relative_path"]
            .replace("\\", "/")
            .lower()
            .endswith(expected_suffix)
        )
    ]

    if not matches:
        raise FileNotFoundError(
            f"Could not find {expected_suffix}"
        )

    matches = sorted(
        matches,
        key=lambda record: (
            record.get("modified_time") or ""
        ),
        reverse=True,
    )

    return matches[0]


def read_text(path):
    return Path(path).read_text(
        encoding="utf-8",
        errors="replace",
    ).strip()


# ============================================================
# INDEX LINKED DRIVE FOLDER
# ============================================================

print("=" * 100)
print("INDEXING LINKED DRIVE FOLDER")
print("=" * 100)

drive_records = walk_drive_folder(
    DELAWARE_FOLDER_ID
)

print(
    "Files indexed:",
    len(drive_records),
)


# ============================================================
# LOCATE REQUIRED SAVED FILES
# ============================================================

required_drive_files = {
    "model_splits_zip": (
        "Delaware_Recall_Model_Splits",
        "Delaware_Recall_Model_Splits.zip",
    ),
    "content_features": (
        "Delaware_Recall_Content_Model",
        "recall_content_features.csv",
    ),
    "audio_features": (
        "Delaware_Recall_Audio_Baseline",
        "egemaps_features_all.csv",
    ),
}

located = {}

for key, (
    folder_name,
    filename,
) in required_drive_files.items():

    record = find_latest_file(
        drive_records,
        folder_name,
        filename,
    )

    destination = (
        INPUT_ROOT
        / key
        / filename
    )

    download_drive_file(
        record["file_id"],
        destination,
    )

    located[key] = destination

    print(
        "Downloaded:",
        record["relative_path"],
    )


# ============================================================
# EXTRACT FIXED DATASET ZIP
# ============================================================

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(
    located["model_splits_zip"],
    "r",
) as archive:
    archive.extractall(
        EXTRACT_ROOT
    )

manifest_candidates = list(
    EXTRACT_ROOT.rglob(
        "recall_model_split_manifest.csv"
    )
)

if not manifest_candidates:
    raise FileNotFoundError(
        "Missing recall_model_split_manifest.csv"
    )

SOURCE_ROOT = (
    manifest_candidates[0].parent
)

manifest = pd.read_csv(
    manifest_candidates[0]
)

manifest["subject_id"] = (
    manifest["subject_id"]
    .astype(str)
)

manifest["recording_id"] = (
    manifest["recording_id"]
    .astype(str)
)


# ============================================================
# LOAD TRANSCRIPTS
# ============================================================

print()
print("=" * 100)
print("LOADING TRANSCRIPTS")
print("=" * 100)

transcript_rows = []

for index, row in enumerate(
    manifest.itertuples(index=False),
    start=1,
):
    transcript_path = (
        SOURCE_ROOT
        / str(row.transcript_path)
    )

    if not transcript_path.exists():
        raise FileNotFoundError(
            str(transcript_path)
        )

    text = read_text(
        transcript_path
    )

    if not text:
        raise ValueError(
            f"Empty transcript: {row.recording_id}"
        )

    transcript_rows.append({
        "subject_id": row.subject_id,
        "recording_id": row.recording_id,
        "diagnosis": row.diagnosis,
        "binary_label": int(
            row.binary_label
        ),
        "text": text,
    })

    if (
        index % 50 == 0
        or index == len(manifest)
    ):
        print(
            f"Loaded {index}/{len(manifest)}"
        )

transcripts = pd.DataFrame(
    transcript_rows
)


# ============================================================
# LOAD SAVED CONTENT AND AUDIO FEATURES
# ============================================================

content_features = pd.read_csv(
    located["content_features"]
)

audio_features = pd.read_csv(
    located["audio_features"]
)

for dataframe in [
    content_features,
    audio_features,
]:
    dataframe["subject_id"] = (
        dataframe["subject_id"]
        .astype(str)
    )

    dataframe["recording_id"] = (
        dataframe["recording_id"]
        .astype(str)
    )


# ============================================================
# DEFINE FEATURE COLUMNS
# ============================================================

merge_keys = [
    "subject_id",
    "recording_id",
    "diagnosis",
    "binary_label",
]

content_metadata_columns = {
    "split",
    "diagnosis",
    "binary_label",
    "subject_id",
    "recording_id",
    "transcript_path",
    "text",
}

content_numeric_columns_original = [
    column
    for column in content_features.columns
    if column not in content_metadata_columns
]

audio_metadata_columns = {
    "split",
    "diagnosis",
    "binary_label",
    "subject_id",
    "recording_id",
    "audio_path",
    "duration_seconds_manifest",
    "duration_seconds_file",
    "sample_rate_hz",
    "channels",
}

audio_numeric_columns_original = [
    column
    for column in audio_features.columns
    if column not in audio_metadata_columns
]


# ============================================================
# RENAME NUMERIC COLUMNS TO PREVENT COLLISIONS
# ============================================================

content_rename_map = {
    column: f"content__{column}"
    for column
    in content_numeric_columns_original
}

audio_rename_map = {
    column: f"audio__{column}"
    for column
    in audio_numeric_columns_original
}

content_selected = content_features[
    merge_keys
    + content_numeric_columns_original
].rename(
    columns=content_rename_map
)

audio_selected = audio_features[
    merge_keys
    + audio_numeric_columns_original
].rename(
    columns=audio_rename_map
)

content_numeric_columns = list(
    content_rename_map.values()
)

audio_numeric_columns = list(
    audio_rename_map.values()
)


# ============================================================
# MERGE ALL MODALITIES
# ============================================================

data = transcripts.merge(
    content_selected,
    on=merge_keys,
    how="inner",
    validate="one_to_one",
)

data = data.merge(
    audio_selected,
    on=merge_keys,
    how="inner",
    validate="one_to_one",
)

if len(data) != 283:
    raise RuntimeError(
        f"Expected 283 merged subjects, "
        f"found {len(data)}."
    )

if data["subject_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate subject IDs found after merging."
    )

if (
    data.groupby("subject_id")[
        "diagnosis"
    ].nunique()
    .max()
    > 1
):
    raise RuntimeError(
        "A subject has multiple diagnosis labels."
    )

data = data.sort_values(
    "subject_id"
).reset_index(drop=True)

print()
print("=" * 100)
print("MERGED DATASET")
print("=" * 100)

print("Subjects:", len(data))
print(
    "Content features:",
    len(content_numeric_columns),
)
print(
    "Audio features:",
    len(audio_numeric_columns),
)

print()
print(
    data["diagnosis"]
    .value_counts()
)


# ============================================================
# METRIC FUNCTIONS
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "threshold": float(threshold),

        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),

        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),

        "recall_sensitivity": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),

        "specificity": float(
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        ),

        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),

        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),

        "average_precision": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),

        "mcc": float(
            matthews_corrcoef(
                y_true,
                predictions,
            )
        ),

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def choose_threshold(
    y_true,
    probabilities,
):
    threshold_rows = []

    for threshold in np.linspace(
        0.10,
        0.90,
        161,
    ):
        metrics = calculate_metrics(
            y_true,
            probabilities,
            threshold,
        )

        threshold_rows.append(
            metrics
        )

    table = pd.DataFrame(
        threshold_rows
    )

    table[
        "distance_from_half"
    ] = (
        table["threshold"]
        - 0.5
    ).abs()

    table = (
        table
        .sort_values(
            [
                "balanced_accuracy",
                "f1",
                "mcc",
                "distance_from_half",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    return (
        float(
            table.loc[
                0,
                "threshold",
            ]
        ),
        table,
    )


# ============================================================
# MODEL DEFINITIONS
# ============================================================

def build_character_pipeline():
    return Pipeline([
        (
            "features",
            ColumnTransformer(
                transformers=[
                    (
                        "character",
                        TfidfVectorizer(
                            analyzer="char_wb",
                            lowercase=True,
                            min_df=2,
                            sublinear_tf=True,
                        ),
                        "text",
                    ),
                ],
                remainder="drop",
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                solver="liblinear",
                max_iter=10000,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


def build_numeric_pipeline(
    numeric_columns,
):
    return Pipeline([
        (
            "features",
            ColumnTransformer(
                transformers=[
                    (
                        "numeric",
                        Pipeline([
                            (
                                "imputer",
                                SimpleImputer(
                                    strategy="median"
                                ),
                            ),
                            (
                                "scaler",
                                StandardScaler(),
                            ),
                        ]),
                        numeric_columns,
                    ),
                ],
                remainder="drop",
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                solver="liblinear",
                max_iter=10000,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


def build_combined_pipeline():
    return Pipeline([
        (
            "features",
            ColumnTransformer(
                transformers=[
                    (
                        "character",
                        TfidfVectorizer(
                            analyzer="char_wb",
                            lowercase=True,
                            min_df=2,
                            sublinear_tf=True,
                        ),
                        "text",
                    ),
                    (
                        "content",
                        Pipeline([
                            (
                                "imputer",
                                SimpleImputer(
                                    strategy="median"
                                ),
                            ),
                            (
                                "scaler",
                                StandardScaler(),
                            ),
                        ]),
                        content_numeric_columns,
                    ),
                ],
                remainder="drop",
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                solver="liblinear",
                max_iter=10000,
                random_state=RANDOM_SEED,
            ),
        ),
    ])


MODEL_SPECS = {
    "character_tfidf": {
        "pipeline": build_character_pipeline(),

        "parameter_grid": {
            "features__character__ngram_range": [
                (3, 5),
                (2, 5),
            ],
            "features__character__max_features": [
                20000,
                40000,
            ],
            "classifier__C": [
                0.1,
                0.25,
                0.5,
                1.0,
                2.0,
            ],
        },
    },

    "recall_content": {
        "pipeline": build_numeric_pipeline(
            content_numeric_columns
        ),

        "parameter_grid": [
            {
                "classifier": [
                    LogisticRegression(
                        class_weight="balanced",
                        solver="liblinear",
                        max_iter=10000,
                        random_state=RANDOM_SEED,
                    )
                ],
                "classifier__C": [
                    0.01,
                    0.05,
                    0.1,
                    0.25,
                    0.5,
                    1.0,
                ],
            },
            {
                "classifier": [
                    RandomForestClassifier(
                        n_estimators=350,
                        class_weight="balanced",
                        max_features="sqrt",
                        random_state=RANDOM_SEED,
                        n_jobs=-1,
                    )
                ],
                "classifier__max_depth": [
                    None,
                    6,
                ],
                "classifier__min_samples_leaf": [
                    4,
                    8,
                ],
            },
        ],
    },

    "audio_egemaps": {
        "pipeline": build_numeric_pipeline(
            audio_numeric_columns
        ),

        "parameter_grid": [
            {
                "classifier": [
                    LogisticRegression(
                        class_weight="balanced",
                        solver="liblinear",
                        max_iter=10000,
                        random_state=RANDOM_SEED,
                    )
                ],
                "classifier__C": [
                    0.01,
                    0.05,
                    0.1,
                    0.25,
                    0.5,
                    1.0,
                ],
            },
            {
                "classifier": [
                    SVC(
                        kernel="rbf",
                        gamma="scale",
                        class_weight="balanced",
                        probability=True,
                        random_state=RANDOM_SEED,
                    )
                ],
                "classifier__C": [
                    0.1,
                    0.5,
                    1.0,
                    2.0,
                ],
            },
        ],
    },

    "character_plus_content": {
        "pipeline": build_combined_pipeline(),

        "parameter_grid": {
            "features__character__ngram_range": [
                (3, 5),
                (2, 5),
            ],
            "features__character__max_features": [
                20000,
                40000,
            ],
            "features__transformer_weights": [
                {
                    "character": 1.0,
                    "content": 0.10,
                },
                {
                    "character": 1.0,
                    "content": 0.25,
                },
                {
                    "character": 1.0,
                    "content": 0.50,
                },
            ],
            "classifier__C": [
                0.1,
                0.25,
                0.5,
                1.0,
            ],
        },
    },
}


# ============================================================
# OUTER AND INNER CROSS-VALIDATION
# ============================================================

X = data.copy()

y = (
    data["binary_label"]
    .to_numpy(dtype=int)
)

outer_cv = RepeatedStratifiedKFold(
    n_splits=OUTER_FOLDS,
    n_repeats=OUTER_REPEATS,
    random_state=RANDOM_SEED,
)

outer_splits = list(
    outer_cv.split(
        X,
        y,
    )
)

expected_outer_folds = (
    OUTER_FOLDS
    * OUTER_REPEATS
)

if len(outer_splits) != expected_outer_folds:
    raise RuntimeError(
        "Unexpected number of outer folds."
    )


# ============================================================
# RUN REPEATED NESTED CROSS-VALIDATION
# ============================================================

print()
print("=" * 100)
print("RUNNING REPEATED NESTED CROSS-VALIDATION")
print("=" * 100)

print(
    f"Outer folds: {OUTER_FOLDS}"
)

print(
    f"Outer repetitions: {OUTER_REPEATS}"
)

print(
    f"Total outer evaluations per model: "
    f"{expected_outer_folds}"
)

print(
    f"Inner folds: {INNER_FOLDS}"
)

outer_metric_rows = []
outer_prediction_rows = []
inner_selection_rows = []
threshold_search_rows = []

best_estimators = {}

for outer_index, (
    train_indices,
    test_indices,
) in enumerate(
    outer_splits,
    start=1,
):
    repeat_number = (
        (outer_index - 1)
        // OUTER_FOLDS
        + 1
    )

    fold_number = (
        (outer_index - 1)
        % OUTER_FOLDS
        + 1
    )

    X_outer_train = X.iloc[
        train_indices
    ].copy()

    y_outer_train = y[
        train_indices
    ]

    X_outer_test = X.iloc[
        test_indices
    ].copy()

    y_outer_test = y[
        test_indices
    ]

    train_subjects = set(
        X_outer_train["subject_id"]
    )

    test_subjects = set(
        X_outer_test["subject_id"]
    )

    overlap = (
        train_subjects
        & test_subjects
    )

    if overlap:
        raise RuntimeError(
            f"Subject leakage in outer fold: "
            f"{sorted(overlap)}"
        )

    print()
    print("-" * 100)
    print(
        f"REPEAT {repeat_number}/"
        f"{OUTER_REPEATS} — "
        f"FOLD {fold_number}/"
        f"{OUTER_FOLDS}"
    )
    print("-" * 100)

    for model_number, (
        model_name,
        model_spec,
    ) in enumerate(
        MODEL_SPECS.items(),
        start=1,
    ):
        print(
            f"[{model_number}/"
            f"{len(MODEL_SPECS)}] "
            f"{model_name}"
        )

        inner_seed = (
            RANDOM_SEED
            + outer_index * 100
            + model_number
        )

        inner_cv = StratifiedKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=inner_seed,
        )

        grid_search = GridSearchCV(
            estimator=clone(
                model_spec["pipeline"]
            ),
            param_grid=(
                model_spec[
                    "parameter_grid"
                ]
            ),
            scoring="balanced_accuracy",
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
            return_train_score=False,
            error_score="raise",
        )

        grid_search.fit(
            X_outer_train,
            y_outer_train,
        )

        best_estimator = (
            grid_search.best_estimator_
        )

        # Produce inner out-of-fold probabilities using
        # the selected hyperparameters. These probabilities
        # are used only to choose the classification threshold.
        inner_probabilities = (
            cross_val_predict(
                clone(best_estimator),
                X_outer_train,
                y_outer_train,
                cv=inner_cv,
                method="predict_proba",
                n_jobs=-1,
            )[:, 1]
        )

        (
            selected_threshold,
            threshold_table,
        ) = choose_threshold(
            y_outer_train,
            inner_probabilities,
        )

        threshold_table[
            "model"
        ] = model_name

        threshold_table[
            "repeat"
        ] = repeat_number

        threshold_table[
            "fold"
        ] = fold_number

        threshold_search_rows.append(
            threshold_table
        )

        best_estimator.fit(
            X_outer_train,
            y_outer_train,
        )

        outer_probabilities = (
            best_estimator.predict_proba(
                X_outer_test
            )[:, 1]
        )

        outer_predictions = (
            outer_probabilities
            >= selected_threshold
        ).astype(int)

        metrics = calculate_metrics(
            y_outer_test,
            outer_probabilities,
            selected_threshold,
        )

        outer_metric_rows.append({
            "model": model_name,
            "repeat": repeat_number,
            "fold": fold_number,
            "outer_train_subjects": int(
                len(X_outer_train)
            ),
            "outer_test_subjects": int(
                len(X_outer_test)
            ),
            "inner_best_balanced_accuracy": float(
                grid_search.best_score_
            ),
            "selected_threshold": (
                selected_threshold
            ),
            "best_parameters": json.dumps(
                grid_search.best_params_,
                default=str,
                sort_keys=True,
            ),
            **metrics,
        })

        for local_index, (
            dataframe_index,
            probability,
            prediction,
        ) in enumerate(
            zip(
                test_indices,
                outer_probabilities,
                outer_predictions,
            )
        ):
            subject_row = data.iloc[
                dataframe_index
            ]

            outer_prediction_rows.append({
                "model": model_name,
                "repeat": repeat_number,
                "fold": fold_number,
                "subject_id": (
                    subject_row[
                        "subject_id"
                    ]
                ),
                "recording_id": (
                    subject_row[
                        "recording_id"
                    ]
                ),
                "diagnosis": (
                    subject_row[
                        "diagnosis"
                    ]
                ),
                "binary_label": int(
                    subject_row[
                        "binary_label"
                    ]
                ),
                "predicted_probability_MCI": float(
                    probability
                ),
                "selected_threshold": float(
                    selected_threshold
                ),
                "predicted_label": int(
                    prediction
                ),
                "correct": bool(
                    prediction
                    == subject_row[
                        "binary_label"
                    ]
                ),
            })

        inner_results = pd.DataFrame(
            grid_search.cv_results_
        )

        best_inner_row = (
            inner_results.iloc[
                grid_search.best_index_
            ]
        )

        inner_selection_rows.append({
            "model": model_name,
            "repeat": repeat_number,
            "fold": fold_number,
            "best_inner_balanced_accuracy": float(
                grid_search.best_score_
            ),
            "best_parameters": json.dumps(
                grid_search.best_params_,
                default=str,
                sort_keys=True,
            ),
            "inner_rank": int(
                best_inner_row[
                    "rank_test_score"
                ]
            ),
            "inner_mean_fit_time_seconds": float(
                best_inner_row[
                    "mean_fit_time"
                ]
            ),
        })

        best_estimators[
            (
                model_name,
                repeat_number,
                fold_number,
            )
        ] = best_estimator

        print(
            "  Outer balanced accuracy:",
            round(
                metrics[
                    "balanced_accuracy"
                ],
                4,
            ),
            "| ROC-AUC:",
            round(
                metrics["roc_auc"],
                4,
            ),
            "| threshold:",
            round(
                selected_threshold,
                3,
            ),
        )


# ============================================================
# SAVE RAW NESTED-CV OUTPUTS
# ============================================================

outer_metrics = pd.DataFrame(
    outer_metric_rows
)

outer_predictions = pd.DataFrame(
    outer_prediction_rows
)

inner_selections = pd.DataFrame(
    inner_selection_rows
)

threshold_search = pd.concat(
    threshold_search_rows,
    ignore_index=True,
)

outer_metrics_path = (
    OUTPUT_ROOT
    / "outer_fold_metrics.csv"
)

outer_predictions_path = (
    PREDICTION_ROOT
    / "repeated_outer_fold_predictions.csv"
)

inner_selections_path = (
    AUDIT_ROOT
    / "inner_model_selections.csv"
)

threshold_search_path = (
    AUDIT_ROOT
    / "inner_threshold_searches.csv"
)

outer_metrics.to_csv(
    outer_metrics_path,
    index=False,
)

outer_predictions.to_csv(
    outer_predictions_path,
    index=False,
)

inner_selections.to_csv(
    inner_selections_path,
    index=False,
)

threshold_search.to_csv(
    threshold_search_path,
    index=False,
)


# ============================================================
# VERIFY OUTER PREDICTION COVERAGE
# ============================================================

coverage = (
    outer_predictions
    .groupby(
        [
            "model",
            "subject_id",
        ]
    )
    .size()
    .reset_index(
        name="outer_test_appearances"
    )
)

expected_appearances = (
    OUTER_REPEATS
)

invalid_coverage = coverage[
    coverage[
        "outer_test_appearances"
    ]
    != expected_appearances
]

if len(invalid_coverage) > 0:
    invalid_coverage.to_csv(
        AUDIT_ROOT
        / "invalid_prediction_coverage.csv",
        index=False,
    )

    raise RuntimeError(
        "Not every subject received exactly one "
        "outer prediction per repetition."
    )

coverage_path = (
    AUDIT_ROOT
    / "outer_prediction_coverage.csv"
)

coverage.to_csv(
    coverage_path,
    index=False,
)


# ============================================================
# METRICS BY REPETITION
# ============================================================

print()
print("=" * 100)
print("RESULTS BY REPETITION")
print("=" * 100)

repeat_metric_rows = []

for (
    model_name,
    repeat_number,
), group in outer_predictions.groupby(
    [
        "model",
        "repeat",
    ]
):
    metrics = calculate_metrics(
        group[
            "binary_label"
        ].to_numpy(dtype=int),

        group[
            "predicted_probability_MCI"
        ].to_numpy(),

        # Predictions used fold-specific thresholds,
        # so threshold-dependent metrics are calculated
        # separately below from saved labels.
        0.5,
    )

    true_labels = (
        group["binary_label"]
        .to_numpy(dtype=int)
    )

    predictions = (
        group["predicted_label"]
        .to_numpy(dtype=int)
    )

    probabilities = (
        group[
            "predicted_probability_MCI"
        ].to_numpy()
    )

    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        predictions,
        labels=[0, 1],
    ).ravel()

    repeat_metric_rows.append({
        "model": model_name,
        "repeat": int(
            repeat_number
        ),

        "accuracy": float(
            accuracy_score(
                true_labels,
                predictions,
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                true_labels,
                predictions,
            )
        ),

        "precision": float(
            precision_score(
                true_labels,
                predictions,
                zero_division=0,
            )
        ),

        "recall_sensitivity": float(
            recall_score(
                true_labels,
                predictions,
                zero_division=0,
            )
        ),

        "specificity": float(
            tn / (tn + fp)
        ),

        "f1": float(
            f1_score(
                true_labels,
                predictions,
                zero_division=0,
            )
        ),

        "roc_auc": float(
            roc_auc_score(
                true_labels,
                probabilities,
            )
        ),

        "average_precision": float(
            average_precision_score(
                true_labels,
                probabilities,
            )
        ),

        "mcc": float(
            matthews_corrcoef(
                true_labels,
                predictions,
            )
        ),

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    })


repeat_metrics = pd.DataFrame(
    repeat_metric_rows
)

repeat_metrics_path = (
    OUTPUT_ROOT
    / "repeat_level_metrics.csv"
)

repeat_metrics.to_csv(
    repeat_metrics_path,
    index=False,
)

display(
    repeat_metrics[
        [
            "model",
            "repeat",
            "balanced_accuracy",
            "roc_auc",
            "f1",
            "mcc",
        ]
    ].sort_values(
        [
            "model",
            "repeat",
        ]
    )
)


# ============================================================
# MODEL SUMMARY ACROSS REPETITIONS
# ============================================================

summary_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1",
    "roc_auc",
    "average_precision",
    "mcc",
]

summary_rows = []

for model_name, group in (
    repeat_metrics.groupby("model")
):
    row = {
        "model": model_name,
        "repetitions": int(
            len(group)
        ),
    }

    for metric_name in summary_metrics:
        values = group[
            metric_name
        ].to_numpy()

        row[
            f"{metric_name}_mean"
        ] = float(
            np.mean(values)
        )

        row[
            f"{metric_name}_std"
        ] = float(
            np.std(
                values,
                ddof=1,
            )
        )

        row[
            f"{metric_name}_min"
        ] = float(
            np.min(values)
        )

        row[
            f"{metric_name}_max"
        ] = float(
            np.max(values)
        )

    summary_rows.append(row)


model_summary = pd.DataFrame(
    summary_rows
)

model_summary = (
    model_summary
    .sort_values(
        [
            "balanced_accuracy_mean",
            "roc_auc_mean",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

model_summary_path = (
    OUTPUT_ROOT
    / "nested_cv_model_summary.csv"
)

model_summary.to_csv(
    model_summary_path,
    index=False,
)

print()
print("=" * 100)
print("NESTED-CV MODEL SUMMARY")
print("=" * 100)

display(
    model_summary[
        [
            "model",
            "balanced_accuracy_mean",
            "balanced_accuracy_std",
            "roc_auc_mean",
            "roc_auc_std",
            "f1_mean",
            "f1_std",
            "mcc_mean",
            "mcc_std",
        ]
    ]
)


# ============================================================
# SUBJECT-CLUSTERED BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

print()
print("=" * 100)
print("SUBJECT-CLUSTERED BOOTSTRAP")
print("=" * 100)

rng = np.random.default_rng(
    RANDOM_SEED
)

bootstrap_rows = []

for model_name, model_predictions in (
    outer_predictions.groupby("model")
):
    unique_subjects = (
        model_predictions[
            "subject_id"
        ]
        .drop_duplicates()
        .to_numpy()
    )

    for bootstrap_index in range(
        BOOTSTRAP_RUNS
    ):
        sampled_subjects = rng.choice(
            unique_subjects,
            size=len(unique_subjects),
            replace=True,
        )

        sampled_groups = []

        for sampled_position, subject_id in enumerate(
            sampled_subjects
        ):
            subject_rows = (
                model_predictions[
                    model_predictions[
                        "subject_id"
                    ]
                    == subject_id
                ]
                .copy()
            )

            # Assign a bootstrap-cluster identifier so
            # repeated selections of the same subject
            # remain separate bootstrap observations.
            subject_rows[
                "bootstrap_cluster"
            ] = sampled_position

            sampled_groups.append(
                subject_rows
            )

        sampled = pd.concat(
            sampled_groups,
            ignore_index=True,
        )

        true_labels = (
            sampled["binary_label"]
            .to_numpy(dtype=int)
        )

        probabilities = (
            sampled[
                "predicted_probability_MCI"
            ]
            .to_numpy()
        )

        predictions = (
            sampled[
                "predicted_label"
            ]
            .to_numpy(dtype=int)
        )

        if len(
            np.unique(true_labels)
        ) < 2:
            continue

        tn, fp, fn, tp = confusion_matrix(
            true_labels,
            predictions,
            labels=[0, 1],
        ).ravel()

        bootstrap_rows.append({
            "model": model_name,

            "accuracy": float(
                accuracy_score(
                    true_labels,
                    predictions,
                )
            ),

            "balanced_accuracy": float(
                balanced_accuracy_score(
                    true_labels,
                    predictions,
                )
            ),

            "precision": float(
                precision_score(
                    true_labels,
                    predictions,
                    zero_division=0,
                )
            ),

            "recall_sensitivity": float(
                recall_score(
                    true_labels,
                    predictions,
                    zero_division=0,
                )
            ),

            "specificity": float(
                tn / (tn + fp)
            ),

            "f1": float(
                f1_score(
                    true_labels,
                    predictions,
                    zero_division=0,
                )
            ),

            "roc_auc": float(
                roc_auc_score(
                    true_labels,
                    probabilities,
                )
            ),

            "average_precision": float(
                average_precision_score(
                    true_labels,
                    probabilities,
                )
            ),

            "mcc": float(
                matthews_corrcoef(
                    true_labels,
                    predictions,
                )
            ),
        })


bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

bootstrap_results_path = (
    AUDIT_ROOT
    / "subject_clustered_bootstrap_raw.csv"
)

bootstrap_results.to_csv(
    bootstrap_results_path,
    index=False,
)

confidence_rows = []

for model_name, group in (
    bootstrap_results.groupby("model")
):
    for metric_name in summary_metrics:
        values = (
            group[metric_name]
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna()
            .to_numpy()
        )

        confidence_rows.append({
            "model": model_name,
            "metric": metric_name,
            "bootstrap_mean": float(
                np.mean(values)
            ),
            "ci_95_lower": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),
            "ci_95_upper": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),
            "valid_bootstrap_runs": int(
                len(values)
            ),
        })


confidence_intervals = pd.DataFrame(
    confidence_rows
)

confidence_intervals_path = (
    OUTPUT_ROOT
    / "nested_cv_bootstrap_confidence_intervals.csv"
)

confidence_intervals.to_csv(
    confidence_intervals_path,
    index=False,
)

display(
    confidence_intervals[
        confidence_intervals[
            "metric"
        ].isin(
            [
                "balanced_accuracy",
                "roc_auc",
                "f1",
                "mcc",
            ]
        )
    ].sort_values(
        [
            "metric",
            "bootstrap_mean",
        ],
        ascending=[
            True,
            False,
        ]
    )
)


# ============================================================
# PAIRED MODEL COMPARISONS
# ============================================================

# Compare models using subject-level mean predictions across
# repetitions. This is descriptive rather than a clinical claim.

subject_aggregated = (
    outer_predictions
    .groupby(
        [
            "model",
            "subject_id",
            "recording_id",
            "diagnosis",
            "binary_label",
        ],
        as_index=False,
    )
    .agg(
        mean_probability_MCI=(
            "predicted_probability_MCI",
            "mean",
        ),
        majority_MCI_fraction=(
            "predicted_label",
            "mean",
        ),
    )
)

subject_aggregated[
    "majority_predicted_label"
] = (
    subject_aggregated[
        "majority_MCI_fraction"
    ]
    >= 0.5
).astype(int)

subject_aggregated_path = (
    PREDICTION_ROOT
    / "subject_aggregated_predictions.csv"
)

subject_aggregated.to_csv(
    subject_aggregated_path,
    index=False,
)


# ============================================================
# SELECT BEST MODEL FROM NESTED CV
# ============================================================

best_model_name = str(
    model_summary.loc[
        0,
        "model",
    ]
)

print()
print("Best nested-CV model:")
print(best_model_name)


# ============================================================
# FIT FINAL RESEARCH MODEL USING ALL 283 SUBJECTS
# ============================================================

# The final model is fit only after the nested-CV evaluation.
# It is intended for future external evaluation, not for
# estimating performance on this same Delaware dataset.

best_parameter_rows = (
    inner_selections[
        inner_selections[
            "model"
        ]
        == best_model_name
    ][
        "best_parameters"
    ]
    .value_counts()
)

most_common_parameter_json = str(
    best_parameter_rows.index[0]
)

most_common_parameters = json.loads(
    most_common_parameter_json
)

final_model = clone(
    MODEL_SPECS[
        best_model_name
    ]["pipeline"]
)

final_model.set_params(
    **most_common_parameters
)

# Create out-of-fold probabilities across the entire dataset
# to select a final operational threshold without using
# in-sample fitted probabilities.
final_threshold_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED + 999,
)

final_oof_probabilities = (
    cross_val_predict(
        clone(final_model),
        X,
        y,
        cv=final_threshold_cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]
)

(
    final_threshold,
    final_threshold_table,
) = choose_threshold(
    y,
    final_oof_probabilities,
)

final_threshold_table_path = (
    AUDIT_ROOT
    / "final_model_oof_threshold_search.csv"
)

final_threshold_table.to_csv(
    final_threshold_table_path,
    index=False,
)

final_model.fit(
    X,
    y,
)

final_model_path = (
    MODEL_ROOT
    / "final_nested_cv_selected_model.joblib"
)

joblib.dump(
    final_model,
    final_model_path,
)


# ============================================================
# SAVE FINAL MODEL CONFIGURATION
# ============================================================

final_configuration = {
    "dataset": "Delaware",
    "task": "Cinderella recall",
    "classification_target": (
        "MCI versus Control"
    ),

    "subjects": int(
        len(data)
    ),

    "class_counts": {
        str(key): int(value)
        for key, value in (
            data["diagnosis"]
            .value_counts()
            .to_dict()
            .items()
        )
    },

    "evaluation_design": {
        "outer_folds": OUTER_FOLDS,
        "outer_repeats": OUTER_REPEATS,
        "total_outer_evaluations_per_model": (
            expected_outer_folds
        ),
        "inner_folds": INNER_FOLDS,
        "selection_metric": (
            "balanced_accuracy"
        ),
        "threshold_selection": (
            "inner out-of-fold probabilities"
        ),
    },

    "best_model": (
        best_model_name
    ),

    "most_common_nested_cv_parameters": (
        most_common_parameters
    ),

    "final_oof_selected_threshold": (
        final_threshold
    ),

    "nested_cv_summary": (
        model_summary[
            model_summary["model"]
            == best_model_name
        ]
        .iloc[0]
        .to_dict()
    ),

    "important_interpretation": (
        "Nested cross-validation estimates "
        "within-corpus generalization. "
        "External-corpus evaluation is still required."
    ),

    "clinical_status": (
        "Research model only; not a diagnostic test."
    ),
}

configuration_path = (
    MODEL_ROOT
    / "final_model_configuration.json"
)

configuration_path.write_text(
    json.dumps(
        final_configuration,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)


# ============================================================
# DATASET AUDIT
# ============================================================

dataset_audit = {
    "rows": int(len(data)),
    "unique_subjects": int(
        data["subject_id"].nunique()
    ),
    "duplicate_subject_rows": int(
        data["subject_id"]
        .duplicated()
        .sum()
    ),
    "control_subjects": int(
        (
            data["diagnosis"]
            == "Control"
        ).sum()
    ),
    "mci_subjects": int(
        (
            data["diagnosis"]
            == "MCI"
        ).sum()
    ),
    "content_feature_count": int(
        len(content_numeric_columns)
    ),
    "audio_feature_count": int(
        len(audio_numeric_columns)
    ),
    "outer_prediction_rows": int(
        len(outer_predictions)
    ),
    "expected_outer_prediction_rows": int(
        len(data)
        * OUTER_REPEATS
        * len(MODEL_SPECS)
    ),
}

dataset_audit_path = (
    AUDIT_ROOT
    / "dataset_and_prediction_audit.json"
)

dataset_audit_path.write_text(
    json.dumps(
        dataset_audit,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# CREATE README
# ============================================================

best_summary = (
    model_summary[
        model_summary["model"]
        == best_model_name
    ]
    .iloc[0]
)

readme_text = f"""
DELAWARE RECALL REPEATED NESTED CROSS-VALIDATION
================================================

Task
----
Cinderella recall classification:
MCI versus healthy Control.

Subjects
--------
Total: {len(data)}
Control: {(data['diagnosis'] == 'Control').sum()}
MCI: {(data['diagnosis'] == 'MCI').sum()}

Evaluation design
-----------------
Outer evaluation:
{OUTER_FOLDS}-fold stratified cross-validation repeated
{OUTER_REPEATS} times.

Total outer evaluations per model:
{expected_outer_folds}

Inner model selection:
{INNER_FOLDS}-fold stratified cross-validation.

Hyperparameters and classification thresholds were selected
inside each outer-training set. Outer-test subjects were not
used for either model or threshold selection.

Models compared
---------------
1. Character TF-IDF
2. Recall-content features
3. Audio openSMILE eGeMAPS features
4. Character TF-IDF plus recall-content features

Best model
----------
{best_model_name}

Mean repeated nested-CV results
-------------------------------
Balanced accuracy:
{best_summary['balanced_accuracy_mean']:.4f}
± {best_summary['balanced_accuracy_std']:.4f}

ROC-AUC:
{best_summary['roc_auc_mean']:.4f}
± {best_summary['roc_auc_std']:.4f}

F1:
{best_summary['f1_mean']:.4f}
± {best_summary['f1_std']:.4f}

MCC:
{best_summary['mcc_mean']:.4f}
± {best_summary['mcc_std']:.4f}

Final fitted research model
---------------------------
The most frequently selected nested-CV hyperparameters were
used to fit a final model on all 283 subjects.

Final operational threshold:
{final_threshold:.3f}

This fitted model must be tested on an independent external
dataset before making any generalization or clinical claims.

Important limitation
--------------------
Repeated nested cross-validation estimates within-Delaware
generalization. It does not demonstrate clinical validity or
cross-corpus transportability.

This is a research classifier, not a diagnostic test.
"""

readme_path = (
    OUTPUT_ROOT
    / "README.txt"
)

readme_path.write_text(
    readme_text.strip() + "\n",
    encoding="utf-8",
)


# ============================================================
# CREATE FILE MANIFEST
# ============================================================

file_manifest_rows = []

for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if not path.is_file():
        continue

    file_manifest_rows.append({
        "relative_path": str(
            path.relative_to(
                OUTPUT_ROOT
            )
        ),
        "size_bytes": (
            path.stat().st_size
        ),
        "sha256": sha256_file(path),
    })

file_manifest = pd.DataFrame(
    file_manifest_rows
)

file_manifest_path = (
    OUTPUT_ROOT
    / "file_manifest.csv"
)

file_manifest.to_csv(
    file_manifest_path,
    index=False,
)


# ============================================================
# CREATE ZIP
# ============================================================

zip_base = (
    LOCAL_ROOT
    / "Delaware_Recall_Repeated_Nested_CV"
)

final_zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUTPUT_ROOT.parent,
    base_dir=OUTPUT_ROOT.name,
)

print()
print("Created ZIP:")
print(final_zip_path)


# ============================================================
# UPLOAD RESULTS TO DRIVE
# ============================================================

output_drive_folder_id = (
    get_or_create_folder(
        DELAWARE_FOLDER_ID,
        "Delaware_Recall_Repeated_Nested_CV",
    )
)

upload_paths = [
    Path(final_zip_path),
    final_model_path,
    configuration_path,
    model_summary_path,
    repeat_metrics_path,
    outer_metrics_path,
    confidence_intervals_path,
    outer_predictions_path,
    subject_aggregated_path,
    inner_selections_path,
    threshold_search_path,
    final_threshold_table_path,
    coverage_path,
    dataset_audit_path,
    readme_path,
    file_manifest_path,
]

print()
print("=" * 100)
print("UPLOADING NESTED-CV RESULTS")
print("=" * 100)

for local_path in upload_paths:
    result = upload_file(
        local_path,
        output_drive_folder_id,
    )

    print(
        "Uploaded:",
        result["name"],
    )


# ============================================================
# REMOVE TEMPORARY LOCAL INPUT FILES
# ============================================================

if INPUT_ROOT.exists():
    shutil.rmtree(
        INPUT_ROOT
    )

print()
print(
    "Deleted temporary downloaded and extracted "
    "runtime files."
)


# ============================================================
# DONE
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print("Best nested-CV model:")
print(best_model_name)

print()
print("Nested-CV model summary:")

display(
    model_summary[
        [
            "model",
            "balanced_accuracy_mean",
            "balanced_accuracy_std",
            "roc_auc_mean",
            "roc_auc_std",
            "f1_mean",
            "f1_std",
            "mcc_mean",
            "mcc_std",
        ]
    ]
)

print()
print(
    "Final fitted-model threshold:",
    final_threshold,
)

print()
print("Use this ZIP:")
print(
    "Delaware_Recall_Repeated_Nested_CV.zip"
)

In [ ]:
# ============================================================
# DELAWARE RECALL:
# NESTED STABLE FEATURE-SELECTION MODEL
#
# Uses only saved recall-content features.
# Feature ranking and selection happen INSIDE each outer fold.
#
# Tests:
# - Best single feature
# - Top 3 stable features
# - Top 5 stable features
# - Top 10 stable features
# - L1-selected features
#
# Restart-safe and Drive-folder-ID only.
# ============================================================

!pip -q install google-api-python-client scikit-learn joblib

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import (
    MediaIoBaseDownload,
    MediaFileUpload,
)

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    StratifiedKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from pathlib import Path

import hashlib
import json
import shutil
import warnings

import joblib
import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

DELAWARE_FOLDER_ID = "1RV-tYijUIOBV4O7jb8RcvnWC3RdV8U5J"

RANDOM_SEED = 42

OUTER_FOLDS = 5
OUTER_REPEATS = 5
INNER_FOLDS = 5

TOP_K_OPTIONS = [1, 3, 5, 10]

C_OPTIONS = [
    0.01,
    0.05,
    0.1,
    0.25,
    0.5,
    1.0,
    2.0,
]

CORRELATION_LIMITS = [
    0.75,
    0.85,
    0.95,
]

LOCAL_ROOT = Path(
    "/content/delaware_recall_stable_features"
)

INPUT_ROOT = LOCAL_ROOT / "input"

OUTPUT_ROOT = (
    LOCAL_ROOT
    / "Delaware_Recall_Stable_Feature_Model"
)

AUDIT_ROOT = OUTPUT_ROOT / "audit"
MODEL_ROOT = OUTPUT_ROOT / "model"
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"

for path in [
    INPUT_ROOT,
    OUTPUT_ROOT,
    AUDIT_ROOT,
    MODEL_ROOT,
    PREDICTION_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

warnings.filterwarnings("ignore")

drive_service = build(
    "drive",
    "v3",
)


# ============================================================
# DRIVE FUNCTIONS
# ============================================================

def list_folder(folder_id):
    items = []
    page_token = None

    while True:
        response = (
            drive_service.files()
            .list(
                q=(
                    f"'{folder_id}' in parents "
                    "and trashed = false"
                ),
                fields=(
                    "nextPageToken,"
                    "files("
                    "id,name,mimeType,size,"
                    "modifiedTime,parents"
                    ")"
                ),
                pageToken=page_token,
                pageSize=1000,
            )
            .execute()
        )

        items.extend(
            response.get("files", [])
        )

        page_token = response.get(
            "nextPageToken"
        )

        if not page_token:
            break

    return items


def walk_drive_folder(
    folder_id,
    relative_path=Path(),
):
    records = []

    for item in list_folder(folder_id):
        current_path = (
            relative_path / item["name"]
        )

        if (
            item["mimeType"]
            == "application/vnd.google-apps.folder"
        ):
            records.extend(
                walk_drive_folder(
                    item["id"],
                    current_path,
                )
            )

        else:
            records.append({
                "file_id": item["id"],
                "name": item["name"],
                "relative_path": str(
                    current_path
                ),
                "modified_time": item.get(
                    "modifiedTime"
                ),
            })

    return records


def download_drive_file(
    file_id,
    destination,
):
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    request = (
        drive_service.files()
        .get_media(fileId=file_id)
    )

    with destination.open("wb") as output_file:
        downloader = MediaIoBaseDownload(
            output_file,
            request,
            chunksize=16 * 1024 * 1024,
        )

        done = False

        while not done:
            _, done = downloader.next_chunk()

    return destination


def get_or_create_folder(
    parent_id,
    folder_name,
):
    safe_name = folder_name.replace(
        "'",
        "\\'",
    )

    response = (
        drive_service.files()
        .list(
            q=(
                f"'{parent_id}' in parents and "
                f"name = '{safe_name}' and "
                "mimeType = "
                "'application/vnd.google-apps.folder' "
                "and trashed = false"
            ),
            fields="files(id,name)",
        )
        .execute()
    )

    matches = response.get(
        "files",
        [],
    )

    if matches:
        return matches[0]["id"]

    created = (
        drive_service.files()
        .create(
            body={
                "name": folder_name,
                "mimeType": (
                    "application/"
                    "vnd.google-apps.folder"
                ),
                "parents": [parent_id],
            },
            fields="id",
        )
        .execute()
    )

    return created["id"]


def upload_file(
    local_path,
    parent_id,
):
    local_path = Path(local_path)

    return (
        drive_service.files()
        .create(
            body={
                "name": local_path.name,
                "parents": [parent_id],
            },
            media_body=MediaFileUpload(
                str(local_path),
                resumable=True,
            ),
            fields="id,name",
        )
        .execute()
    )


def find_latest_file(
    records,
    folder_name,
    filename,
):
    expected_suffix = (
        f"{folder_name}/{filename}"
    ).lower()

    matches = [
        record
        for record in records
        if (
            record["relative_path"]
            .replace("\\", "/")
            .lower()
            .endswith(expected_suffix)
        )
    ]

    if not matches:
        raise FileNotFoundError(
            f"Missing {expected_suffix}"
        )

    return sorted(
        matches,
        key=lambda record: (
            record.get("modified_time") or ""
        ),
        reverse=True,
    )[0]


# ============================================================
# FILE FUNCTIONS
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ============================================================
# DOWNLOAD SAVED RECALL FEATURES
# ============================================================

print("=" * 100)
print("LOCATING SAVED RECALL FEATURES")
print("=" * 100)

drive_records = walk_drive_folder(
    DELAWARE_FOLDER_ID
)

feature_record = find_latest_file(
    drive_records,
    "Delaware_Recall_Content_Model",
    "recall_content_features.csv",
)

feature_path = (
    INPUT_ROOT
    / "recall_content_features.csv"
)

download_drive_file(
    feature_record["file_id"],
    feature_path,
)

print(
    "Downloaded:",
    feature_record["relative_path"],
)


# ============================================================
# LOAD DATA
# ============================================================

data = pd.read_csv(
    feature_path
)

data["subject_id"] = (
    data["subject_id"]
    .astype(str)
)

data["recording_id"] = (
    data["recording_id"]
    .astype(str)
)

metadata_columns = {
    "split",
    "diagnosis",
    "binary_label",
    "subject_id",
    "recording_id",
    "transcript_path",
    "text",
}

feature_columns = [
    column
    for column in data.columns
    if column not in metadata_columns
]

X = data[
    feature_columns
].copy()

y = data[
    "binary_label"
].to_numpy(dtype=int)

subjects = data[
    "subject_id"
].to_numpy()

recordings = data[
    "recording_id"
].to_numpy()

diagnoses = data[
    "diagnosis"
].to_numpy()

if len(data) != 283:
    raise RuntimeError(
        f"Expected 283 subjects, found {len(data)}"
    )

if data["subject_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate subject IDs found."
    )

print()
print("Subjects:", len(data))
print("Features:", len(feature_columns))

print()
print(
    data["diagnosis"]
    .value_counts()
)


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(
    y_true,
    probabilities,
    threshold,
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "threshold": float(threshold),

        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),

        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),

        "sensitivity": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),

        "specificity": float(
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        ),

        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),

        "roc_auc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),

        "average_precision": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),

        "mcc": float(
            matthews_corrcoef(
                y_true,
                predictions,
            )
        ),

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def choose_threshold(
    y_true,
    probabilities,
):
    rows = []

    for threshold in np.linspace(
        0.20,
        0.80,
        121,
    ):
        metrics = calculate_metrics(
            y_true,
            probabilities,
            threshold,
        )

        metrics[
            "distance_from_half"
        ] = abs(
            threshold - 0.5
        )

        rows.append(metrics)

    table = pd.DataFrame(rows)

    table = (
        table
        .sort_values(
            [
                "balanced_accuracy",
                "f1",
                "mcc",
                "distance_from_half",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    return float(
        table.loc[
            0,
            "threshold",
        ]
    )


# ============================================================
# SINGLE-FEATURE AUC
# ============================================================

def safe_single_feature_auc(
    values,
    labels,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    median_value = np.nanmedian(
        values
    )

    values = np.where(
        np.isnan(values),
        median_value,
        values,
    )

    if np.std(values) == 0:
        return 0.5, 1

    auc = roc_auc_score(
        labels,
        values,
    )

    if auc < 0.5:
        return 1.0 - auc, -1

    return auc, 1


# ============================================================
# INNER STABILITY RANKING
# ============================================================

def rank_features_stably(
    X_train,
    y_train,
    random_seed,
):
    inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=random_seed,
    )

    ranking_rows = []

    for inner_fold, (
        inner_train_indices,
        inner_validation_indices,
    ) in enumerate(
        inner_cv.split(
            X_train,
            y_train,
        ),
        start=1,
    ):
        fold_X_train = X_train.iloc[
            inner_train_indices
        ]

        fold_y_train = y_train[
            inner_train_indices
        ]

        for feature_name in feature_columns:
            auc, direction = (
                safe_single_feature_auc(
                    fold_X_train[
                        feature_name
                    ].to_numpy(),
                    fold_y_train,
                )
            )

            ranking_rows.append({
                "inner_fold": inner_fold,
                "feature": feature_name,
                "auc": auc,
                "direction": direction,
            })

    ranking_table = pd.DataFrame(
        ranking_rows
    )

    stability = (
        ranking_table
        .groupby("feature")
        .agg(
            mean_auc=(
                "auc",
                "mean",
            ),
            std_auc=(
                "auc",
                "std",
            ),
            minimum_auc=(
                "auc",
                "min",
            ),
            positive_direction_fraction=(
                "direction",
                lambda values: (
                    np.mean(
                        np.asarray(values)
                        == 1
                    )
                ),
            ),
        )
        .reset_index()
    )

    stability[
        "direction_consistency"
    ] = np.maximum(
        stability[
            "positive_direction_fraction"
        ],
        1.0
        - stability[
            "positive_direction_fraction"
        ],
    )

    stability[
        "stability_score"
    ] = (
        stability["mean_auc"]
        - 0.50
        - 0.50
        * stability[
            "std_auc"
        ].fillna(0)
        + 0.05
        * (
            stability[
                "direction_consistency"
            ]
            - 0.50
        )
    )

    stability = (
        stability
        .sort_values(
            [
                "stability_score",
                "mean_auc",
                "direction_consistency",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    return stability


# ============================================================
# CORRELATION PRUNING
# ============================================================

def select_top_uncorrelated_features(
    X_train,
    ranked_features,
    top_k,
    correlation_limit,
):
    selected = []

    correlation_matrix = (
        X_train[
            ranked_features
        ]
        .corr(
            method="spearman"
        )
        .abs()
    )

    for feature_name in ranked_features:
        if len(selected) >= top_k:
            break

        too_correlated = False

        for existing_feature in selected:
            correlation = (
                correlation_matrix
                .loc[
                    feature_name,
                    existing_feature,
                ]
            )

            if (
                pd.notna(correlation)
                and correlation
                >= correlation_limit
            ):
                too_correlated = True
                break

        if not too_correlated:
            selected.append(
                feature_name
            )

    return selected


# ============================================================
# MODEL
# ============================================================

def build_logistic_model(
    C,
    penalty="l2",
):
    if penalty == "elasticnet":
        classifier = LogisticRegression(
            C=C,
            penalty="elasticnet",
            l1_ratio=0.5,
            class_weight="balanced",
            solver="saga",
            max_iter=20000,
            random_state=RANDOM_SEED,
        )

    elif penalty == "l1":
        classifier = LogisticRegression(
            C=C,
            penalty="l1",
            class_weight="balanced",
            solver="liblinear",
            max_iter=20000,
            random_state=RANDOM_SEED,
        )

    else:
        classifier = LogisticRegression(
            C=C,
            penalty="l2",
            class_weight="balanced",
            solver="liblinear",
            max_iter=20000,
            random_state=RANDOM_SEED,
        )

    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            classifier,
        ),
    ])


# ============================================================
# INNER CONFIGURATION SELECTION
# ============================================================

def select_configuration(
    X_outer_train,
    y_outer_train,
    stability_table,
    outer_seed,
):
    ranked_features = (
        stability_table[
            "feature"
        ]
        .tolist()
    )

    inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=outer_seed,
    )

    configuration_rows = []

    for top_k in TOP_K_OPTIONS:
        for correlation_limit in (
            CORRELATION_LIMITS
        ):
            selected_features = (
                select_top_uncorrelated_features(
                    X_outer_train,
                    ranked_features,
                    top_k,
                    correlation_limit,
                )
            )

            if not selected_features:
                continue

            for penalty in [
                "l2",
                "l1",
                "elasticnet",
            ]:
                for C in C_OPTIONS:
                    inner_probabilities = np.zeros(
                        len(y_outer_train),
                        dtype=float,
                    )

                    for (
                        inner_train_indices,
                        inner_validation_indices,
                    ) in inner_cv.split(
                        X_outer_train,
                        y_outer_train,
                    ):
                        model = (
                            build_logistic_model(
                                C=C,
                                penalty=penalty,
                            )
                        )

                        model.fit(
                            X_outer_train.iloc[
                                inner_train_indices
                            ][selected_features],
                            y_outer_train[
                                inner_train_indices
                            ],
                        )

                        inner_probabilities[
                            inner_validation_indices
                        ] = (
                            model.predict_proba(
                                X_outer_train.iloc[
                                    inner_validation_indices
                                ][selected_features]
                            )[:, 1]
                        )

                    threshold = choose_threshold(
                        y_outer_train,
                        inner_probabilities,
                    )

                    metrics = calculate_metrics(
                        y_outer_train,
                        inner_probabilities,
                        threshold,
                    )

                    configuration_rows.append({
                        "top_k": top_k,
                        "correlation_limit": (
                            correlation_limit
                        ),
                        "penalty": penalty,
                        "C": C,
                        "selected_features": (
                            json.dumps(
                                selected_features
                            )
                        ),
                        **metrics,
                    })

    results = pd.DataFrame(
        configuration_rows
    )

    results[
        "distance_from_half"
    ] = (
        results["threshold"]
        - 0.5
    ).abs()

    results = (
        results
        .sort_values(
            [
                "balanced_accuracy",
                "roc_auc",
                "f1",
                "mcc",
                "top_k",
                "distance_from_half",
            ],
            ascending=[
                False,
                False,
                False,
                False,
                True,
                True,
            ],
        )
        .reset_index(drop=True)
    )

    return results.iloc[0], results


# ============================================================
# REPEATED NESTED CROSS-VALIDATION
# ============================================================

outer_cv = RepeatedStratifiedKFold(
    n_splits=OUTER_FOLDS,
    n_repeats=OUTER_REPEATS,
    random_state=RANDOM_SEED,
)

outer_metric_rows = []
outer_prediction_rows = []
feature_selection_rows = []
configuration_rows = []

outer_splits = list(
    outer_cv.split(
        X,
        y,
    )
)

print()
print("=" * 100)
print("RUNNING NESTED STABLE FEATURE SELECTION")
print("=" * 100)

for outer_index, (
    outer_train_indices,
    outer_test_indices,
) in enumerate(
    outer_splits,
    start=1,
):
    repeat_number = (
        (outer_index - 1)
        // OUTER_FOLDS
        + 1
    )

    fold_number = (
        (outer_index - 1)
        % OUTER_FOLDS
        + 1
    )

    print()
    print(
        f"Repeat {repeat_number}/"
        f"{OUTER_REPEATS}, "
        f"Fold {fold_number}/"
        f"{OUTER_FOLDS}"
    )

    X_outer_train = X.iloc[
        outer_train_indices
    ].copy()

    y_outer_train = y[
        outer_train_indices
    ]

    X_outer_test = X.iloc[
        outer_test_indices
    ].copy()

    y_outer_test = y[
        outer_test_indices
    ]

    outer_seed = (
        RANDOM_SEED
        + outer_index * 100
    )

    stability_table = (
        rank_features_stably(
            X_outer_train,
            y_outer_train,
            random_seed=outer_seed,
        )
    )

    stability_table[
        "repeat"
    ] = repeat_number

    stability_table[
        "fold"
    ] = fold_number

    feature_selection_rows.append(
        stability_table
    )

    (
        best_configuration,
        all_configurations,
    ) = select_configuration(
        X_outer_train,
        y_outer_train,
        stability_table,
        outer_seed,
    )

    all_configurations[
        "repeat"
    ] = repeat_number

    all_configurations[
        "fold"
    ] = fold_number

    configuration_rows.append(
        all_configurations
    )

    selected_features = json.loads(
        best_configuration[
            "selected_features"
        ]
    )

    final_model = build_logistic_model(
        C=float(
            best_configuration["C"]
        ),
        penalty=str(
            best_configuration[
                "penalty"
            ]
        ),
    )

    final_model.fit(
        X_outer_train[
            selected_features
        ],
        y_outer_train,
    )

    outer_probabilities = (
        final_model.predict_proba(
            X_outer_test[
                selected_features
            ]
        )[:, 1]
    )

    threshold = float(
        best_configuration[
            "threshold"
        ]
    )

    metrics = calculate_metrics(
        y_outer_test,
        outer_probabilities,
        threshold,
    )

    outer_metric_rows.append({
        "repeat": repeat_number,
        "fold": fold_number,
        "selected_feature_count": (
            len(selected_features)
        ),
        "selected_features": json.dumps(
            selected_features
        ),
        "correlation_limit": float(
            best_configuration[
                "correlation_limit"
            ]
        ),
        "penalty": str(
            best_configuration[
                "penalty"
            ]
        ),
        "C": float(
            best_configuration["C"]
        ),
        "inner_balanced_accuracy": float(
            best_configuration[
                "balanced_accuracy"
            ]
        ),
        **metrics,
    })

    predictions = (
        outer_probabilities
        >= threshold
    ).astype(int)

    for position, dataset_index in enumerate(
        outer_test_indices
    ):
        outer_prediction_rows.append({
            "repeat": repeat_number,
            "fold": fold_number,
            "subject_id": subjects[
                dataset_index
            ],
            "recording_id": recordings[
                dataset_index
            ],
            "diagnosis": diagnoses[
                dataset_index
            ],
            "binary_label": int(
                y[dataset_index]
            ),
            "probability_MCI": float(
                outer_probabilities[
                    position
                ]
            ),
            "predicted_label": int(
                predictions[position]
            ),
            "threshold": threshold,
            "selected_features": (
                json.dumps(
                    selected_features
                )
            ),
            "correct": bool(
                predictions[position]
                == y[dataset_index]
            ),
        })

    print(
        "Selected:",
        selected_features,
    )

    print(
        "Outer balanced accuracy:",
        round(
            metrics[
                "balanced_accuracy"
            ],
            4,
        ),
        "| ROC-AUC:",
        round(
            metrics[
                "roc_auc"
            ],
            4,
        ),
    )


# ============================================================
# SAVE RAW RESULTS
# ============================================================

outer_metrics = pd.DataFrame(
    outer_metric_rows
)

outer_predictions = pd.DataFrame(
    outer_prediction_rows
)

feature_stability = pd.concat(
    feature_selection_rows,
    ignore_index=True,
)

all_configurations = pd.concat(
    configuration_rows,
    ignore_index=True,
)

outer_metrics_path = (
    OUTPUT_ROOT
    / "outer_fold_metrics.csv"
)

outer_predictions_path = (
    PREDICTION_ROOT
    / "outer_predictions.csv"
)

feature_stability_path = (
    AUDIT_ROOT
    / "feature_stability_by_fold.csv"
)

configuration_path = (
    AUDIT_ROOT
    / "inner_configuration_search.csv"
)

outer_metrics.to_csv(
    outer_metrics_path,
    index=False,
)

outer_predictions.to_csv(
    outer_predictions_path,
    index=False,
)

feature_stability.to_csv(
    feature_stability_path,
    index=False,
)

all_configurations.to_csv(
    configuration_path,
    index=False,
)


# ============================================================
# RESULTS BY REPETITION
# ============================================================

repeat_rows = []

for repeat_number, group in (
    outer_predictions.groupby(
        "repeat"
    )
):
    true_labels = (
        group["binary_label"]
        .to_numpy(dtype=int)
    )

    probabilities = (
        group["probability_MCI"]
        .to_numpy()
    )

    predictions = (
        group["predicted_label"]
        .to_numpy(dtype=int)
    )

    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        predictions,
        labels=[0, 1],
    ).ravel()

    repeat_rows.append({
        "repeat": int(
            repeat_number
        ),
        "accuracy": float(
            accuracy_score(
                true_labels,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                true_labels,
                predictions,
            )
        ),
        "precision": float(
            precision_score(
                true_labels,
                predictions,
                zero_division=0,
            )
        ),
        "sensitivity": float(
            recall_score(
                true_labels,
                predictions,
                zero_division=0,
            )
        ),
        "specificity": float(
            tn / (tn + fp)
        ),
        "f1": float(
            f1_score(
                true_labels,
                predictions,
                zero_division=0,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                true_labels,
                probabilities,
            )
        ),
        "average_precision": float(
            average_precision_score(
                true_labels,
                probabilities,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                true_labels,
                predictions,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    })

repeat_metrics = pd.DataFrame(
    repeat_rows
)

repeat_metrics_path = (
    OUTPUT_ROOT
    / "repeat_metrics.csv"
)

repeat_metrics.to_csv(
    repeat_metrics_path,
    index=False,
)

print()
print("=" * 100)
print("REPEATED NESTED-CV RESULTS")
print("=" * 100)

display(repeat_metrics)


# ============================================================
# SUMMARY
# ============================================================

summary_metrics = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "sensitivity",
    "specificity",
    "f1",
    "roc_auc",
    "average_precision",
    "mcc",
]

summary = {}

for metric_name in summary_metrics:
    values = repeat_metrics[
        metric_name
    ].to_numpy()

    summary[
        metric_name
    ] = {
        "mean": float(
            np.mean(values)
        ),
        "standard_deviation": float(
            np.std(
                values,
                ddof=1,
            )
        ),
        "minimum": float(
            np.min(values)
        ),
        "maximum": float(
            np.max(values)
        ),
    }

summary_path = (
    OUTPUT_ROOT
    / "nested_cv_summary.json"
)

summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

print()
print(
    json.dumps(
        summary,
        indent=2,
    )
)


# ============================================================
# FEATURE SELECTION FREQUENCY
# ============================================================

selected_feature_lists = (
    outer_metrics[
        "selected_features"
    ]
    .map(json.loads)
)

selection_counts = {
    feature_name: 0
    for feature_name
    in feature_columns
}

for selected_features in (
    selected_feature_lists
):
    for feature_name in (
        selected_features
    ):
        selection_counts[
            feature_name
        ] += 1

feature_frequency = pd.DataFrame({
    "feature": list(
        selection_counts.keys()
    ),
    "selection_count": list(
        selection_counts.values()
    ),
})

feature_frequency[
    "selection_fraction"
] = (
    feature_frequency[
        "selection_count"
    ]
    / len(outer_metrics)
)

stability_summary = (
    feature_stability
    .groupby("feature")
    .agg(
        mean_inner_auc=(
            "mean_auc",
            "mean",
        ),
        std_inner_auc=(
            "mean_auc",
            "std",
        ),
        mean_direction_consistency=(
            "direction_consistency",
            "mean",
        ),
        mean_stability_score=(
            "stability_score",
            "mean",
        ),
    )
    .reset_index()
)

feature_report = (
    feature_frequency
    .merge(
        stability_summary,
        on="feature",
        how="left",
    )
    .sort_values(
        [
            "selection_fraction",
            "mean_inner_auc",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

feature_report_path = (
    OUTPUT_ROOT
    / "stable_feature_report.csv"
)

feature_report.to_csv(
    feature_report_path,
    index=False,
)

print()
print("=" * 100)
print("MOST STABLE FEATURES")
print("=" * 100)

display(
    feature_report.head(20)
)


# ============================================================
# CHOOSE FINAL FEATURES
# ============================================================

# Keep features selected in at least 40% of outer folds.
final_selected_features = (
    feature_report[
        feature_report[
            "selection_fraction"
        ]
        >= 0.40
    ]["feature"]
    .tolist()
)

if not final_selected_features:
    final_selected_features = (
        feature_report
        .head(5)["feature"]
        .tolist()
    )

most_common_penalty = (
    outer_metrics[
        "penalty"
    ]
    .value_counts()
    .index[0]
)

most_common_C = float(
    outer_metrics["C"]
    .round(6)
    .value_counts()
    .index[0]
)

final_threshold = float(
    outer_metrics[
        "threshold"
    ].median()
)

print()
print("Final selected features:")
print(final_selected_features)

print()
print("Final penalty:")
print(most_common_penalty)

print("Final C:")
print(most_common_C)

print("Final threshold:")
print(final_threshold)


# ============================================================
# FIT FINAL MODEL ON ALL 283 SUBJECTS
# ============================================================

final_model = build_logistic_model(
    C=most_common_C,
    penalty=most_common_penalty,
)

final_model.fit(
    X[
        final_selected_features
    ],
    y,
)

final_model_path = (
    MODEL_ROOT
    / "stable_recall_feature_model.joblib"
)

joblib.dump(
    final_model,
    final_model_path,
)

final_configuration = {
    "dataset": "Delaware",
    "task": "Cinderella recall",
    "target": "MCI versus Control",
    "subjects": int(
        len(data)
    ),
    "evaluation": {
        "outer_folds": OUTER_FOLDS,
        "outer_repeats": (
            OUTER_REPEATS
        ),
        "inner_folds": INNER_FOLDS,
    },
    "selected_features": (
        final_selected_features
    ),
    "penalty": (
        most_common_penalty
    ),
    "C": most_common_C,
    "operational_threshold": (
        final_threshold
    ),
    "nested_cv_results": summary,
    "clinical_status": (
        "Research use only; not a diagnostic test."
    ),
}

final_configuration_path = (
    MODEL_ROOT
    / "final_configuration.json"
)

final_configuration_path.write_text(
    json.dumps(
        final_configuration,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# README
# ============================================================

readme_text = f"""
DELAWARE RECALL STABLE FEATURE MODEL
====================================

Subjects
--------
{len(data)}

Features initially considered
-----------------------------
{len(feature_columns)}

Evaluation
----------
{OUTER_FOLDS}-fold nested cross-validation repeated
{OUTER_REPEATS} times.

Feature ranking, correlation pruning, model selection,
regularization selection, and threshold selection occurred
inside each outer-training fold.

Final stable features
---------------------
{json.dumps(final_selected_features, indent=2)}

Mean balanced accuracy
----------------------
{summary['balanced_accuracy']['mean']:.4f}
± {summary['balanced_accuracy']['standard_deviation']:.4f}

Mean ROC-AUC
------------
{summary['roc_auc']['mean']:.4f}
± {summary['roc_auc']['standard_deviation']:.4f}

Mean F1
-------
{summary['f1']['mean']:.4f}
± {summary['f1']['standard_deviation']:.4f}

Mean MCC
--------
{summary['mcc']['mean']:.4f}
± {summary['mcc']['standard_deviation']:.4f}

This is a research model and not a clinical diagnostic test.
"""

readme_path = (
    OUTPUT_ROOT
    / "README.txt"
)

readme_path.write_text(
    readme_text.strip() + "\n",
    encoding="utf-8",
)


# ============================================================
# FILE MANIFEST
# ============================================================

file_manifest_rows = []

for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if not path.is_file():
        continue

    file_manifest_rows.append({
        "relative_path": str(
            path.relative_to(
                OUTPUT_ROOT
            )
        ),
        "size_bytes": (
            path.stat().st_size
        ),
        "sha256": sha256_file(
            path
        ),
    })

file_manifest = pd.DataFrame(
    file_manifest_rows
)

file_manifest_path = (
    OUTPUT_ROOT
    / "file_manifest.csv"
)

file_manifest.to_csv(
    file_manifest_path,
    index=False,
)


# ============================================================
# ZIP
# ============================================================

zip_base = (
    LOCAL_ROOT
    / "Delaware_Recall_Stable_Feature_Model"
)

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUTPUT_ROOT.parent,
    base_dir=OUTPUT_ROOT.name,
)

print()
print("Created:")
print(zip_path)


# ============================================================
# UPLOAD
# ============================================================

output_drive_folder_id = (
    get_or_create_folder(
        DELAWARE_FOLDER_ID,
        "Delaware_Recall_Stable_Feature_Model",
    )
)

upload_paths = [
    Path(zip_path),
    final_model_path,
    final_configuration_path,
    repeat_metrics_path,
    outer_metrics_path,
    outer_predictions_path,
    feature_stability_path,
    feature_report_path,
    configuration_path,
    summary_path,
    readme_path,
    file_manifest_path,
]

print()
print("=" * 100)
print("UPLOADING")
print("=" * 100)

for local_path in upload_paths:
    result = upload_file(
        local_path,
        output_drive_folder_id,
    )

    print(
        "Uploaded:",
        result["name"],
    )


# ============================================================
# CLEAN LOCAL TEMP FILES
# ============================================================

if INPUT_ROOT.exists():
    shutil.rmtree(
        INPUT_ROOT
    )

print()
print(
    "Deleted temporary runtime input files."
)


# ============================================================
# DONE
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print("Final stable features:")
print(final_selected_features)

print()
print("Balanced accuracy:")
print(
    summary[
        "balanced_accuracy"
    ]
)

print()
print("ROC-AUC:")
print(
    summary["roc_auc"]
)

print()
print("Use:")
print(
    "Delaware_Recall_Stable_Feature_Model.zip"
)